# Gemma-2-2B-IT Local Feature Homeostasis (LFH) on AGNews — single-user prompt control

This notebook is the **Gemma2-2B-IT** variant of the LFH AGNews SAE analysis.

It preserves the LFH feature-discovery, attention-head negative-write analysis, attention-head causal intervention, and MLP control from the supplied notebook. All eight AGNews demonstrations remain inside one Gemma user prompt. The default `BLOCK_SCORE_MODE = "All"` measures and intervenes on all selected content tokens in each exemplar.

In [ ]:

import os
import re
import json
import math
import random
import warnings
from collections import defaultdict

# Silence the harmless tqdm/Jupyter widget warning before datasets imports tqdm.auto.
warnings.filterwarnings("ignore", message=r"IProgress not found.*")

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from scipy.stats import pearsonr


# ============================================================
# 0. Reproducibility and experiment config
# ============================================================

def set_seed(seed: int):
    print("seed:", seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(42)

# Model/tokenizer loading policy. Gated model checkpoints are loaded from the
# existing local cache; public Qwen checkpoints may be downloaded anonymously.
# token=False prevents an expired ambient Hugging Face credential from being attached.
MODEL_LOCAL_FILES_ONLY = True
SAE_LOCAL_FILES_ONLY = False
MODEL_TRUST_REMOTE_CODE = False

# Model/SAE variant generated from the supplied LFH notebook.
REQUESTED_MODEL_NAME = 'Gemma2-2B-IT'
MODEL_NAME = 'google/gemma-2-2b-it'
MODEL_SHORT_NAME = 'Gemma-2-2B-IT'
TARGET_LAYER = 12
SAE_BACKEND = 'gemma_scope_npz'

SAE_REPO_ID = 'google/gemma-scope-2b-pt-res'
SAE_TRAINER = None
SAE_BASE_DIR = None
SAE_WIDTH = 'width_16k'
SAE_AVERAGE_L0 = 'average_l0_82'
SAE_FILENAME = f"layer_{TARGET_LAYER}/{SAE_WIDTH}/{SAE_AVERAGE_L0}/params.npz"
SAE_HYPERPARAMS_FILENAME = None
SAE_LENS_RELEASE = None
SAE_LENS_SAE_ID = None
SAE_TOPK_K = None

EXPECTED_MODEL_NUM_LAYERS = 26
EXPECTED_D_MODEL = 2304
EXPECTED_D_SAE = 16384

# Prompt family:
#   raw: raw AGNews few-shot text, optionally prefixed by PROMPT_BOS_TOKEN;
#   qwen_chat_user_blocks: one Qwen user message per demonstration block;
#   qwen_chat_single_user_block: one Qwen user turn containing the full 8-shot prompt;
#   gemma_chat_single_user_block: one Gemma user turn containing the full 8-shot prompt;
#   gemma_chat_turn_pairs: Gemma <start_of_turn>user/model pairs;
#   llama_chat_single_user_block: one Llama user turn containing the full 8-shot prompt;
#   llama_chat_turn_pairs: Llama 3.1 user/assistant turn pairs.
PROMPT_FORMAT = 'gemma_chat_single_user_block'
PROMPT_BOS_TOKEN = '<bos>'
TOKENIZE_ADD_SPECIAL_TOKENS = False

# High-norm-token filtering is useful for the Qwen BatchTopK SAE suite and disabled by default elsewhere.
FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION = False
HIGH_NORM_FILTER_MULTIPLIER = 10.0
# Backward-compatible alias used by a few downstream helper cells.
QWEN_HIGH_NORM_FILTER_MULTIPLIER = HIGH_NORM_FILTER_MULTIPLIER

DTYPE = torch.bfloat16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Original notebook defaults were 50 x 50 = 2500 prompts.
# For debugging, set smaller values such as NUM_TEST_QUERY=5, NUM_PREVIOUS=5.
NUM_TEST_QUERY = 50
NUM_PREVIOUS = 50
NUM_BLOCKS_TOTAL = 8
FREQ_THRESHOLD = 0.05

# --------------------------------------------------------------------
# LFH source/target block configuration
# --------------------------------------------------------------------
# The target block stays fixed at Example 8. Change LFH_SOURCE_BLOCK_NUM
# to 7, 6, or 5 to resample that source demonstration while keeping the
# other context demonstrations fixed. This lets us test whether the LFH
# correlation decays as the varied exemplar becomes less local to Example 8.
LFH_TARGET_BLOCK_NUM = 8
LFH_SOURCE_BLOCK_NUM = 7
LFH_LOCALITY_SWEEP_SOURCE_BLOCK_NUMS = [7, 6, 5]


def block_num_to_index(block_num: int, total_blocks: int = NUM_BLOCKS_TOTAL) -> int:
    """Convert a 1-indexed Example block number to a Python list index."""
    block_num = int(block_num)
    if block_num < 1 or block_num > int(total_blocks):
        raise ValueError(f"block_num must be in [1, {total_blocks}], got {block_num}.")
    return block_num - 1


LFH_SOURCE_BLOCK_INDEX = block_num_to_index(LFH_SOURCE_BLOCK_NUM)
LFH_TARGET_BLOCK_INDEX = block_num_to_index(LFH_TARGET_BLOCK_NUM)

# --------------------------------------------------------------------
# Block measurement configuration
# --------------------------------------------------------------------
# Chat/instruct variants use each model family's native turn terminator as
# the block end marker. Base/raw variants do not inject per-block special
# terminators; their block span ends immediately before the next Example header.
#
# BLOCK_SCORE_MODE controls which tokens in each exemplar are averaged:
#   - 'selected_offsets_from_end': use exact token positions specified by
#       BLOCK_TOKEN_OFFSETS_FROM_END. Negative offsets count backward from the
#       exemplar end: -1 is the final token, -2 is the second-to-final token, etc.
#       Examples:
#           [-1]          == the old BLOCK_LAST_K_TOKENS = 1 setting
#           [-1, -2, -3]  == the old BLOCK_LAST_K_TOKENS = 3 setting
#           [-2]          == select only the second-to-final token
#   - 'All': use all tokens in each exemplar span.
# Backward-compatible aliases accepted downstream: 'last_k', 'content', 'all'.
BLOCK_END_TOKEN = '<end_of_turn>'
BLOCK_SCORE_MODE = 'All'
BLOCK_TOKEN_OFFSETS_FROM_END = None

# Not
# BLOCK_TOKEN_OFFSETS_FROM_END = [-1, -2]
# BLOCK_TOKEN_OFFSETS_FROM_END = [-3, -4, -5, -6, -7]
# Candidate BLOCK_TOKEN_OFFSETS_FROM_END = [-1, -2, -3, -4, -5]
# Very 유력 후보: BLOCK_TOKEN_OFFSETS_FROM_END = [-1, -2, -3, -4, -5, -6, -7]
# Very 유력 후보: BLOCK_TOKEN_OFFSETS_FROM_END = [-2, -3, -4, -5, -6, -7] => 9805 = 7748


# Alternative examples:
# BLOCK_TOKEN_OFFSETS_FROM_END = [-1, -2, -3]
# BLOCK_TOKEN_OFFSETS_FROM_END = [-2]
# BLOCK_SCORE_MODE = 'All'
# BLOCK_TOKEN_OFFSETS_FROM_END = None

# Backward-compatible fallback only: used if BLOCK_SCORE_MODE is set to a legacy
# last_k* mode and BLOCK_TOKEN_OFFSETS_FROM_END is None.
BLOCK_LAST_K_TOKENS = 1

QWEN_SYSTEM_PROMPT = None
QWEN_CHAT_TEMPLATE_KWARGS = {"enable_thinking": False}
LLAMA_SYSTEM_PROMPT = None

NOTE_FOR_THIS_VARIANT = 'Uses the Gemma Scope 2B pretrained residual SAE at layer 12 on Gemma-2-2B-IT, with all eight AGNews demonstrations in one Gemma user prompt.'
print(NOTE_FOR_THIS_VARIANT)


# ============================================================
# 1. Load AGNews, model, tokenizer, and SAE
# ============================================================

def load_agnews_splits():
    """
    Load AGNews robustly across recent datasets / huggingface_hub versions.

    Newer huggingface_hub versions reject single-segment Hub dataset ids such
    as "ag_news" when resolving hf:// URIs. The namespaced dataset id below
    avoids:
        HfUriError: Repository id must be 'namespace/name', got 'ag_news'.
    """
    dataset_id_candidates = [
        "fancyzhx/ag_news",  # current namespaced HF dataset id
        "ag_news",          # fallback for older local datasets installations/caches
    ]

    errors = []
    for dataset_id in dataset_id_candidates:
        try:
            print(f"Loading AGNews from {dataset_id!r}...")
            return load_dataset(dataset_id, split=["train", "test"], token=False)
        except Exception as e:
            errors.append((dataset_id, type(e).__name__, str(e)))
            print(f"Failed to load {dataset_id!r}: {type(e).__name__}: {e}")

    error_text = "\n\n".join(
        f"{dataset_id}: {err_type}: {err_msg}"
        for dataset_id, err_type, err_msg in errors
    )
    raise RuntimeError(
        "Could not load AGNews from any configured dataset id.\n\n"
        f"Tried:\n{error_text}"
    )


dataset = load_agnews_splits()
train_data_agnews = dataset[0]
test_data_agnews = dataset[1]

label_to_words_agnews = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Technology",
}

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=MODEL_TRUST_REMOTE_CODE,
    use_fast=True,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    token=False,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=MODEL_TRUST_REMOTE_CODE,
    low_cpu_mem_usage=True,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    token=False,
)
model.eval()


def get_model_input_device(model):
    return model.get_input_embeddings().weight.device


def get_layer_device(model, layer_idx: int):
    block = model.get_submodule(f"model.layers.{layer_idx}")
    try:
        return next(block.parameters()).device
    except StopIteration:
        return get_model_input_device(model)


MODEL_INPUT_DEVICE = get_model_input_device(model)
TARGET_LAYER_DEVICE = get_layer_device(model, TARGET_LAYER)

print("REQUESTED_MODEL_NAME:", REQUESTED_MODEL_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_SHORT_NAME:", MODEL_SHORT_NAME)
print("TARGET_LAYER:", TARGET_LAYER)
print("MODEL_INPUT_DEVICE:", MODEL_INPUT_DEVICE)
print("TARGET_LAYER_DEVICE:", TARGET_LAYER_DEVICE)
print("SAE_BACKEND:", SAE_BACKEND)
print("SAE_REPO_ID:", SAE_REPO_ID)
print("SAE_FILENAME:", SAE_FILENAME)
print("SAE_HYPERPARAMS_FILENAME:", SAE_HYPERPARAMS_FILENAME)
print("SAE_LENS_RELEASE:", SAE_LENS_RELEASE)
print("SAE_LENS_SAE_ID:", SAE_LENS_SAE_ID)
print("PROMPT_FORMAT:", PROMPT_FORMAT)
print("BLOCK_END_TOKEN:", BLOCK_END_TOKEN)


class QwenBatchTopKSAE(torch.nn.Module):
    """
    Minimal inference wrapper for Adam Karvonen's Qwen3 BatchTopK SAEs.

    The checkpoint has dictionary_learning-style keys:
      b_dec, k, threshold, decoder.weight, encoder.weight, encoder.bias

    This wrapper exposes an SAE-Lens-like interface:
      W_enc: [d_model, n_features]
      W_dec: [n_features, d_model]
      b_enc: [n_features]
      b_dec: [d_model]
      process_sae_in(x)
      activation_fn(pre)
      encode(x)
      decode(acts)
    """

    def __init__(self, state_dict, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        if isinstance(state_dict, dict) and "model" in state_dict and isinstance(state_dict["model"], dict):
            state_dict = state_dict["model"]

        b_dec = state_dict["b_dec"].detach()
        b_enc = state_dict.get("encoder.bias", state_dict.get("b_enc")).detach()
        enc_w = state_dict.get("encoder.weight", state_dict.get("W_enc")).detach()
        dec_w = state_dict.get("decoder.weight", state_dict.get("W_dec")).detach()
        threshold = state_dict["threshold"]
        threshold = threshold.detach() if torch.is_tensor(threshold) else torch.tensor(threshold)
        k = state_dict.get("k", None)
        if torch.is_tensor(k):
            k = int(k.item())

        d_model = int(b_dec.numel())
        n_features = int(b_enc.numel())

        if enc_w.shape == (n_features, d_model):
            W_enc = enc_w.T.contiguous()
        elif enc_w.shape == (d_model, n_features):
            W_enc = enc_w.contiguous()
        else:
            raise ValueError(
                f"Unexpected encoder.weight shape {tuple(enc_w.shape)} "
                f"for d_model={d_model}, n_features={n_features}"
            )

        if dec_w.shape == (d_model, n_features):
            W_dec = dec_w.T.contiguous()
        elif dec_w.shape == (n_features, d_model):
            W_dec = dec_w.contiguous()
        else:
            raise ValueError(
                f"Unexpected decoder.weight shape {tuple(dec_w.shape)} "
                f"for d_model={d_model}, n_features={n_features}"
            )

        self.register_buffer("W_enc", W_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.to(device=self.device, dtype=self.dtype))

        self.k = k
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        return x - self.b_dec

    def process_sae_out(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, pre, torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


class GemmaScopeJumpReLUSAE(torch.nn.Module):
    """
    Minimal inference wrapper for Gemma Scope JumpReLU residual-stream SAEs.
    """

    def __init__(self, params, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        def get_param(name, *aliases):
            for key in (name,) + aliases:
                if key in params:
                    value = params[key]
                    if isinstance(value, np.ndarray):
                        value = torch.from_numpy(value)
                    return value.detach() if torch.is_tensor(value) else torch.tensor(value)
            raise KeyError(f"Missing SAE parameter {name!r}. Available keys: {sorted(params.keys())}")

        W_enc = get_param("W_enc")
        W_dec = get_param("W_dec")
        b_enc = get_param("b_enc")
        b_dec = get_param("b_dec")
        threshold = get_param("threshold", "thresholds")

        if W_enc.ndim != 2 or W_dec.ndim != 2:
            raise ValueError(f"Expected 2D W_enc/W_dec, got {tuple(W_enc.shape)} and {tuple(W_dec.shape)}")

        d_model = int(W_enc.shape[0])
        n_features = int(W_enc.shape[1])

        if W_dec.shape == (d_model, n_features):
            W_dec = W_dec.T.contiguous()
        if W_dec.shape != (n_features, d_model):
            raise ValueError(f"Expected W_dec shape {(n_features, d_model)}, got {tuple(W_dec.shape)}")
        if b_enc.numel() != n_features:
            raise ValueError(f"Expected b_enc length {n_features}, got {b_enc.numel()}")
        if b_dec.numel() != d_model:
            raise ValueError(f"Expected b_dec length {d_model}, got {b_dec.numel()}")
        if threshold.numel() != n_features:
            raise ValueError(f"Expected threshold length {n_features}, got {threshold.numel()}")

        self.register_buffer("W_enc", W_enc.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))

        self.k = None
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        # Gemma Scope folds the pre-encoder bias into the encoder parameters.
        return x

    def process_sae_out(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, torch.relu(pre), torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


class SAELensAdapter(torch.nn.Module):
    """
    Thin adapter exposing a consistent interface for SAELens-loaded SAEs.

    This keeps the rest of the LFH notebook independent of SAELens internals:
      W_enc, W_dec, b_enc, b_dec, threshold, k, encode_pre, activation_fn.
    """

    def __init__(self, saelens_sae, cfg_dict=None, device=DEVICE, dtype=DTYPE, fallback_k=None):
        super().__init__()
        self.saelens_sae = saelens_sae
        self.cfg_dict = cfg_dict or {}
        self.device = torch.device(device)
        self.dtype = dtype

        W_enc = self._get_tensor_attr(["W_enc", "encoder_weight", "W_E"])
        W_dec = self._get_tensor_attr(["W_dec", "decoder_weight", "W_D"])
        b_enc = self._get_tensor_attr(["b_enc", "encoder_bias", "b_E"], required=False)
        b_dec = self._get_tensor_attr(["b_dec", "decoder_bias", "b_D"], required=False)
        threshold = self._get_tensor_attr(["threshold", "thresholds", "jumprelu_threshold"], required=False)

        if W_enc.ndim != 2 or W_dec.ndim != 2:
            raise ValueError(f"Expected 2D W_enc/W_dec, got {tuple(W_enc.shape)} and {tuple(W_dec.shape)}")

        # Normalize common orientations to W_enc=[d_model,n_features], W_dec=[n_features,d_model].
        if W_enc.shape[0] == W_dec.shape[1] and W_enc.shape[1] == W_dec.shape[0]:
            pass
        elif W_enc.shape[1] == W_dec.shape[1]:
            W_enc = W_enc.T.contiguous()
        elif W_enc.shape[0] == W_dec.shape[0]:
            W_dec = W_dec.T.contiguous()
        else:
            raise ValueError(f"Could not infer SAELens W_enc/W_dec orientation: {tuple(W_enc.shape)}, {tuple(W_dec.shape)}")

        d_model = int(W_enc.shape[0])
        n_features = int(W_enc.shape[1])
        if W_dec.shape != (n_features, d_model):
            raise ValueError(f"Expected W_dec shape {(n_features, d_model)}, got {tuple(W_dec.shape)}")

        if b_enc is None:
            b_enc = torch.zeros(n_features)
        if b_dec is None:
            b_dec = torch.zeros(d_model)
        if threshold is None:
            threshold = torch.zeros(n_features)

        self.register_buffer("W_enc", W_enc.detach().contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.detach().contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.detach().reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.detach().reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.detach().reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))

        self.d_model = d_model
        self.n_features = n_features
        self.k = self._infer_k(fallback_k=fallback_k)

        self.saelens_sae.to(self.device)
        self.saelens_sae.eval()

    def _cfg_get(self, key, default=None):
        if isinstance(self.cfg_dict, dict) and key in self.cfg_dict:
            return self.cfg_dict[key]
        cfg = getattr(self.saelens_sae, "cfg", None)
        if isinstance(cfg, dict) and key in cfg:
            return cfg[key]
        if cfg is not None and hasattr(cfg, key):
            return getattr(cfg, key)
        return default

    def _get_tensor_attr(self, names, required=True):
        for name in names:
            if hasattr(self.saelens_sae, name):
                value = getattr(self.saelens_sae, name)
                if isinstance(value, torch.nn.Parameter):
                    value = value.data
                if torch.is_tensor(value):
                    return value.detach().cpu()
        if required:
            raise AttributeError(f"Could not find any of {names} on SAELens SAE.")
        return None

    def _infer_k(self, fallback_k=None):
        for key in ["k", "topk", "top_k", "topk_k"]:
            value = self._cfg_get(key, None)
            if value is not None:
                try:
                    return int(value)
                except Exception:
                    pass
        return int(fallback_k) if fallback_k is not None else None

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        fn = getattr(self.saelens_sae, "process_sae_in", None)
        if callable(fn):
            try:
                out = fn(x)
                if isinstance(out, tuple):
                    out = out[0]
                return out.to(device=self.device, dtype=self.dtype)
            except Exception:
                pass
        return x - self.b_dec

    def process_sae_out(self, x: torch.Tensor) -> torch.Tensor:
        fn = getattr(self.saelens_sae, "process_sae_out", None)
        if callable(fn):
            try:
                out = fn(x)
                if isinstance(out, tuple):
                    out = out[0]
                return out
            except Exception:
                pass
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def _topk_relu(self, pre: torch.Tensor) -> torch.Tensor:
        if self.k is None:
            return torch.relu(pre)
        k = min(int(self.k), pre.shape[-1])
        values, indices = torch.topk(torch.relu(pre), k=k, dim=-1)
        out = torch.zeros_like(pre)
        return out.scatter(dim=-1, index=indices, src=values)

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        pre = pre.to(device=self.device, dtype=self.dtype)
        fn = getattr(self.saelens_sae, "activation_fn", None)
        if callable(fn):
            try:
                out = fn(pre)
                if isinstance(out, tuple):
                    out = out[0]
                if torch.is_tensor(out) and out.shape == pre.shape:
                    return out.to(device=self.device, dtype=self.dtype)
            except Exception:
                pass
        if self.threshold.numel() == pre.shape[-1] and torch.any(self.threshold.detach().float() != 0):
            return torch.where(pre > self.threshold, torch.relu(pre), torch.zeros_like(pre))
        return self._topk_relu(pre)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


def load_qwen_batchtopk_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Qwen BatchTopK SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_files_only=SAE_LOCAL_FILES_ONLY,
        token=False,
    )
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    sae = QwenBatchTopKSAE(state, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"k={sae.k}, threshold={float(sae.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae


def load_gemma_scope_jumprelu_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Gemma Scope SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_files_only=SAE_LOCAL_FILES_ONLY,
        token=False,
    )
    with np.load(path, allow_pickle=False) as params_np:
        params = {k: torch.from_numpy(params_np[k]) for k in params_np.files}
    sae = GemmaScopeJumpReLUSAE(params, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"mean_threshold={float(sae.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae


def load_saelens_sae(release: str, sae_id: str, device=DEVICE, dtype=DTYPE, fallback_k=None):
    try:
        from sae_lens import SAE
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            "This Llama Scope variant requires sae-lens. Install it with: pip install sae-lens"
        ) from e

    print("Downloading/loading SAELens SAE:", release, sae_id)
    loaded = SAE.from_pretrained(release=release, sae_id=sae_id, device=str(device))
    if isinstance(loaded, tuple):
        if len(loaded) == 3:
            sae_obj, cfg_dict, sparsity = loaded
        elif len(loaded) == 2:
            sae_obj, cfg_dict = loaded
            sparsity = None
        else:
            sae_obj = loaded[0]
            cfg_dict = {}
            sparsity = None
    else:
        sae_obj = loaded
        cfg_dict = getattr(sae_obj, "cfg", {})
        sparsity = None

    sae = SAELensAdapter(
        saelens_sae=sae_obj,
        cfg_dict=cfg_dict,
        device=device,
        dtype=dtype,
        fallback_k=fallback_k,
    )
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"k={sae.k}, release={release}, sae_id={sae_id}"
    )
    return sae


def load_configured_sae():
    if SAE_BACKEND == "qwen_batchtopk":
        return load_qwen_batchtopk_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            device=TARGET_LAYER_DEVICE,
            dtype=DTYPE,
        )
    if SAE_BACKEND == "gemma_scope_npz":
        return load_gemma_scope_jumprelu_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            device=TARGET_LAYER_DEVICE,
            dtype=DTYPE,
        )
    if SAE_BACKEND == "saelens_llama_scope":
        return load_saelens_sae(
            release=SAE_LENS_RELEASE,
            sae_id=SAE_LENS_SAE_ID,
            device=TARGET_LAYER_DEVICE,
            dtype=DTYPE,
            fallback_k=SAE_TOPK_K,
        )
    raise ValueError(f"Unknown SAE_BACKEND={SAE_BACKEND!r}")


sae = load_configured_sae()
sae.eval()

_model_cfg = getattr(model.config, "text_config", model.config)
_model_hidden_size = int(getattr(_model_cfg, "hidden_size"))
_model_num_layers = int(getattr(_model_cfg, "num_hidden_layers"))
if _model_num_layers != int(EXPECTED_MODEL_NUM_LAYERS):
    raise ValueError(
        f"Expected {EXPECTED_MODEL_NUM_LAYERS} transformer layers, found {_model_num_layers}."
    )
if _model_hidden_size != int(EXPECTED_D_MODEL):
    raise ValueError(
        f"Expected model hidden size {EXPECTED_D_MODEL}, found {_model_hidden_size}."
    )
if int(sae.d_model) != int(EXPECTED_D_MODEL):
    raise ValueError(
        f"SAE/model width mismatch: sae.d_model={sae.d_model}, expected={EXPECTED_D_MODEL}."
    )
if int(sae.n_features) != int(EXPECTED_D_SAE):
    raise ValueError(
        f"Expected {EXPECTED_D_SAE} SAE features, found {sae.n_features}."
    )
print("Model/SAE compatibility checks passed.")


# ============================================================
# 2. SAE and model utilities
# ============================================================

def clear_all_hooks(model: torch.nn.Module):
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()


@torch.no_grad()
def sae_logits_before_jumprelu(
    sae,
    x: torch.Tensor,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
) -> torch.Tensor:
    """
    Return SAE encoder pre-activations before thresholding.

    x shape can be (..., d_model). Output shape is (..., n_features).
    """
    x_sae = x.to(device=sae.device, dtype=sae.dtype)
    hidden_pre = sae.encode_pre(x_sae)

    if filter_high_norm_tokens and x_sae.ndim >= 3:
        norms = x_sae.detach().float().norm(dim=-1)  # [B, S]
        med = norms.median(dim=1, keepdim=True).values.clamp_min(1e-6)
        high_norm_mask = norms > (HIGH_NORM_FILTER_MULTIPLIER * med)
        hidden_pre = hidden_pre.masked_fill(high_norm_mask.to(hidden_pre.device)[..., None], 0.0)

    return hidden_pre


@torch.no_grad()
def sae_activations(
    sae,
    x: torch.Tensor,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
) -> torch.Tensor:
    pre = sae_logits_before_jumprelu(
        sae=sae,
        x=x,
        filter_high_norm_tokens=filter_high_norm_tokens,
    )
    return sae.activation_fn(pre)


@torch.no_grad()
def get_layer_output(model, input_ids, attention_mask, layer_idx: int):
    """
    Run a forward pass and capture the output tensor of model.layers[layer_idx].
    Returns shape [batch, seq, hidden].
    """
    captured = {}

    def hook_fn(module, inputs, output):
        captured["x"] = output[0].detach() if isinstance(output, tuple) else output.detach()
        return output

    block = model.get_submodule(f"model.layers.{layer_idx}")
    handle = block.register_forward_hook(hook_fn)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        handle.remove()

    if "x" not in captured:
        raise RuntimeError(f"Failed to capture output from layer {layer_idx}")

    return captured["x"]


In [ ]:
# Variant audit (runs after the setup/model-loading cell)
print({
    "model": MODEL_NAME,
    "model_short_name": MODEL_SHORT_NAME,
    "target_layer": TARGET_LAYER,
    "sae_backend": SAE_BACKEND,
    "sae_repo": SAE_REPO_ID,
    "sae_file": SAE_FILENAME,
    "prompt_format": PROMPT_FORMAT,
    "block_score_mode": BLOCK_SCORE_MODE,
    "block_end_token": BLOCK_END_TOKEN,
    "high_norm_filter": FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
})
assert str(BLOCK_SCORE_MODE).lower() == "all"


In [ ]:

# ============================================================
# 3. AGNews 8-shot prompt generation
# ============================================================

DEFAULT_PROMPT_TEMPLATE_AGNEWS = """You are given news articles and their topics.
Valid topics: World, Sports, Business, Technology.
For each example, identify the correct topic.

Example 1:
News article:
[sent1]
Topic:
[answer1]

Example 2:
News article:
[sent2]
Topic:
[answer2]

Example 3:
News article:
[sent3]
Topic:
[answer3]

Example 4:
News article:
[sent4]
Topic:
[answer4]

Example 5:
News article:
[sent5]
Topic:
[answer5]

Example 6:
News article:
[sent6]
Topic:
[answer6]

Example 7:
News article:
[sent7]
Topic:
[answer7]

Example 8:
News article:
[sent8]
Topic:
[answer8]
"""


PROMPT_TEMPLATE_PATH = "prompt_agnews_8shot.txt"

if os.path.exists(PROMPT_TEMPLATE_PATH):
    with open(PROMPT_TEMPLATE_PATH, "r", encoding="utf-8") as f:
        prompt_template_agnews = f.read()
    print(f"Loaded prompt template from {PROMPT_TEMPLATE_PATH!r}")
else:
    prompt_template_agnews = DEFAULT_PROMPT_TEMPLATE_AGNEWS
    print(f"{PROMPT_TEMPLATE_PATH!r} not found. Using built-in default AGNews 8-shot template.")


EXAMPLE_HEADER_FOR_RENDER_RE = re.compile(r"(?m)^Example\s+\d+:?\s*$")
TOPIC_LINE_RE = re.compile(r"(?m)^Topic:\s*$")
TURN_PAIR_PROMPT_FORMATS = {"gemma_chat_turn_pairs", "llama_chat_turn_pairs"}
SINGLE_USER_PROMPT_FORMATS = {"gemma_chat_single_user_block", "qwen_chat_single_user_block", "llama_chat_single_user_block"}


def fill_agnews_prompt_body(prompt_template, train_data_agnews, indices, target_query_index):
    prompt = prompt_template

    for j in range(len(indices)):
        prompt = (
            prompt.replace(f"[sent{j+1}]", train_data_agnews[indices[j]]["text"])
                  .replace(f"[answer{j+1}]", label_to_words_agnews[train_data_agnews[indices[j]]["label"]])
        )

    prompt = (
        prompt.replace("[sent8]", train_data_agnews[target_query_index]["text"])
              .replace("[answer8]", label_to_words_agnews[train_data_agnews[target_query_index]["label"]])
    )

    return prompt


def _split_agnews_prompt_body_into_qwen_user_messages(prompt_body: str, expected_num_blocks: int = NUM_BLOCKS_TOTAL):
    """
    Split an AGNews prompt body into Qwen user-message contents.

    Each example block is explicitly terminated by BLOCK_END_TOKEN. The first
    user message contains the global task instruction plus Example 1; later user
    messages contain exactly one example each.
    """
    end_token = globals().get("BLOCK_END_TOKEN", "<|im_end|>")
    if not end_token:
        raise ValueError("Qwen chat block rendering requires a non-empty BLOCK_END_TOKEN.")

    matches = list(EXAMPLE_HEADER_FOR_RENDER_RE.finditer(prompt_body))
    if len(matches) != expected_num_blocks:
        raise ValueError(
            f"Expected {expected_num_blocks} Example headers when splitting Qwen user messages, "
            f"but found {len(matches)}."
        )

    prefix = prompt_body[:matches[0].start()].rstrip()
    contents = []

    for i, m in enumerate(matches):
        block_start = m.start()
        block_end = matches[i + 1].start() if i + 1 < len(matches) else len(prompt_body)
        block = prompt_body[block_start:block_end].strip()
        if not block.endswith(end_token):
            block = block.rstrip() + "\n" + end_token

        if i == 0 and prefix:
            content = prefix + "\n\n" + block
        else:
            content = block
        contents.append(content.rstrip())

    return contents


def _split_agnews_prompt_body_into_turn_pairs(prompt_body: str, expected_num_blocks: int = NUM_BLOCKS_TOTAL):
    """
    Split the filled AGNews prompt body into instruction prefix plus one
    (user_content, label) pair per example.

    This is used for Gemma/Llama instruction-tuned variants, where each
    demonstration should be a proper user -> model/assistant turn pair:
      user: task/example text ending at Topic:
      model/assistant: label
    """
    matches = list(EXAMPLE_HEADER_FOR_RENDER_RE.finditer(prompt_body))
    if len(matches) != expected_num_blocks:
        raise ValueError(
            f"Expected {expected_num_blocks} Example headers when splitting turn pairs, "
            f"but found {len(matches)}."
        )

    prefix = prompt_body[:matches[0].start()].strip()
    pairs = []

    for i, m in enumerate(matches):
        block_start = m.start()
        block_end = matches[i + 1].start() if i + 1 < len(matches) else len(prompt_body)
        block = prompt_body[block_start:block_end].strip()

        topic_matches = list(TOPIC_LINE_RE.finditer(block))
        if not topic_matches:
            raise ValueError(f"Could not find Topic: line in block {i + 1}:\n{block[:500]}")
        topic_m = topic_matches[-1]
        user_text = block[:topic_m.end()].rstrip()
        label_text = block[topic_m.end():].strip()
        if not label_text:
            raise ValueError(f"Empty label after Topic: line in block {i + 1}.")

        if i == 0 and prefix:
            user_text = prefix + "\n\n" + user_text
        pairs.append((user_text, label_text))

    return pairs


def append_block_end_tokens_to_agnews_prompt_body(prompt_body: str, expected_num_blocks: int = NUM_BLOCKS_TOTAL) -> str:
    """
    Insert an explicit block terminator after each Example block when the prompt
    format needs one. Raw/base formats intentionally do not inject per-block
    special tokens; their block boundaries are recovered from Example headers.
    """
    end_token = globals().get("BLOCK_END_TOKEN", None)
    prompt_format = globals().get("PROMPT_FORMAT", "raw")

    if prompt_format in TURN_PAIR_PROMPT_FORMATS or prompt_format in SINGLE_USER_PROMPT_FORMATS or not end_token:
        return prompt_body.rstrip() + "\n"

    matches = list(EXAMPLE_HEADER_FOR_RENDER_RE.finditer(prompt_body))
    if len(matches) != expected_num_blocks:
        raise ValueError(
            f"Expected {expected_num_blocks} Example headers before adding block end tokens, "
            f"but found {len(matches)}."
        )

    pieces = []
    for i, m in enumerate(matches):
        block_start = m.start()
        block_end = matches[i + 1].start() if i + 1 < len(matches) else len(prompt_body)
        block = prompt_body[block_start:block_end].rstrip()
        if not block.endswith(end_token):
            block = block + "\n" + end_token
        pieces.append(block)

    prefix = prompt_body[:matches[0].start()].rstrip()
    if prefix:
        return prefix + "\n\n" + "\n\n".join(pieces) + "\n"
    return "\n\n".join(pieces) + "\n"


def render_qwen_chat_user_blocks(prompt_body: str) -> str:
    user_contents = _split_agnews_prompt_body_into_qwen_user_messages(
        prompt_body,
        expected_num_blocks=NUM_BLOCKS_TOTAL,
    )
    parts = [f"<|im_start|>system\n{QWEN_SYSTEM_PROMPT}<|im_end|>"]
    for content in user_contents:
        parts.append(f"<|im_start|>user\n{content.rstrip()}")
    return "\n".join(parts) + "\n"


def render_qwen_chat_single_user_block(prompt_body: str) -> str:
    """
    Render the full AGNews 8-shot body as exactly one Qwen user turn.

    This is the Qwen analogue of gemma_chat_single_user_block: all
    demonstrations and labels stay inside one user message, with no
    per-demonstration user/assistant wrapping. When supported by the tokenizer,
    enable_thinking=False is passed to keep the Qwen3 chat template in its
    non-thinking instruction-following mode. No assistant-generation header is
    added, because this notebook measures activations inside the prompt itself.
    """
    matches = list(EXAMPLE_HEADER_FOR_RENDER_RE.finditer(prompt_body))
    if len(matches) != NUM_BLOCKS_TOTAL:
        raise ValueError(
            f"Expected {NUM_BLOCKS_TOTAL} Example headers in single-user Qwen prompt, "
            f"but found {len(matches)}."
        )

    body = prompt_body.rstrip()
    messages = []
    if QWEN_SYSTEM_PROMPT:
        messages.append({"role": "system", "content": QWEN_SYSTEM_PROMPT})
    messages.append({"role": "user", "content": body})

    kwargs = dict(globals().get("QWEN_CHAT_TEMPLATE_KWARGS", {}))
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            **kwargs,
        )
    except TypeError:
        # Older tokenizer versions may reject enable_thinking. Because no
        # assistant-generation header is added, this fallback still renders a
        # single user prompt without inserting thinking text.
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        # Final fallback for environments where apply_chat_template is
        # unavailable or misconfigured.
        prefix = (
            f"<|im_start|>system\n{QWEN_SYSTEM_PROMPT}<|im_end|>\n"
            if QWEN_SYSTEM_PROMPT
            else ""
        )
        return prefix + f"<|im_start|>user\n{body}<|im_end|>\n"


def render_gemma_chat_single_user_block(prompt_body: str) -> str:
    """
    Render the full AGNews 8-shot body as exactly one Gemma user turn.

    Unlike gemma_chat_turn_pairs, this leaves all demonstration labels inside
    the same user message and does not create per-demonstration model turns.
    The LFH source/target block analysis below is otherwise unchanged.
    """
    matches = list(EXAMPLE_HEADER_FOR_RENDER_RE.finditer(prompt_body))
    if len(matches) != NUM_BLOCKS_TOTAL:
        raise ValueError(
            f"Expected {NUM_BLOCKS_TOTAL} Example headers in single-user Gemma prompt, "
            f"but found {len(matches)}."
        )

    bos = PROMPT_BOS_TOKEN or "<bos>"
    body = prompt_body.rstrip()
    return bos + "<start_of_turn>user\n" + body + "<end_of_turn>\n"


def render_gemma_chat_turn_pairs(prompt_body: str) -> str:
    pairs = _split_agnews_prompt_body_into_turn_pairs(
        prompt_body,
        expected_num_blocks=NUM_BLOCKS_TOTAL,
    )
    bos = PROMPT_BOS_TOKEN or "<bos>"
    parts = []
    for user_text, label_text in pairs:
        parts.append(
            "<start_of_turn>user\n"
            + user_text.rstrip()
            + "<end_of_turn>\n"
            + "<start_of_turn>model\n"
            + label_text.strip()
            + "<end_of_turn>"
        )
    # Gemma chat prompts are conventionally rendered as <bos><start_of_turn>user,
    # without an extra newline token between <bos> and the first turn marker.
    return bos + "\n".join(parts) + "\n"



def render_llama_chat_single_user_block(prompt_body: str) -> str:
    """Render all eight demonstrations inside exactly one Llama user turn."""
    matches = list(EXAMPLE_HEADER_FOR_RENDER_RE.finditer(prompt_body))
    if len(matches) != NUM_BLOCKS_TOTAL:
        raise ValueError(
            f"Expected {NUM_BLOCKS_TOTAL} Example headers in single-user Llama prompt, "
            f"but found {len(matches)}."
        )

    bos = PROMPT_BOS_TOKEN or "<|begin_of_text|>"
    body = prompt_body.rstrip()
    parts = [bos]
    if LLAMA_SYSTEM_PROMPT:
        parts.append(
            "<|start_header_id|>system<|end_header_id|>\n\n"
            + LLAMA_SYSTEM_PROMPT
            + "<|eot_id|>"
        )
    parts.append(
        "<|start_header_id|>user<|end_header_id|>\n\n"
        + body
        + "<|eot_id|>"
    )
    return "".join(parts) + "\n"

def render_llama_chat_turn_pairs(prompt_body: str) -> str:
    pairs = _split_agnews_prompt_body_into_turn_pairs(
        prompt_body,
        expected_num_blocks=NUM_BLOCKS_TOTAL,
    )
    bos = PROMPT_BOS_TOKEN or "<|begin_of_text|>"
    parts = [
        bos
        + "<|start_header_id|>system<|end_header_id|>\n\n"
        + LLAMA_SYSTEM_PROMPT
        + "<|eot_id|>"
    ]
    for user_text, label_text in pairs:
        parts.append(
            "<|start_header_id|>user<|end_header_id|>\n\n"
            + user_text.rstrip()
            + "<|eot_id|>"
            + "<|start_header_id|>assistant<|end_header_id|>\n\n"
            + label_text.strip()
            + "<|eot_id|>"
        )
    return "\n".join(parts) + "\n"


def render_raw_prompt(prompt_body: str) -> str:
    body = prompt_body.rstrip() + "\n"
    bos = PROMPT_BOS_TOKEN or ""
    return bos + body


def render_agnews_prompt_from_body(prompt_body: str) -> str:
    prompt_format = globals().get("PROMPT_FORMAT", "raw")
    if prompt_format == "qwen_chat_user_blocks":
        return render_qwen_chat_user_blocks(prompt_body)
    if prompt_format == "qwen_chat_single_user_block":
        return render_qwen_chat_single_user_block(prompt_body)
    if prompt_format == "gemma_chat_single_user_block":
        return render_gemma_chat_single_user_block(prompt_body)
    if prompt_format == "gemma_chat_turn_pairs":
        return render_gemma_chat_turn_pairs(prompt_body)
    if prompt_format == "llama_chat_single_user_block":
        return render_llama_chat_single_user_block(prompt_body)
    if prompt_format == "llama_chat_turn_pairs":
        return render_llama_chat_turn_pairs(prompt_body)
    if prompt_format == "raw":
        return render_raw_prompt(prompt_body)
    raise ValueError(f"Unknown PROMPT_FORMAT={prompt_format!r}")


def build_prompts_all_agnews(
    train_data_agnews,
    prompt_template_agnews,
    num_test_query=25,
    num_previous=25,
    varied_block_num=None,
    target_block_num=None,
    test_query_indices=None,
):
    """
    For each target query:
      - keep the target block fixed, currently Example 8;
      - keep all non-source context demonstrations fixed;
      - vary one controllable source demonstration across prompt variants.

    Default source block is Example 7, reproducing the original adjacent
    previous-to-target manipulation. Set varied_block_num=6 or 5 to test
    whether the source-to-target LFH correlation decays with distance.
    """
    varied_block_num = int(LFH_SOURCE_BLOCK_NUM if varied_block_num is None else varied_block_num)
    target_block_num = int(LFH_TARGET_BLOCK_NUM if target_block_num is None else target_block_num)

    if target_block_num != NUM_BLOCKS_TOTAL:
        raise NotImplementedError(
            "This AGNews template fills Example 8 as the fixed target. "
            f"Got target_block_num={target_block_num}."
        )
    if not (1 <= varied_block_num < target_block_num):
        raise ValueError(
            f"varied_block_num must be before the target block {target_block_num}; "
            f"got {varied_block_num}."
        )

    source_pos = block_num_to_index(varied_block_num, total_blocks=NUM_BLOCKS_TOTAL)
    num_context_blocks = NUM_BLOCKS_TOTAL - 1

    prompts_all_agnews = {i: [] for i in range(num_test_query)}
    prompt_bodies_all_agnews = {i: [] for i in range(num_test_query)}
    varied_source_indices_by_query = {}

    # Distinct target queries. If supplied, reuse the exact same target examples
    # across locality controls so source-block comparisons are matched.
    if test_query_indices is None:
        set_seed(42)
        test_query_indices = random.sample(range(len(train_data_agnews)), num_test_query)
    else:
        test_query_indices = list(test_query_indices)
        if len(test_query_indices) < num_test_query:
            raise ValueError(
                f"Need at least {num_test_query} target query indices, got {len(test_query_indices)}."
            )
        test_query_indices = test_query_indices[:num_test_query]

    for i in range(num_test_query):
        if i % 10 == 0:
            print(
                f"building prompts for query {i}; "
                f"varied source block={varied_block_num}, fixed target block={target_block_num}"
            )

        used_varied_source_indices = set()
        varied_source_indices_by_query[i] = []
        fixed_context_indices = set()
        indices = None

        for k in range(num_previous):
            if k == 0:
                set_seed(i)
                while True:
                    # indices fills Examples 1..7; Example 8 is the fixed target.
                    indices = random.sample(range(len(train_data_agnews)), num_context_blocks)
                    if test_query_indices[i] not in indices:
                        fixed_context_indices = {
                            idx for pos, idx in enumerate(indices)
                            if pos != source_pos
                        }
                        used_varied_source_indices.add(indices[source_pos])
                        varied_source_indices_by_query[i].append(indices[source_pos])
                        break
            else:
                while True:
                    indices_t = random.sample(range(len(train_data_agnews)), 1)[0]
                    if (
                        indices_t != test_query_indices[i]
                        and indices_t not in fixed_context_indices
                        and indices_t not in used_varied_source_indices
                    ):
                        indices[source_pos] = indices_t
                        used_varied_source_indices.add(indices_t)
                        varied_source_indices_by_query[i].append(indices_t)
                        break

            prompt_body = fill_agnews_prompt_body(
                prompt_template=prompt_template_agnews,
                train_data_agnews=train_data_agnews,
                indices=indices,
                target_query_index=test_query_indices[i],
            )
            prompt_body = append_block_end_tokens_to_agnews_prompt_body(
                prompt_body,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            prompt = render_agnews_prompt_from_body(prompt_body)

            prompt_bodies_all_agnews[i].append(prompt_body)
            prompts_all_agnews[i].append(prompt)

    return prompts_all_agnews, prompt_bodies_all_agnews, test_query_indices, varied_source_indices_by_query


prompts_all_agnews, prompt_bodies_all_agnews, test_query_indices, varied_source_indices_by_query = build_prompts_all_agnews(
    train_data_agnews=train_data_agnews,
    prompt_template_agnews=prompt_template_agnews,
    num_test_query=NUM_TEST_QUERY,
    num_previous=NUM_PREVIOUS,
    varied_block_num=LFH_SOURCE_BLOCK_NUM,
    target_block_num=LFH_TARGET_BLOCK_NUM,
)

# Backward-compatible alias for older downstream scratch code.
varied_previous_indices_by_query = varied_source_indices_by_query

assert len(prompts_all_agnews) == NUM_TEST_QUERY
assert len(prompts_all_agnews[0]) == NUM_PREVIOUS

print("MODEL_SHORT_NAME:", MODEL_SHORT_NAME)
print("PROMPT_FORMAT:", PROMPT_FORMAT)
print("LFH source block:", LFH_SOURCE_BLOCK_NUM)
print("LFH target block:", LFH_TARGET_BLOCK_NUM)
print("Example rendered prompt head:")
print(prompts_all_agnews[0][0][:1200])

if PROMPT_FORMAT == "gemma_chat_single_user_block":
    _sample_prompt = prompts_all_agnews[0][0]
    assert _sample_prompt.startswith((PROMPT_BOS_TOKEN or "<bos>") + "<start_of_turn>user\n")
    assert _sample_prompt.count("<start_of_turn>user") == 1
    assert _sample_prompt.count("<start_of_turn>model") == 0
    assert _sample_prompt.count("<end_of_turn>") == 1
    assert len(EXAMPLE_HEADER_FOR_RENDER_RE.findall(_sample_prompt)) == NUM_BLOCKS_TOTAL
    print("Single-user Gemma prompt validation passed: one user turn, zero model turns, one turn terminator.")

if PROMPT_FORMAT == "qwen_chat_single_user_block":
    _sample_prompt = prompts_all_agnews[0][0]
    assert _sample_prompt.count("<|im_start|>user") == 1
    assert _sample_prompt.count("<|im_start|>assistant") == 0
    _expected_qwen_ends = 1 + int(bool(QWEN_SYSTEM_PROMPT))
    assert _sample_prompt.count("<|im_end|>") == _expected_qwen_ends
    assert len(EXAMPLE_HEADER_FOR_RENDER_RE.findall(_sample_prompt)) == NUM_BLOCKS_TOTAL
    print(
        "Single-user Qwen prompt validation passed: one user turn, zero assistant turns, "
        f"{_expected_qwen_ends} turn terminator(s)."
    )

if PROMPT_FORMAT == "llama_chat_single_user_block":
    _sample_prompt = prompts_all_agnews[0][0]
    assert _sample_prompt.startswith(PROMPT_BOS_TOKEN or "<|begin_of_text|>")
    assert _sample_prompt.count("<|start_header_id|>user<|end_header_id|>") == 1
    assert _sample_prompt.count("<|start_header_id|>assistant<|end_header_id|>") == 0
    _expected_llama_ends = 1 + int(bool(LLAMA_SYSTEM_PROMPT))
    assert _sample_prompt.count("<|eot_id|>") == _expected_llama_ends
    assert len(EXAMPLE_HEADER_FOR_RENDER_RE.findall(_sample_prompt)) == NUM_BLOCKS_TOTAL
    print(
        "Single-user Llama prompt validation passed: one user turn, zero assistant turns, "
        f"{_expected_llama_ends} turn terminator(s)."
    )


In [ ]:

# ============================================================
# 4. Block token-selection helpers with selectable scoring
# ============================================================

# Matches each example block header. The measured content starts immediately
# after the header, but the scored tokens are controlled by BLOCK_SCORE_MODE.
EXAMPLE_HEADER_RE = re.compile(
    r"(?m)^Example\s+\d+:?\s*\r?\n(?:Article|News article):\s*\r?\n",
    flags=re.IGNORECASE,
)


class TokenSelection:
    """
    Token selection for one exemplar block.

    The object behaves like a legacy (start, end) span when unpacked, so older
    plotting/logging code can still do ``s, e = selection``. Actual analyses use
    ``selection.indices`` through token_selection_to_* helpers, which allows
    non-contiguous choices such as [-1, -3] if desired.
    """

    def __init__(self, indices, *, full_start=None, full_end=None, label=None):
        indices = tuple(sorted(dict.fromkeys(int(i) for i in indices)))
        if len(indices) == 0:
            raise ValueError("TokenSelection requires at least one token index.")
        self.indices = indices
        self.start = int(indices[0])
        self.end = int(indices[-1]) + 1
        self.full_start = int(full_start) if full_start is not None else self.start
        self.full_end = int(full_end) if full_end is not None else self.end
        self.label = label

    def __iter__(self):
        # Backward-compatible unpacking: s, e = selection
        yield self.start
        yield self.end

    def __getitem__(self, item):
        return (self.start, self.end)[item]

    def __len__(self):
        # Legacy span-like length. Use n_selected for number of selected tokens.
        return 2

    @property
    def n_selected(self):
        return len(self.indices)

    @property
    def is_contiguous(self):
        return self.n_selected == (self.end - self.start)

    def __repr__(self):
        idx = list(self.indices)
        if len(idx) > 10:
            idx_repr = f"{idx[:5]} ... {idx[-5:]}"
        else:
            idx_repr = repr(idx)
        return (
            f"TokenSelection(start={self.start}, end={self.end}, "
            f"n_selected={self.n_selected}, contiguous={self.is_contiguous}, "
            f"indices={idx_repr})"
        )


def token_selection_to_indices(selection):
    """Return selected token indices as a Python list."""
    if isinstance(selection, TokenSelection):
        return list(selection.indices)
    s, e = selection
    return list(range(int(s), int(e)))


def token_selection_to_torch_indices(selection, device=None):
    """Return selected token indices as a torch.LongTensor."""
    return torch.tensor(token_selection_to_indices(selection), dtype=torch.long, device=device)


def token_selection_bounds(selection):
    """Return the legacy [start, end) envelope for logging/plotting."""
    if isinstance(selection, TokenSelection):
        return int(selection.start), int(selection.end)
    s, e = selection
    return int(s), int(e)


def token_selection_mean(x: torch.Tensor, selection, dim: int = 0):
    """Mean over selected token positions along ``dim``."""
    idx = token_selection_to_torch_indices(selection, device=x.device)
    return x.index_select(dim, idx).mean(dim=dim)


def _normalize_block_score_mode(mode):
    """Normalize user-facing BLOCK_SCORE_MODE strings."""
    mode_str = str(mode).strip()
    mode_key = mode_str.lower().replace("-", "_").replace(" ", "_")

    # User-facing full-exemplar mode. "content" is kept as a backward-compatible alias.
    if mode_key in {"all", "all_tokens", "full", "full_block", "content"}:
        return "all"

    # New exact relative-position mode.
    if mode_key in {
        "selected_offsets_from_end",
        "select_offsets_from_end",
        "token_offsets_from_end",
        "selected_token_offsets_from_end",
        "selected_tokens_from_end",
        "selected_tokens",
        "offsets_from_end",
    }:
        return "selected_offsets_from_end"

    # Legacy last-K family of modes, including "last_k_with_end_of_turn".
    if mode_key.startswith("last_k"):
        return "last_k"

    raise ValueError(
        f"Unknown BLOCK_SCORE_MODE={mode!r}. Supported modes are 'selected_offsets_from_end', 'All', and legacy last_k* modes."
    )


def _normalize_offsets_from_end(offsets):
    """
    Validate exact token offsets from the end of each exemplar span.

    Negative offsets count backward from the exemplar end:
      -1 = final token, -2 = second-to-final token, etc.
    """
    if offsets is None:
        return None
    if isinstance(offsets, (int, np.integer)):
        offsets = [int(offsets)]
    offsets = [int(x) for x in list(offsets)]
    if len(offsets) == 0:
        raise ValueError("BLOCK_TOKEN_OFFSETS_FROM_END must be non-empty when using selected-offset mode.")
    if any(x >= 0 for x in offsets):
        raise ValueError(
            "BLOCK_TOKEN_OFFSETS_FROM_END expects negative integers only: "
            "-1 is the final token, -2 is the second-to-final token, etc."
        )
    return offsets


def _legacy_last_k_to_offsets(last_k_tokens: int):
    last_k_tokens = int(last_k_tokens)
    if last_k_tokens <= 0:
        raise ValueError(f"BLOCK_LAST_K_TOKENS must be positive for legacy last-K modes, got {last_k_tokens}.")
    return [-i for i in range(1, last_k_tokens + 1)]


def _get_block_scoring_config():
    """Read scoring options, with safe defaults for older cached notebooks."""
    raw_mode = globals().get("BLOCK_SCORE_MODE", "selected_offsets_from_end")
    mode = _normalize_block_score_mode(raw_mode)
    end_token = globals().get("BLOCK_END_TOKEN", None)

    if mode == "all":
        return mode, end_token, None

    if mode == "last_k":
        # Strict backward compatibility: legacy last_k* modes are controlled by
        # BLOCK_LAST_K_TOKENS, irrespective of BLOCK_TOKEN_OFFSETS_FROM_END.
        offsets = _legacy_last_k_to_offsets(globals().get("BLOCK_LAST_K_TOKENS", 8))
    else:
        explicit_offsets = globals().get("BLOCK_TOKEN_OFFSETS_FROM_END", None)
        if explicit_offsets is not None:
            offsets = _normalize_offsets_from_end(explicit_offsets)
        else:
            # Safe fallback for notebooks configured before BLOCK_TOKEN_OFFSETS_FROM_END existed.
            offsets = _legacy_last_k_to_offsets(globals().get("BLOCK_LAST_K_TOKENS", 8))

    if len(offsets) == 0:
        raise ValueError("No token offsets selected for block scoring.")

    return "selected_offsets_from_end", end_token, offsets


def find_example_block_char_spans(prompt: str, expected_num_blocks: int):
    """
    Find character spans of all Example blocks.

    Each span starts immediately AFTER the block header. If BLOCK_END_TOKEN is
    non-empty, the span ends at the final occurrence of that terminator before
    the next Example header. This is important for turn-pair prompts, where a
    user turn and a model/assistant turn can both contain terminators and the
    block should end at the model/assistant turn terminator. In the single-user
    Gemma prompt, only the final block sees the single user-turn terminator;
    earlier block boundaries are recovered from the next Example header.
    """
    _, end_token, _ = _get_block_scoring_config()

    matches = list(EXAMPLE_HEADER_RE.finditer(prompt))
    if len(matches) != expected_num_blocks:
        raise ValueError(
            f"Expected {expected_num_blocks} Example blocks, but found {len(matches)}.\n"
            f"Prompt head:\n{prompt[:800]}"
        )

    spans = []
    for i, m in enumerate(matches):
        start = m.end()
        next_header_start = matches[i + 1].start() if i + 1 < len(matches) else len(prompt)

        if end_token:
            # Use the final terminator before the next Example header. For
            # Gemma/Llama turn-pair prompts this captures the assistant/model
            # label turn rather than the user turn.
            end_token_start = prompt.rfind(end_token, start, next_header_start)
            if end_token_start >= 0:
                end = end_token_start + len(end_token)
            else:
                end = next_header_start
        else:
            end = next_header_start
        spans.append((start, end))

    return spans


def char_span_to_token_span(offset_mapping, char_start: int, char_end: int):
    """
    Convert character span [char_start, char_end) into token span [tok_start, tok_end).
    """
    token_ids = []
    for tok_idx, (s, e) in enumerate(offset_mapping):
        # Skip padding / special tokens / empty offsets.
        if s == 0 and e == 0:
            continue
        if e > char_start and s < char_end:
            token_ids.append(tok_idx)

    if not token_ids:
        raise ValueError(f"Could not map char span [{char_start}, {char_end}) to tokens.")

    return token_ids[0], token_ids[-1] + 1


def _get_inline_end_token_ids(end_token: str | None):
    """Token IDs for the explicit inline block terminator."""
    if not end_token:
        return []
    ids = tokenizer.encode(end_token, add_special_tokens=False)
    if not ids:
        raise ValueError(f"Could not tokenize BLOCK_END_TOKEN={end_token!r}")
    return [int(x) for x in ids]


def _ensure_span_includes_inline_end_token(prompt: str, token_span, end_token: str | None):
    """
    Some fast tokenizers report offset (0, 0) for special tokens such as
    <|im_end|> or <end_of_turn>. In that case char-offset mapping can end just
    before the explicit terminator. This helper checks token IDs and extends the
    span by the terminator sequence if needed.
    """
    if not end_token:
        return token_span

    tok_start, tok_end = token_span
    end_ids = _get_inline_end_token_ids(end_token)
    if not end_ids:
        return token_span

    enc_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    if isinstance(enc_ids[0], list):
        enc_ids = enc_ids[0]
    enc_ids = [int(x) for x in enc_ids]
    L = len(end_ids)

    if tok_end - tok_start >= L and enc_ids[tok_end - L:tok_end] == end_ids:
        return tok_start, tok_end

    if enc_ids[tok_end:tok_end + L] == end_ids:
        return tok_start, tok_end + L

    for shift in range(1, 5):
        if enc_ids[tok_end + shift:tok_end + shift + L] == end_ids:
            return tok_start, tok_end + shift + L

    return tok_start, tok_end


def _select_token_offsets_from_end(tok_start: int, tok_end: int, offsets_from_end):
    """
    Convert negative offsets from a full exemplar span into exact token indices.

    Offsets that run before tok_start are ignored. Thus [-1, -2, -3] behaves
    like the old last-3 selection, but automatically clips on very short spans.
    """
    if tok_end <= tok_start:
        raise ValueError(f"Empty token span [{tok_start}, {tok_end}) cannot be scored.")

    selected = []
    for offset in offsets_from_end:
        idx = int(tok_end) + int(offset)  # offset is negative: -1 -> tok_end - 1
        if tok_start <= idx < tok_end:
            selected.append(idx)

    if len(selected) == 0:
        raise ValueError(
            f"No selected offsets {offsets_from_end} fall inside token span [{tok_start}, {tok_end})."
        )

    return selected


def get_example_block_token_spans(prompt: str, offset_mapping, expected_num_blocks: int):
    """
    Return token selections for all Example blocks in the final rendered prompt.

    If a block terminator is configured, the terminator is included in the full
    exemplar span before token selection is applied. If not, the full span ends
    before the next Example header.

    Supported BLOCK_SCORE_MODE values:
      - "selected_offsets_from_end": select exact relative token positions from
        BLOCK_TOKEN_OFFSETS_FROM_END. For example, [-1] selects only the final
        token and [-2] selects only the second-to-final token.
      - legacy "last_k...": if BLOCK_TOKEN_OFFSETS_FROM_END is None, this is
        converted to [-1, -2, ..., -K] using BLOCK_LAST_K_TOKENS.
      - "All": return all tokens in each exemplar span.

    The function name is kept for backward compatibility; returned objects are
    TokenSelection instances that can still be unpacked as legacy (start, end)
    spans for logging/plotting.
    """
    mode, end_token, offsets_from_end = _get_block_scoring_config()
    char_spans = find_example_block_char_spans(
        prompt,
        expected_num_blocks=expected_num_blocks,
    )

    full_token_spans = [
        _ensure_span_includes_inline_end_token(
            prompt,
            char_span_to_token_span(offset_mapping, s, e),
            end_token=end_token,
        )
        for (s, e) in char_spans
    ]

    selections = []
    for block_idx, (tok_start, tok_end) in enumerate(full_token_spans, start=1):
        if mode == "all":
            indices = list(range(tok_start, tok_end))
        elif mode == "selected_offsets_from_end":
            indices = _select_token_offsets_from_end(tok_start, tok_end, offsets_from_end)
        else:
            # _get_block_scoring_config should already have validated the mode.
            raise ValueError(f"Unknown normalized BLOCK_SCORE_MODE={mode!r}")

        selections.append(
            TokenSelection(
                indices,
                full_start=tok_start,
                full_end=tok_end,
                label=f"Example {block_idx}",
            )
        )

    return selections


def tokenize_prompt_for_offsets(prompt: str):
    """
    Prompts are already rendered with the intended BOS/chat special tokens, so
    add_special_tokens is disabled by default to avoid duplicating them.
    """
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=bool(globals().get("TOKENIZE_ADD_SPECIAL_TOKENS", False)),
        return_offsets_mapping=True,
        padding=False,
        truncation=False,
    )
    return enc


# Sanity check on one prompt.
_test_prompt = prompts_all_agnews[0][0]
_test_enc = tokenize_prompt_for_offsets(_test_prompt)
_test_offsets = _test_enc["offset_mapping"][0].tolist()
_test_spans = get_example_block_token_spans(
    prompt=_test_prompt,
    offset_mapping=_test_offsets,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
)
print("Example block token selections under current scoring mode:")
print("  PROMPT_FORMAT:", globals().get("PROMPT_FORMAT", "raw"))
print("  BLOCK_SCORE_MODE:", globals().get("BLOCK_SCORE_MODE", "selected_offsets_from_end"))
print("  BLOCK_END_TOKEN:", globals().get("BLOCK_END_TOKEN", None))
_printed_mode, _, _printed_offsets = _get_block_scoring_config()
if _printed_mode == "all":
    print("  BLOCK_TOKEN_OFFSETS_FROM_END: not used because BLOCK_SCORE_MODE='All'")
else:
    print("  BLOCK_TOKEN_OFFSETS_FROM_END:", _printed_offsets)
for i, sel in enumerate(_test_spans, start=1):
    s, e = token_selection_bounds(sel)
    idx = token_selection_to_indices(sel)
    toks = _test_enc["input_ids"][0, idx]
    print(
        f"  Example {i}: envelope=[{s}, {e}) selected_n={len(idx)} "
        f"contiguous={sel.is_contiguous} decoded={tokenizer.decode(toks)!r}"
    )


In [ ]:
print(prompts_all_agnews[0][0])

In [ ]:

# ============================================================
# 5. First pass: select features by source-block activation frequency
# ============================================================

@torch.no_grad()
def first_pass_collect_previous_feature_frequency(
    model,
    tokenizer,
    sae,
    prompts_all_agnews,
    layer_idx: int,
    expected_num_blocks: int,
    source_block_index: int | None = None,
):
    """
    For every prompt:
      - capture layer output,
      - encode with SAE,
      - compute active-token frequency on the varied source block.

    The function name is kept for backward compatibility with earlier notebook
    versions, where the source block was always the immediately previous block.

    Returns:
      mean_prev_freq: np.ndarray [num_features]
      used_prompt_count: int
    """
    if source_block_index is None:
        source_block_index = int(LFH_SOURCE_BLOCK_INDEX)

    num_features = sae.n_features
    freq_sum = torch.zeros(num_features, dtype=torch.float64)
    used_prompt_count = 0

    for query_idx in range(len(prompts_all_agnews)):
        for variant_idx in range(len(prompts_all_agnews[query_idx])):
            prompt = prompts_all_agnews[query_idx][variant_idx]

            enc = tokenize_prompt_for_offsets(prompt)
            offset_mapping = enc["offset_mapping"][0].tolist()
            input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
            attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

            block_spans = get_example_block_token_spans(
                prompt=prompt,
                offset_mapping=offset_mapping,
                expected_num_blocks=expected_num_blocks,
            )

            source_span = block_spans[source_block_index]

            x = get_layer_output(
                model=model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                layer_idx=layer_idx,
            )  # [1, seq, hidden]

            sae_encoded = sae_activations(sae, x)[0].float().cpu()  # [seq, features]
            source_idx = token_selection_to_torch_indices(source_span, device=sae_encoded.device)
            source_freq = (sae_encoded.index_select(0, source_idx) > 0).float().mean(dim=0).double()

            freq_sum += source_freq
            used_prompt_count += 1

        print(f"first pass done query {query_idx + 1}/{len(prompts_all_agnews)}")

    mean_prev_freq = (freq_sum / max(used_prompt_count, 1)).numpy()
    return mean_prev_freq, used_prompt_count


mean_prev_freq, used_prompt_count = first_pass_collect_previous_feature_frequency(
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=TARGET_LAYER,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
    source_block_index=LFH_SOURCE_BLOCK_INDEX,
)

freq_over_threshold = {
    feat: float(freq)
    for feat, freq in enumerate(mean_prev_freq)
    if freq >= FREQ_THRESHOLD
}

print(f"Used {used_prompt_count} prompts.")
print(
    f"Selected {len(freq_over_threshold)} features with mean source-block "
    f"frequency >= {FREQ_THRESHOLD:.2f}"
)
print(f"Source block: Example {LFH_SOURCE_BLOCK_NUM}; target block: Example {LFH_TARGET_BLOCK_NUM}")


# ============================================================
# 6. Second pass: collect source/target mean SAE pre-activations
# ============================================================

@torch.no_grad()
def second_pass_collect_prev_tgt_logits(
    model,
    tokenizer,
    sae,
    prompts_all_agnews,
    layer_idx: int,
    expected_num_blocks: int,
    selected_feature_ids: list[int],
    source_block_index: int | None = None,
    target_block_index: int | None = None,
):
    """
    Collect for every prompt:
      - mean SAE pre-activation on the varied source block;
      - mean SAE pre-activation on the fixed target block.

    The returned variable names retain prev/tgt for compatibility with the
    original notebook; here prev means the configured source block.

    Returns:
      prev_logits_by_query: torch.Tensor [num_queries, num_variants, num_selected_features]
      tgt_logits_by_query : torch.Tensor [num_queries, num_variants, num_selected_features]
    """
    if source_block_index is None:
        source_block_index = int(LFH_SOURCE_BLOCK_INDEX)
    if target_block_index is None:
        target_block_index = int(LFH_TARGET_BLOCK_INDEX)

    num_queries = len(prompts_all_agnews)
    num_variants = len(prompts_all_agnews[0])
    selected_feature_ids_tensor = torch.tensor(selected_feature_ids, dtype=torch.long, device=sae.device)
    num_selected = len(selected_feature_ids)

    prev_logits_by_query = torch.zeros(num_queries, num_variants, num_selected, dtype=torch.float32)
    tgt_logits_by_query = torch.zeros(num_queries, num_variants, num_selected, dtype=torch.float32)

    for query_idx in range(num_queries):
        for variant_idx in range(num_variants):
            prompt = prompts_all_agnews[query_idx][variant_idx]

            enc = tokenize_prompt_for_offsets(prompt)
            offset_mapping = enc["offset_mapping"][0].tolist()
            input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
            attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

            block_spans = get_example_block_token_spans(
                prompt=prompt,
                offset_mapping=offset_mapping,
                expected_num_blocks=expected_num_blocks,
            )

            source_span = block_spans[source_block_index]
            tgt_span = block_spans[target_block_index]

            x = get_layer_output(
                model=model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                layer_idx=layer_idx,
            )  # [1, seq, hidden]

            logits = sae_logits_before_jumprelu(sae, x)[0]  # [seq, features]

            source_idx = token_selection_to_torch_indices(source_span, device=logits.device)
            tgt_idx = token_selection_to_torch_indices(tgt_span, device=logits.device)
            source_mean = logits.index_select(0, source_idx)[:, selected_feature_ids_tensor].mean(dim=0).detach().float().cpu()
            tgt_mean = logits.index_select(0, tgt_idx)[:, selected_feature_ids_tensor].mean(dim=0).detach().float().cpu()

            prev_logits_by_query[query_idx, variant_idx] = source_mean
            tgt_logits_by_query[query_idx, variant_idx] = tgt_mean

        print(f"second pass done query {query_idx + 1}/{num_queries}")

    return prev_logits_by_query, tgt_logits_by_query


selected_feature_ids = sorted(freq_over_threshold.keys())

prev_logits_by_query, tgt_logits_by_query = second_pass_collect_prev_tgt_logits(
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=TARGET_LAYER,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
    selected_feature_ids=selected_feature_ids,
    source_block_index=LFH_SOURCE_BLOCK_INDEX,
    target_block_index=LFH_TARGET_BLOCK_INDEX,
)

print("prev/source_logits_by_query:", tuple(prev_logits_by_query.shape))
print("tgt_logits_by_query        :", tuple(tgt_logits_by_query.shape))


In [ ]:

# ============================================================
# 7. Compute per-query Pearson correlations, then average
# ============================================================

def pearson_corr_per_feature_across_variants(prev_mat: np.ndarray, tgt_mat: np.ndarray):
    """
    prev_mat, tgt_mat: [num_variants, num_features]
    Returns:
      r: [num_features]
    """
    if prev_mat.shape != tgt_mat.shape:
        raise ValueError(f"Shape mismatch: prev={prev_mat.shape}, tgt={tgt_mat.shape}")

    x = prev_mat.astype(np.float64)
    y = tgt_mat.astype(np.float64)

    x = x - x.mean(axis=0, keepdims=True)
    y = y - y.mean(axis=0, keepdims=True)

    num = (x * y).sum(axis=0)
    den = np.sqrt((x * x).sum(axis=0) * (y * y).sum(axis=0))

    r = np.full(x.shape[1], np.nan, dtype=np.float64)
    valid = den > 0
    r[valid] = num[valid] / den[valid]
    return r


def compute_final_rho_results(
    prev_logits_by_query: torch.Tensor,
    tgt_logits_by_query: torch.Tensor,
    selected_feature_ids: list[int],
    sort_ascending: bool = True,
):
    """
    Compute:
      - per-query correlation for every selected feature,
      - average correlation across queries.

    Original notebook sorted ascending, i.e. most negative correlations first.
    This is useful if you interpret negative correlation as a prediction-error-like
    or source-target local homeostasis effect.
    """
    prev_np = prev_logits_by_query.numpy()   # [Q, V, F]
    tgt_np = tgt_logits_by_query.numpy()     # [Q, V, F]

    num_queries = prev_np.shape[0]

    rho_by_query = np.stack(
        [
            pearson_corr_per_feature_across_variants(prev_np[q], tgt_np[q])
            for q in range(num_queries)
        ],
        axis=0,
    )  # [Q, F]

    querywise_rho = {}
    final_rho_results = {}

    for feat_local_idx, feat_id in enumerate(selected_feature_ids):
        vals = rho_by_query[:, feat_local_idx]
        querywise_rho[feat_id] = vals.tolist()

        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            continue
        final_rho_results[feat_id] = float(finite_vals.mean())

    final_rho_results = dict(
        sorted(final_rho_results.items(), key=lambda kv: kv[1], reverse=not sort_ascending)
    )

    return querywise_rho, final_rho_results, rho_by_query


querywise_rho, Final_rho_results, rho_by_query = compute_final_rho_results(
    prev_logits_by_query=prev_logits_by_query,
    tgt_logits_by_query=tgt_logits_by_query,
    selected_feature_ids=selected_feature_ids,
    sort_ascending=True,
)

print(f"Computed final correlations for {len(Final_rho_results)} features.")

rows = []
for rank, (feat, rho) in enumerate(Final_rho_results.items(), start=1):
    rows.append({
        "rank": rank,
        "feature_idx": int(feat),
        "source_block_num": int(LFH_SOURCE_BLOCK_NUM),
        "target_block_num": int(LFH_TARGET_BLOCK_NUM),
        "source_to_target_distance": int(LFH_TARGET_BLOCK_NUM - LFH_SOURCE_BLOCK_NUM),
        "mean_source_block_frequency": float(freq_over_threshold[feat]),
        # Backward-compatible column name used in earlier scratch analysis.
        "mean_prev_block_frequency": float(freq_over_threshold[feat]),
        "avg_querywise_pearson_r": float(rho),
    })

Final_rho_df = pd.DataFrame(rows)

print("=" * 100)
print("Top candidate LFH features, sorted by avg Pearson r ascending")
print(
    f"Source block Example {LFH_SOURCE_BLOCK_NUM} -> target block Example {LFH_TARGET_BLOCK_NUM}; "
    "most negative correlations first."
)
print("=" * 100)
display(Final_rho_df.head(50))

# Convenience defaults for plotting cells.
target_query_idx = 0
prev_demo_idx = 0
target_feature = int(Final_rho_df.iloc[0]["feature_idx"])
print("Default target_feature:", target_feature)


In [ ]:

# ============================================================
# 8. Scatter plot for one feature and one fixed target query
# ============================================================

import plotly.graph_objects as go
from scipy.stats import pearsonr

def plot_lfh_scatter(
    target_query_idx: int,
    target_feature: int,
    prev_logits_by_query,
    tgt_logits_by_query,
    selected_feature_ids,
):
    feature_local_idx = list(selected_feature_ids).index(target_feature)

    X = np.array(prev_logits_by_query[target_query_idx][:, feature_local_idx])
    Y = np.array(tgt_logits_by_query[target_query_idx][:, feature_local_idx])

    finite = np.isfinite(X) & np.isfinite(Y)
    X = X[finite]
    Y = Y[finite]

    if len(X) >= 2 and np.std(X) > 0 and np.std(Y) > 0:
        r, p = pearsonr(X, Y)
        slope, intercept = np.polyfit(X, Y, 1)
        x_line = np.linspace(X.min(), X.max(), 200)
        y_line = slope * x_line + intercept
    else:
        r, p = np.nan, np.nan
        x_line, y_line = np.array([]), np.array([])

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=X,
        y=Y,
        mode="markers",
        marker=dict(
            size=8,
            opacity=0.8,
            line=dict(width=1, color="black"),
        ),
        name="Prompt variants",
    ))

    if len(x_line) > 0:
        fig.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(width=3),
            name="Linear fit",
        ))

    fig.update_layout(
        template="seaborn",
        title=dict(
            text=(
                f"Local Feature Homeostasis "
                f"({MODEL_SHORT_NAME}, Layer {TARGET_LAYER}, Feature {target_feature})<br>"
                f"corr={r:.3f}, p={p:.2e}"
            ),
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title=f"Activation on source demonstration (Example {LFH_SOURCE_BLOCK_NUM})",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=14),
        ),
        yaxis=dict(
            title=f"Activation on target demonstration (Example {LFH_TARGET_BLOCK_NUM})",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=14),
        ),
        height=600,
        width=900,
        showlegend=False,
    )

    fig.show()
    return fig, {"r": r, "p": p, "X": X, "Y": Y}


scatter_fig, scatter_stats = plot_lfh_scatter(
    target_query_idx=target_query_idx,
    target_feature=target_feature,
    prev_logits_by_query=prev_logits_by_query,
    tgt_logits_by_query=tgt_logits_by_query,
    selected_feature_ids=selected_feature_ids,
)


# Histograms

# Current configuration: `BLOCK_SCORE_MODE = "All"`

In [ ]:
# ============================================================
# 9. Histogram of average query-wise Pearson correlations
# ============================================================

import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

def plot_rho_histogram(Final_rho_results, num_bins=25):
    values = np.array(list(Final_rho_results.values()), dtype=np.float64)
    values = values[np.isfinite(values)]
    N = len(values)

    if N == 0:
        raise ValueError("No finite rho values to plot.")

    min_val, max_val = float(np.min(values)), float(np.max(values))
    if min_val == max_val:
        min_val -= 1e-3
        max_val += 1e-3

    bin_width = (max_val - min_val) / num_bins

    fig = go.Figure()

    # Histogram with green bars
    fig.add_trace(go.Histogram(
        x=values,
        xbins=dict(start=min_val, end=max_val, size=bin_width),
        opacity=0.7,
        marker_color="green",  # <-- Added green color for bars
        name="Feature count",
    ))

    # KDE curve with dark green line
    if N >= 3 and np.std(values) > 0:
        kde = gaussian_kde(values)
        x_grid = np.linspace(-1, 1, 400)
        y_kde = kde(x_grid) * N * bin_width

        fig.add_trace(go.Scatter(
            x=x_grid,
            y=y_kde,
            mode="lines",
            line=dict(width=4, color="darkgreen"),  # <-- Added dark green color for KDE
            name="KDE Curve",
        ))

    fig.update_layout(
        template="seaborn",  # <-- Changed background to seaborn
        barmode="overlay",
        title=dict(
            text=f"Local Feature Homeostasis Mechanisms in LLMs<br>({MODEL_SHORT_NAME}, Layer {TARGET_LAYER})",
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title="Pearson correlation",
            range=[-1, 1],
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        yaxis=dict(
            title="Feature count",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        height=600,
        width=900,
        showlegend=False
    )

    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'rho_histogram',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("rho_histogram_highres.png", scale=4)

    return fig

# Uncomment to run if Final_rho_results, MODEL_SHORT_NAME, and TARGET_LAYER are defined
rho_hist_fig = plot_rho_histogram(Final_rho_results)


# 12. Empirical test: do target-layer attention heads write the negative feature direction on the next demonstration?

The previous cells identify features whose mean pre-activation on the configured source demonstration block is negatively correlated with their mean pre-activation on the target/next block. This section tests a more mechanistic hypothesis at the same layer as the SAE:

> If feature `f` is high on demonstration `n`, then some target-layer attention heads write a negative component along feature `f`'s SAE direction into the residual stream on the target demonstration.

For each prompt variant, this block captures:

1. target-layer residual output, used to measure SAE pre-activation of feature `f` on the source and target blocks;
2. the input to target-layer `self_attn.o_proj`, split by attention head;
3. each head's signed contribution to the target block along the feature's SAE read direction and write direction.

A candidate implementation head should have:

- negative correlation between source-block feature activation and target-block head write along the feature read/write direction;
- negative high-minus-low difference: prompts where feature `f` is high on the source block should have more negative head writes on the target block;
- ideally, positive correlation between the head's signed write and the target-block feature pre-activation, meaning the head's write direction is actually aligned with the feature's measured target activation.


In [ ]:
# ============================================================
# 12. Attention-head negative-write test for one feature
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

ATTN_WRITE_TEST_LAYER = TARGET_LAYER

# By default, test the strongest anti-correlation feature found above.
# Replace this with a specific feature, e.g. 567.
ATTN_WRITE_TEST_FEATURE_IDX = int(target_feature)

# The prompt manipulation varies the configured source block and holds the
# target block fixed. Default: source Example 7 -> target Example 8.
ATTN_WRITE_TEST_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
ATTN_WRITE_TEST_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

# The selected model can be large. Start with a subset; set both to None for the full run.
ATTN_WRITE_TEST_MAX_QUERIES = min(10, len(prompts_all_agnews))
ATTN_WRITE_TEST_MAX_VARIANTS = None

# For high-vs-low comparisons within each fixed target query.
# 0.25 means bottom quartile vs top quartile across source-demonstration variants.
ATTN_WRITE_TEST_QUANTILE = 0.25

EPS = 1e-9


# ------------------------------------------------------------
# Helpers: SAE single-feature preactivation and attention head geometry
# ------------------------------------------------------------

def _safe_pearson(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _unit(v: torch.Tensor) -> torch.Tensor:
    v = v.float()
    return v / v.norm().clamp_min(1e-8)


@torch.no_grad()
def sae_single_feature_pre_activation(
    sae,
    x: torch.Tensor,
    feature_idx: int,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
):
    """
    Memory-light single-feature version of sae_logits_before_jumprelu.

    Args:
      x: [batch, seq, d_model]

    Returns:
      pre: [batch, seq]
    """
    if feature_idx < 0 or feature_idx >= sae.n_features:
        raise ValueError(f"feature_idx={feature_idx} outside [0, {sae.n_features}).")

    x_sae = x.to(device=sae.device, dtype=sae.dtype)
    x_centered = sae.process_sae_in(x_sae)

    w = sae.W_enc[:, feature_idx]
    b = sae.b_enc[feature_idx]
    pre = x_centered @ w + b

    if filter_high_norm_tokens and x_sae.ndim >= 3:
        norms = x_sae.detach().float().norm(dim=-1)  # [B, S]
        med = norms.median(dim=1, keepdim=True).values.clamp_min(1e-6)
        high_norm_mask = norms > (HIGH_NORM_FILTER_MULTIPLIER * med)
        pre = pre.masked_fill(high_norm_mask.to(pre.device), 0.0)

    return pre


@torch.no_grad()
def capture_layer_output_and_attn_o_proj_input(model, input_ids, attention_mask, layer_idx: int):
    """
    One forward pass that captures:
      - output of model.layers[layer_idx], shape [B, S, d_model]
      - input to self_attn.o_proj at that layer, shape [B, S, d_model]

    The o_proj input is the concatenated per-head attention result before W_O.
    Splitting this tensor by heads lets us compute each head's signed output
    contribution along an SAE feature direction.
    """
    cache = {}

    block = model.get_submodule(f"model.layers.{layer_idx}")
    o_proj = model.get_submodule(f"model.layers.{layer_idx}.self_attn.o_proj")

    def block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["layer_output"] = x.detach()
        return output

    def o_proj_pre_hook(module, inputs):
        cache["o_proj_input"] = inputs[0].detach()
        return None

    h1 = block.register_forward_hook(block_hook)
    h2 = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h1.remove()
        h2.remove()

    if "layer_output" not in cache:
        raise RuntimeError(f"Failed to capture layer output for layer {layer_idx}.")
    if "o_proj_input" not in cache:
        raise RuntimeError(f"Failed to capture o_proj input for layer {layer_idx}.")

    return cache["layer_output"], cache["o_proj_input"]


@torch.no_grad()
def get_head_projection_vectors_for_feature(model, sae, layer_idx: int, feature_idx: int):
    """
    For each attention head h at layer_idx, compute the vector v_h such that:

        head_output_contribution_h dot feature_direction
        = o_proj_input_head_h dot v_h

    Returns head-specific vectors for both SAE read direction and SAE write direction.
    """
    attn = model.get_submodule(f"model.layers.{layer_idx}.self_attn")
    o_proj = attn.o_proj

    num_heads = int(getattr(attn, "num_heads", model.config.num_attention_heads))
    head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))

    W_o = o_proj.weight.detach()  # [d_model_out, d_model_in]
    out_dim, in_dim = W_o.shape

    if in_dim != num_heads * head_dim:
        raise ValueError(
            f"o_proj input dim {in_dim} != num_heads*head_dim {num_heads}*{head_dim}."
        )

    device = W_o.device

    read_dir = _unit(sae.W_enc[:, feature_idx].detach().to(device=device))
    write_dir = _unit(sae.W_dec[feature_idx].detach().to(device=device))

    if read_dir.numel() != out_dim or write_dir.numel() != out_dim:
        raise ValueError(
            f"Feature directions have dim read={read_dir.numel()}, write={write_dir.numel()}, "
            f"but o_proj output dim is {out_dim}."
        )

    # W_o maps concat-head input -> residual output:
    #   y = x @ W_o.T
    # For head h, contribution along direction u is:
    #   (x_h @ W_o_h.T) dot u = x_h dot (W_o_h.T @ u)
    W_heads = W_o.float().reshape(out_dim, num_heads, head_dim).permute(1, 2, 0).contiguous()
    # W_heads: [num_heads, head_dim, out_dim]

    head_vec_read = torch.matmul(W_heads, read_dir.float())    # [num_heads, head_dim]
    head_vec_write = torch.matmul(W_heads, write_dir.float())  # [num_heads, head_dim]

    return {
        "num_heads": num_heads,
        "head_dim": head_dim,
        "read_dir": read_dir.float(),
        "write_dir": write_dir.float(),
        "head_vec_read": head_vec_read.float(),
        "head_vec_write": head_vec_write.float(),
    }


@torch.no_grad()
def measure_prompt_feature_and_head_writes(
    prompt: str,
    feature_idx: int,
    layer_idx: int,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt variant, measure:
      - mean preactivation of feature_idx on source block;
      - mean preactivation of feature_idx on target block;
      - for every attention head at layer_idx, mean signed write on target block
        along SAE read direction and SAE write direction.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    layer_output, o_proj_input = capture_layer_output_and_attn_o_proj_input(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        layer_idx=layer_idx,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_idx = token_selection_to_torch_indices(prev_span, device=pre_f.device)
    tgt_idx = token_selection_to_torch_indices(tgt_span, device=pre_f.device)
    prev_feature_mean = float(pre_f.index_select(0, prev_idx).mean().detach().cpu())
    target_feature_mean = float(pre_f.index_select(0, tgt_idx).mean().detach().cpu())

    head_geom = get_head_projection_vectors_for_feature(
        model=model,
        sae=sae,
        layer_idx=layer_idx,
        feature_idx=feature_idx,
    )

    num_heads = head_geom["num_heads"]
    head_dim = head_geom["head_dim"]
    head_vec_read = head_geom["head_vec_read"].to(device=o_proj_input.device)
    head_vec_write = head_geom["head_vec_write"].to(device=o_proj_input.device)

    # Split o_proj input into heads on the selected target-block tokens.
    tgt_idx_o = token_selection_to_torch_indices(tgt_span, device=o_proj_input.device)
    z = o_proj_input[0].index_select(0, tgt_idx_o)
    z_heads = z.reshape(tgt_idx_o.numel(), num_heads, head_dim).float()

    # Signed contribution of each head's W_O output to the feature read/write directions.
    # Negative values mean the head writes in the negative feature direction.
    target_head_write_to_read = torch.einsum("thd,hd->th", z_heads, head_vec_read.float()).mean(dim=0)
    target_head_write_to_write = torch.einsum("thd,hd->th", z_heads, head_vec_write.float()).mean(dim=0)

    return {
        "prev_feature_mean": prev_feature_mean,
        "target_feature_mean": target_feature_mean,
        "target_head_write_to_read": target_head_write_to_read.detach().cpu().numpy(),
        "target_head_write_to_write": target_head_write_to_write.detach().cpu().numpy(),
        "prev_span": prev_span,
        "target_span": tgt_span,
        "num_heads": num_heads,
    }


def _summarize_head_write_test(prompt_records_df, head_read_mat, head_write_mat, layer_idx: int, feature_idx: int):
    """
    Summarize head-level evidence for:
      high prev activation -> negative target-block attention-head write.
    """
    num_prompts, num_heads = head_read_mat.shape
    rows = []

    x_all = prompt_records_df["prev_feature_mean"].to_numpy(dtype=np.float64)
    y_tgt_all = prompt_records_df["target_feature_mean"].to_numpy(dtype=np.float64)
    q_all = prompt_records_df["query_idx"].to_numpy()

    unique_queries = sorted(prompt_records_df["query_idx"].unique().tolist())

    # Overall feature anti-correlation in the same subset.
    feature_corr_pooled = _safe_pearson(x_all, y_tgt_all)

    feature_corr_by_query = []
    for q in unique_queries:
        mask = q_all == q
        feature_corr_by_query.append(_safe_pearson(x_all[mask], y_tgt_all[mask]))
    feature_corr_mean_query = float(np.nanmean(feature_corr_by_query))

    for h in range(num_heads):
        read_h = head_read_mat[:, h]
        write_h = head_write_mat[:, h]

        corr_prev_to_read_by_q = []
        corr_prev_to_write_by_q = []
        corr_read_to_target_by_q = []
        corr_write_to_target_by_q = []
        delta_read_by_q = []
        delta_write_by_q = []
        delta_read_z_by_q = []
        delta_write_z_by_q = []

        for q in unique_queries:
            mask = q_all == q
            x = x_all[mask]
            y_read = read_h[mask]
            y_write = write_h[mask]
            y_tgt = y_tgt_all[mask]

            corr_prev_to_read_by_q.append(_safe_pearson(x, y_read))
            corr_prev_to_write_by_q.append(_safe_pearson(x, y_write))
            corr_read_to_target_by_q.append(_safe_pearson(y_read, y_tgt))
            corr_write_to_target_by_q.append(_safe_pearson(y_write, y_tgt))

            finite = np.isfinite(x) & np.isfinite(y_read) & np.isfinite(y_write)
            if finite.sum() >= 4:
                xf = x[finite]
                yr = y_read[finite]
                yw = y_write[finite]

                q_low = np.quantile(xf, ATTN_WRITE_TEST_QUANTILE)
                q_high = np.quantile(xf, 1.0 - ATTN_WRITE_TEST_QUANTILE)
                low_mask = xf <= q_low
                high_mask = xf >= q_high

                if low_mask.sum() > 0 and high_mask.sum() > 0:
                    dr = float(np.mean(yr[high_mask]) - np.mean(yr[low_mask]))
                    dw = float(np.mean(yw[high_mask]) - np.mean(yw[low_mask]))
                    delta_read_by_q.append(dr)
                    delta_write_by_q.append(dw)
                    delta_read_z_by_q.append(dr / (float(np.std(yr)) + EPS))
                    delta_write_z_by_q.append(dw / (float(np.std(yw)) + EPS))
                else:
                    delta_read_by_q.append(np.nan)
                    delta_write_by_q.append(np.nan)
                    delta_read_z_by_q.append(np.nan)
                    delta_write_z_by_q.append(np.nan)
            else:
                delta_read_by_q.append(np.nan)
                delta_write_by_q.append(np.nan)
                delta_read_z_by_q.append(np.nan)
                delta_write_z_by_q.append(np.nan)

        corr_prev_to_read_mean = float(np.nanmean(corr_prev_to_read_by_q))
        corr_prev_to_write_mean = float(np.nanmean(corr_prev_to_write_by_q))
        corr_read_to_target_mean = float(np.nanmean(corr_read_to_target_by_q))
        corr_write_to_target_mean = float(np.nanmean(corr_write_to_target_by_q))
        delta_read_mean = float(np.nanmean(delta_read_by_q))
        delta_write_mean = float(np.nanmean(delta_write_by_q))
        delta_read_z_mean = float(np.nanmean(delta_read_z_by_q))
        delta_write_z_mean = float(np.nanmean(delta_write_z_by_q))

        # Mechanism score:
        #   high source feature activation should correlate with NEGATIVE head write.
        #   Therefore, corr(prev, write_to_read) and high-minus-low delta should be negative.
        anti_write_score_read = (
            max(-corr_prev_to_read_mean, 0.0)
            * max(-delta_read_z_mean, 0.0)
            * (0.5 + 0.5 * max(corr_read_to_target_mean, 0.0))
        )
        anti_write_score_write = (
            max(-corr_prev_to_write_mean, 0.0)
            * max(-delta_write_z_mean, 0.0)
            * (0.5 + 0.5 * max(corr_write_to_target_mean, 0.0))
        )

        rows.append({
            "layer": int(layer_idx),
            "head": int(h),
            "feature_idx": int(feature_idx),

            "feature_prev_to_target_corr_pooled": feature_corr_pooled,
            "feature_prev_to_target_corr_mean_query": feature_corr_mean_query,

            "corr_prev_to_target_head_write_to_read_mean_query": corr_prev_to_read_mean,
            "corr_prev_to_target_head_write_to_write_mean_query": corr_prev_to_write_mean,

            "corr_prev_to_target_head_write_to_negative_read_mean_query": -corr_prev_to_read_mean,
            "corr_prev_to_target_head_write_to_negative_write_mean_query": -corr_prev_to_write_mean,

            "high_minus_low_head_write_to_read_mean": delta_read_mean,
            "high_minus_low_head_write_to_write_mean": delta_write_mean,
            "high_minus_low_head_write_to_read_z_mean": delta_read_z_mean,
            "high_minus_low_head_write_to_write_z_mean": delta_write_z_mean,

            "high_minus_low_head_write_to_negative_read_z_mean": -delta_read_z_mean,
            "high_minus_low_head_write_to_negative_write_z_mean": -delta_write_z_mean,

            "corr_head_write_to_read_with_target_feature_mean_query": corr_read_to_target_mean,
            "corr_head_write_to_write_with_target_feature_mean_query": corr_write_to_target_mean,

            "anti_write_score_read": anti_write_score_read,
            "anti_write_score_write": anti_write_score_write,
            "anti_write_score_max": max(anti_write_score_read, anti_write_score_write),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(
        ["anti_write_score_read", "corr_prev_to_target_head_write_to_negative_read_mean_query"],
        ascending=False,
    ).reset_index(drop=True)

    df["rank_by_read_score"] = np.arange(1, len(df) + 1)

    df_by_write = df.sort_values(
        ["anti_write_score_write", "corr_prev_to_target_head_write_to_negative_write_mean_query"],
        ascending=False,
    ).reset_index(drop=True)
    write_ranks = {
        (int(row.layer), int(row.head)): rank
        for rank, row in enumerate(df_by_write.itertuples(index=False), start=1)
    }
    df["rank_by_write_score"] = [write_ranks[(int(r.layer), int(r.head))] for r in df.itertuples(index=False)]

    return df


@torch.no_grad()
def run_attention_negative_write_test_for_feature(
    feature_idx: int,
    prompts_all_agnews,
    layer_idx: int = TARGET_LAYER,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    max_queries=None,
    max_variants=None,
    verbose_every: int = 10,
):
    """
    Main test.

    For every selected prompt variant:
      x = mean feature preactivation on source block
      y_h = mean layer-attention-head signed write to feature direction on target block

    Heads supporting the proposed mechanism should satisfy:
      corr(x, y_h) < 0
      mean(y_h | x high) - mean(y_h | x low) < 0
    """
    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    prompt_rows = []
    read_rows = []
    write_rows = []

    total = 0
    for query_idx in query_indices:
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            m = measure_prompt_feature_and_head_writes(
                prompt=prompt,
                feature_idx=feature_idx,
                layer_idx=layer_idx,
                previous_block_index=previous_block_index,
                target_block_index=target_block_index,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            prompt_rows.append({
                "query_idx": int(query_idx),
                "variant_idx": int(variant_idx),
                "feature_idx": int(feature_idx),
                "layer": int(layer_idx),
                "prev_feature_mean": m["prev_feature_mean"],
                "target_feature_mean": m["target_feature_mean"],
                "prev_span_start": int(m["prev_span"][0]),
                "prev_span_end": int(m["prev_span"][1]),
                "target_span_start": int(m["target_span"][0]),
                "target_span_end": int(m["target_span"][1]),
            })
            read_rows.append(m["target_head_write_to_read"])
            write_rows.append(m["target_head_write_to_write"])

            total += 1

        if verbose_every is not None and (len(prompt_rows) % max(verbose_every, 1) == 0 or query_idx == query_indices[-1]):
            print(f"processed query {query_idx + 1}/{query_indices[-1] + 1}; prompt rows={len(prompt_rows)}")

    prompt_records_df = pd.DataFrame(prompt_rows)
    head_read_mat = np.stack(read_rows, axis=0)
    head_write_mat = np.stack(write_rows, axis=0)

    head_summary_df = _summarize_head_write_test(
        prompt_records_df=prompt_records_df,
        head_read_mat=head_read_mat,
        head_write_mat=head_write_mat,
        layer_idx=layer_idx,
        feature_idx=feature_idx,
    )

    print("=" * 120)
    print(f"Layer-{layer_idx} attention negative-write test for feature {feature_idx}")
    print("Expected mechanism: high source-block feature activation -> negative target-block head write")
    print("Rows/prompts measured:", len(prompt_records_df))
    print("Feature prev->target corr, pooled:", head_summary_df["feature_prev_to_target_corr_pooled"].iloc[0])
    print("Feature prev->target corr, mean query-wise:", head_summary_df["feature_prev_to_target_corr_mean_query"].iloc[0])
    print("=" * 120)

    display(head_summary_df.head(25))

    return head_summary_df, prompt_records_df, head_read_mat, head_write_mat


def plot_attention_write_scatter_for_head(
    prompt_records_df,
    head_mat,
    head: int,
    feature_idx: int,
    layer_idx: int,
    direction_name: str = "read",
):
    """
    Scatter: source feature mean vs target-block attention-head signed write.
    A negative slope supports the proposed negative-write mechanism.
    """
    x = prompt_records_df["prev_feature_mean"].to_numpy(dtype=np.float64)
    y = head_mat[:, int(head)].astype(np.float64)
    q = prompt_records_df["query_idx"].to_numpy()

    r = _safe_pearson(x, y)
    if np.isfinite(r) and np.std(x) > 0 and np.std(y) > 0:
        slope, intercept = np.polyfit(x, y, 1)
        xx = np.linspace(np.min(x), np.max(x), 200)
        yy = slope * xx + intercept
    else:
        slope, intercept = np.nan, np.nan
        xx, yy = np.array([]), np.array([])

    df_plot = pd.DataFrame({
        "prev_feature_mean": x,
        "target_head_write": y,
        "query_idx": q.astype(str),
        "variant_idx": prompt_records_df["variant_idx"].to_numpy(),
        "target_feature_mean": prompt_records_df["target_feature_mean"].to_numpy(),
    })

    fig = px.scatter(
        df_plot,
        x="prev_feature_mean",
        y="target_head_write",
        color="query_idx",
        hover_data=["variant_idx", "target_feature_mean"],
        title=(
            f"Layer {layer_idx} head {head}: source feature activation vs target-block head write<br>"
            f"feature={feature_idx}, direction={direction_name}, pooled r={r:.3f}, slope={slope:.3f}"
        ),
    )

    if len(xx) > 0:
        fig.add_trace(go.Scatter(
            x=xx,
            y=yy,
            mode="lines",
            name="pooled linear fit",
            line=dict(width=3),
        ))

    fig.add_hline(y=0, line_dash="dash", opacity=0.4)
    fig.update_layout(
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source demonstration block",
        yaxis_title=f"Mean attention-head write to SAE {direction_name} direction on target block",
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the one-feature test
# ------------------------------------------------------------

head_summary_df, head_prompt_records_df, head_read_mat, head_write_mat = run_attention_negative_write_test_for_feature(
    feature_idx=ATTN_WRITE_TEST_FEATURE_IDX,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=ATTN_WRITE_TEST_LAYER,
    previous_block_index=ATTN_WRITE_TEST_PREVIOUS_BLOCK_INDEX,
    target_block_index=ATTN_WRITE_TEST_TARGET_BLOCK_INDEX,
    max_queries=ATTN_WRITE_TEST_MAX_QUERIES,
    max_variants=ATTN_WRITE_TEST_MAX_VARIANTS,
    verbose_every=10,
)

TOP_NEGATIVE_WRITE_HEAD = int(head_summary_df.iloc[0]["head"])
print("Top negative-write candidate head:", (ATTN_WRITE_TEST_LAYER, TOP_NEGATIVE_WRITE_HEAD))

_ = plot_attention_write_scatter_for_head(
    prompt_records_df=head_prompt_records_df,
    head_mat=head_read_mat,
    head=TOP_NEGATIVE_WRITE_HEAD,
    feature_idx=ATTN_WRITE_TEST_FEATURE_IDX,
    layer_idx=ATTN_WRITE_TEST_LAYER,
    direction_name="SAE read",
)



# 14. Causal attention-head intervention test

The source/target attention-write section is **observational**: it asks whether a head's signed write on the target exemplar is more negative when the source exemplar's feature activation is high. This section adds a direct causal test.

For a chosen `(feature_idx, layer_idx, head_idx)`, we intervene on that head's `o_proj` input on the **target exemplar tokens only** and then re-measure the source-to-target feature correlation.

The intervention is parameterized by an additive `alpha` on the selected head contribution:

- `alpha = 0`: baseline / no intervention.
- `alpha = -1`: ablation, because the effective multiplier is `1 + alpha = 0`.
- `alpha > 0`: amplification by adding extra copies of the head contribution.

For an anti-relation feature, the expected pattern is:

- If the head is writing a negative feature-direction update on the target exemplar, **ablating** it should weaken the negative correlation.
- **Amplifying** it should strengthen the negative correlation.

This does not prove that the head reads specifically from the source exemplar. It tests the write-side causal claim: the chosen head's target-token output helps implement the source-to-target anti-relation.


In [ ]:
# ============================================================
# 14. Causal validation: ablate/amplify one attention head and remeasure
#      source-exemplar -> target-exemplar feature correlation
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

# Feature to test. By default, use the same feature as the observational
# negative-write test above.
HEAD_CAUSAL_FEATURE_IDX = int(ATTN_WRITE_TEST_FEATURE_IDX) if "ATTN_WRITE_TEST_FEATURE_IDX" in globals() else int(target_feature)

# Attention head to test. By default, use the top head from the observational
# negative-write ranking, if available.
HEAD_CAUSAL_LAYER = int(ATTN_WRITE_TEST_LAYER) if "ATTN_WRITE_TEST_LAYER" in globals() else int(TARGET_LAYER)

# aaaaa
HEAD_CAUSAL_HEAD = int(TOP_NEGATIVE_WRITE_HEAD) if "TOP_NEGATIVE_WRITE_HEAD" in globals() else 0
# HEAD_CAUSAL_HEAD = 13

# The SAE feature is measured at TARGET_LAYER, because the SAE was trained on
# resid_post_layer_{TARGET_LAYER}. A causal head layer must be <= TARGET_LAYER.
HEAD_CAUSAL_FEATURE_SCORE_LAYER = int(TARGET_LAYER)

# Use the configured source/target blocks. Default: source Example 7 -> target Example 8.
HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
HEAD_CAUSAL_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

# Intervention scope:
#   "target_block" edits only target/example-8 tokens. This is the cleanest
#   test of the proposed write-side mechanism.
#   "previous_block" edits only source/example tokens.
#   "previous_and_target" edits both.
#   "all_example_tokens" edits all example-block tokens.
HEAD_CAUSAL_TOKEN_SCOPE = "target_block"

# Additive alphas for the selected head's o_proj input on edited tokens.
# The effective multiplier is (1 + alpha):
#   alpha =  0.0: no intervention / baseline
#   alpha = -1.0: ablate this head on edited tokens
#   alpha =  1.0: add one extra copy of this head's contribution
HEAD_CAUSAL_HEAD_ALPHAS = []

# The selected model can be large. Start small, then scale up.
HEAD_CAUSAL_MAX_QUERIES = None  # full Section 7 prompt set; set smaller manually only for a smoke test
HEAD_CAUSAL_MAX_VARIANTS = None  # all variants, matching Section 7 by default

# For high-vs-low source activation summaries.
HEAD_CAUSAL_QUANTILE = 0.25

EPS = 1e-9

print("=" * 120)
print("Attention-head causal intervention config")
print("=" * 120)
print("feature_idx:", HEAD_CAUSAL_FEATURE_IDX)
print("attention layer:", HEAD_CAUSAL_LAYER)
print("attention head:", HEAD_CAUSAL_HEAD)
print("feature score layer:", HEAD_CAUSAL_FEATURE_SCORE_LAYER)
print("token scope:", HEAD_CAUSAL_TOKEN_SCOPE)
print("head alphas:", HEAD_CAUSAL_HEAD_ALPHAS)
print("max queries:", HEAD_CAUSAL_MAX_QUERIES)
print("max variants:", HEAD_CAUSAL_MAX_VARIANTS)
print("=" * 120)

if HEAD_CAUSAL_LAYER > HEAD_CAUSAL_FEATURE_SCORE_LAYER:
    raise ValueError(
        f"HEAD_CAUSAL_LAYER={HEAD_CAUSAL_LAYER} is later than "
        f"HEAD_CAUSAL_FEATURE_SCORE_LAYER={HEAD_CAUSAL_FEATURE_SCORE_LAYER}. "
        "A later attention head cannot causally affect an earlier SAE-layer activation "
        "in the same forward pass. Choose HEAD_CAUSAL_LAYER <= TARGET_LAYER."
    )


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def _safe_pearson_local(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _attn_num_heads_and_head_dim(model, layer_idx: int):
    attn = model.get_submodule(f"model.layers.{layer_idx}.self_attn")
    num_heads = int(getattr(attn, "num_heads", model.config.num_attention_heads))
    head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))
    return num_heads, head_dim


def build_head_intervention_token_mask(
    seq_len: int,
    block_spans,
    previous_block_index: int,
    target_block_index: int,
    token_scope: str = "target_block",
):
    """
    Build a [1, seq_len] bool mask specifying which token positions receive
    the head ablation/amplification.
    """
    mask = torch.zeros((1, seq_len), dtype=torch.bool)

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    def _mark(selection):
        idx = token_selection_to_torch_indices(selection, device=mask.device)
        mask[0, idx] = True

    if token_scope == "target_block":
        _mark(tgt_span)
    elif token_scope == "previous_block":
        _mark(prev_span)
    elif token_scope == "previous_and_target":
        _mark(prev_span)
        _mark(tgt_span)
    elif token_scope == "all_example_tokens":
        for selection in block_spans:
            _mark(selection)
    else:
        raise ValueError(f"Unknown token_scope: {token_scope}")

    return mask


@torch.no_grad()
def run_forward_capture_feature_layer_with_head_alpha(
    model,
    input_ids,
    attention_mask,
    *,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    token_mask: torch.Tensor,
    head_alpha: float = 0.0,
):
    """
    Run one forward pass while optionally scaling one attention head's o_proj
    input on selected tokens, then capture the feature_layer_idx residual output.

    The edited tensor is the input to self_attn.o_proj, shaped [B, S, hidden].
    It is a concatenation of per-head attention outputs before W_O.

    head_alpha semantics:
        0.0: no intervention / baseline
       -1.0: ablate selected head on selected tokens
        1.0: add one extra copy of the selected head's contribution
        alpha: effective multiplier is (1 + alpha)
    """
    if head_layer_idx > feature_layer_idx:
        raise ValueError(
            f"head_layer_idx={head_layer_idx} > feature_layer_idx={feature_layer_idx}. "
            "This cannot affect the measured feature activation in the same forward pass."
        )

    cache = {}

    feature_block = model.get_submodule(f"model.layers.{feature_layer_idx}")
    o_proj = model.get_submodule(f"model.layers.{head_layer_idx}.self_attn.o_proj")
    num_heads, head_dim = _attn_num_heads_and_head_dim(model, head_layer_idx)

    if not (0 <= int(head_idx) < num_heads):
        raise ValueError(f"head_idx={head_idx} outside [0, {num_heads}).")

    head_start = int(head_idx) * head_dim
    head_end = head_start + head_dim

    def feature_block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["feature_layer_output"] = x.detach().clone()
        return output

    def o_proj_pre_hook(module, inputs):
        z = inputs[0]

        # alpha 0.0 is exactly no intervention.
        if float(head_alpha) == 0.0:
            return None

        if z.shape[-1] != num_heads * head_dim:
            raise RuntimeError(
                f"o_proj input last dim {z.shape[-1]} != num_heads*head_dim "
                f"{num_heads}*{head_dim}."
            )

        edit_mask = token_mask.to(device=z.device, dtype=torch.bool)
        if edit_mask.ndim != 2 or edit_mask.shape[0] != z.shape[0] or edit_mask.shape[1] != z.shape[1]:
            raise RuntimeError(
                f"token_mask shape {tuple(edit_mask.shape)} incompatible with o_proj input shape {tuple(z.shape)}."
            )

        z_edit = z.clone()
        head_slice = z_edit[..., head_start:head_end]
        effective_multiplier = 1.0 + float(head_alpha)
        scaled_head_slice = head_slice * effective_multiplier
        z_edit[..., head_start:head_end] = torch.where(edit_mask[..., None], scaled_head_slice, head_slice)

        return (z_edit,) + tuple(inputs[1:])

    h_feature = feature_block.register_forward_hook(feature_block_hook)
    h_head = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h_head.remove()
        h_feature.remove()

    if "feature_layer_output" not in cache:
        raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")

    return cache["feature_layer_output"]


@torch.no_grad()
def measure_prompt_feature_with_head_alpha(
    prompt: str,
    *,
    feature_idx: int,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    head_alpha: float,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt, run a selected attention-head intervention and measure
    feature pre-activation on previous and target exemplar blocks.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    token_mask = build_head_intervention_token_mask(
        seq_len=input_ids.shape[1],
        block_spans=block_spans,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        token_scope=token_scope,
    ).to(MODEL_INPUT_DEVICE)

    layer_output = run_forward_capture_feature_layer_with_head_alpha(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        head_layer_idx=head_layer_idx,
        head_idx=head_idx,
        feature_layer_idx=feature_layer_idx,
        token_mask=token_mask,
        head_alpha=head_alpha,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_selection = block_spans[previous_block_index]
    target_selection = block_spans[target_block_index]
    prev_idx = token_selection_to_torch_indices(prev_selection, device=pre_f.device)
    target_idx = token_selection_to_torch_indices(target_selection, device=pre_f.device)

    return {
        "previous_feature_mean": float(pre_f.index_select(0, prev_idx).mean().detach().cpu()),
        "target_feature_mean": float(pre_f.index_select(0, target_idx).mean().detach().cpu()),
        "prev_span": block_spans[previous_block_index],
        "target_span": block_spans[target_block_index],
    }


def _original_lfh_pearson_1d(x, y):
    """
    Pearson r using the same algebra as Section 7's
    pearson_corr_per_feature_across_variants, specialized to one feature.

    This deliberately avoids scipy.stats.pearsonr so the causal alpha=0
    baseline is numerically aligned with Final_rho_df.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if x.size < 3:
        return np.nan

    x = x - x.mean()
    y = y - y.mean()

    num = float((x * y).sum())
    den = float(np.sqrt((x * x).sum() * (y * y).sum()))
    if den <= 0.0:
        return np.nan
    return num / den


def _querywise_corr_summary(records_df, target_col: str, prev_col: str = "baseline_previous_feature_mean"):
    """
    Compute pooled and query-wise correlations for prev_col -> target_col.

    The mean query-wise statistic is computed with exactly the same Pearson
    formula used in Section 7. Therefore, when alpha=0 and the same prompts are
    used, this value matches Final_rho_df['avg_querywise_pearson_r'] for the
    selected feature.
    """
    pooled = _original_lfh_pearson_1d(records_df[prev_col].values, records_df[target_col].values)

    q_corrs = []
    for q, g in records_df.groupby("query_idx", sort=True):
        r = _original_lfh_pearson_1d(g[prev_col].values, g[target_col].values)
        if np.isfinite(r):
            q_corrs.append(r)

    if len(q_corrs) == 0:
        return pooled, np.nan, np.nan, 0

    return pooled, float(np.mean(q_corrs)), float(np.std(q_corrs)), int(len(q_corrs))


def _high_low_delta_by_query(records_df, value_col: str, prev_col: str = "baseline_previous_feature_mean", quantile: float = 0.25):
    """
    For each query, compute mean(value | prev high) - mean(value | prev low).
    """
    deltas = []
    for q, g in records_df.groupby("query_idx"):
        x = g[prev_col].to_numpy(dtype=np.float64)
        y = g[value_col].to_numpy(dtype=np.float64)
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]
        if len(x) < 4:
            continue
        lo = np.quantile(x, quantile)
        hi = np.quantile(x, 1.0 - quantile)
        low_mask = x <= lo
        high_mask = x >= hi
        if low_mask.sum() == 0 or high_mask.sum() == 0:
            continue
        deltas.append(float(y[high_mask].mean() - y[low_mask].mean()))
    if len(deltas) == 0:
        return np.nan, np.nan, 0
    return float(np.mean(deltas)), float(np.std(deltas)), int(len(deltas))



def _normalize_lfh_block_index(block_index: int, total_blocks: int = NUM_BLOCKS_TOTAL) -> int:
    block_index = int(block_index)
    if block_index < 0:
        block_index = int(total_blocks) + block_index
    if block_index < 0 or block_index >= int(total_blocks):
        raise ValueError(f"block_index={block_index} outside [0, {total_blocks}).")
    return block_index


def _get_feature_local_idx_in_selected_features(feature_idx: int):
    if "selected_feature_ids" not in globals():
        return None
    selected = [int(f) for f in selected_feature_ids]
    try:
        return selected.index(int(feature_idx))
    except ValueError:
        return None


def _can_use_section7_cached_lfh_tensors(
    *,
    feature_idx: int,
    previous_block_index: int,
    target_block_index: int,
    feature_score_layer: int,
):
    """
    Section 7 cached tensors are valid only for the original LFH source/target
    block pair and the same SAE score layer. If these conditions hold, using
    them makes causal alpha=0 exactly comparable to Final_rho_df.
    """
    required = ["prev_logits_by_query", "tgt_logits_by_query", "selected_feature_ids"]
    missing = [name for name in required if name not in globals()]
    if missing:
        return False, f"missing cached tensors/feature list: {missing}", None

    if int(feature_score_layer) != int(TARGET_LAYER):
        return False, f"feature_score_layer={feature_score_layer} != TARGET_LAYER={TARGET_LAYER}", None

    try:
        prev_idx = _normalize_lfh_block_index(previous_block_index)
        tgt_idx = _normalize_lfh_block_index(target_block_index)
        src_idx = _normalize_lfh_block_index(LFH_SOURCE_BLOCK_INDEX)
        main_tgt_idx = _normalize_lfh_block_index(LFH_TARGET_BLOCK_INDEX)
    except Exception as e:
        return False, str(e), None

    if prev_idx != src_idx or tgt_idx != main_tgt_idx:
        return (
            False,
            f"requested blocks source={prev_idx}, target={tgt_idx} do not match "
            f"Section 7 LFH blocks source={src_idx}, target={main_tgt_idx}",
            None,
        )

    feature_local_idx = _get_feature_local_idx_in_selected_features(feature_idx)
    if feature_local_idx is None:
        return False, f"feature_idx={feature_idx} is not in selected_feature_ids", None

    return True, "using Section 7 cached LFH tensors", int(feature_local_idx)


def _get_section7_cached_pair_for_prompt(feature_local_idx: int, query_idx: int, variant_idx: int):
    prev_arr = prev_logits_by_query
    tgt_arr = tgt_logits_by_query
    if int(query_idx) >= int(prev_arr.shape[0]) or int(variant_idx) >= int(prev_arr.shape[1]):
        raise IndexError(
            f"cached LFH tensor index out of range: query_idx={query_idx}, variant_idx={variant_idx}, "
            f"shape={tuple(prev_arr.shape)}"
        )
    base_prev = float(prev_arr[int(query_idx), int(variant_idx), int(feature_local_idx)].detach().cpu())
    base_tgt = float(tgt_arr[int(query_idx), int(variant_idx), int(feature_local_idx)].detach().cpu())
    return base_prev, base_tgt


def _section7_cached_lfh_reference_for_subset(
    *,
    feature_idx: int,
    query_indices,
    max_variants,
    previous_block_index: int,
    target_block_index: int,
    feature_score_layer: int,
):
    ok, reason, feature_local_idx = _can_use_section7_cached_lfh_tensors(
        feature_idx=feature_idx,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        feature_score_layer=feature_score_layer,
    )
    if not ok:
        return {
            "available": False,
            "reason": reason,
            "feature_local_idx": None,
            "pooled_corr": np.nan,
            "mean_query_corr": np.nan,
            "std_query_corr": np.nan,
            "n_queries": 0,
            "final_rho_df_full_corr": np.nan,
        }

    prev_arr = prev_logits_by_query.detach().cpu().numpy()
    tgt_arr = tgt_logits_by_query.detach().cpu().numpy()

    q_corrs = []
    pooled_prev = []
    pooled_tgt = []

    for q in query_indices:
        q = int(q)
        n_variants_total = int(prev_arr.shape[1])
        n_variants = n_variants_total if max_variants is None else min(int(max_variants), n_variants_total)
        x = prev_arr[q, :n_variants, int(feature_local_idx)]
        y = tgt_arr[q, :n_variants, int(feature_local_idx)]
        pooled_prev.append(x)
        pooled_tgt.append(y)
        r = _original_lfh_pearson_1d(x, y)
        if np.isfinite(r):
            q_corrs.append(r)

    if len(pooled_prev) > 0:
        pooled_prev = np.concatenate(pooled_prev)
        pooled_tgt = np.concatenate(pooled_tgt)
        pooled_corr = _original_lfh_pearson_1d(pooled_prev, pooled_tgt)
    else:
        pooled_corr = np.nan

    if len(q_corrs) == 0:
        mean_query_corr = np.nan
        std_query_corr = np.nan
        n_queries = 0
    else:
        mean_query_corr = float(np.mean(q_corrs))
        std_query_corr = float(np.std(q_corrs))
        n_queries = int(len(q_corrs))

    final_rho_full = np.nan
    if "Final_rho_results" in globals() and int(feature_idx) in Final_rho_results:
        final_rho_full = float(Final_rho_results[int(feature_idx)])
    elif "Final_rho_df" in globals() and isinstance(Final_rho_df, pd.DataFrame) and "feature_idx" in Final_rho_df.columns:
        match = Final_rho_df[Final_rho_df["feature_idx"].astype(int) == int(feature_idx)]
        if len(match) > 0 and "avg_querywise_pearson_r" in match.columns:
            final_rho_full = float(match["avg_querywise_pearson_r"].iloc[0])

    return {
        "available": True,
        "reason": reason,
        "feature_local_idx": int(feature_local_idx),
        "pooled_corr": float(pooled_corr),
        "mean_query_corr": mean_query_corr,
        "std_query_corr": std_query_corr,
        "n_queries": n_queries,
        "final_rho_df_full_corr": final_rho_full,
    }



@torch.no_grad()
def run_attention_head_alpha_causal_test(
    *,
    feature_idx: int,
    layer_idx: int,
    head_idx: int,
    prompts_all_agnews,
    feature_score_layer: int = TARGET_LAYER,
    head_alphas=(-10.0, -5.0, -1.0, 0.0, 1.0, 5.0, 10.0),
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    max_queries=None,
    max_variants=None,
    quantile: float = 0.25,
    verbose_every: int = 10,
    use_section7_cached_baseline: bool = True,
):
    """
    Main causal test.

    For each prompt variant:
      1. Use the exact Section 7 cached source/target feature means for the
         baseline whenever the requested feature, source/target blocks, and
         score layer match the LFH ranking run.
      2. Run each requested intervention alpha.
      3. Recompute corr(source_feature_baseline, target_feature_intervened)
         with the same query-wise Pearson formula used in Section 7.

    Consequence: when head_alpha=0.0 and max_queries/max_variants select the
    same prompt set as Section 7, the alpha=0 correlation is identical to
    Final_rho_df['avg_querywise_pearson_r'] for this feature.
    """
    feature_idx = int(feature_idx)
    layer_idx = int(layer_idx)
    head_idx = int(head_idx)
    feature_score_layer = int(feature_score_layer)

    if layer_idx > feature_score_layer:
        raise ValueError(
            f"layer_idx={layer_idx} > feature_score_layer={feature_score_layer}; "
            "the intervention cannot affect the measured feature in the same forward pass."
        )

    # De-duplicate alphas but keep order; ensure baseline alpha=0.0 is available internally.
    requested_alphas = []
    for alpha in head_alphas:
        af = float(alpha)
        if af not in requested_alphas:
            requested_alphas.append(af)
    if 0.0 not in requested_alphas:
        alphas_to_run = [0.0] + requested_alphas
    else:
        alphas_to_run = requested_alphas

    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    cached_ref = _section7_cached_lfh_reference_for_subset(
        feature_idx=feature_idx,
        query_indices=query_indices,
        max_variants=max_variants,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        feature_score_layer=feature_score_layer,
    )
    use_cached_baseline = bool(use_section7_cached_baseline and cached_ref["available"])
    feature_local_idx = cached_ref["feature_local_idx"] if use_cached_baseline else None

    if use_cached_baseline:
        print("Using Section 7 cached LFH source/target feature means for causal baseline.")
        print(f"Cached subset mean query-wise corr: {cached_ref['mean_query_corr']:.12f}")
        if np.isfinite(cached_ref["final_rho_df_full_corr"]):
            print(f"Final_rho_df full mean query-wise corr: {cached_ref['final_rho_df_full_corr']:.12f}")
    else:
        print("Warning: not using Section 7 cached baseline:", cached_ref["reason"])
        print("The alpha=0 baseline will be recomputed, so exact equality to Final_rho_df is not guaranteed.")

    rows = []

    for query_counter, query_idx in enumerate(query_indices, start=1):
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            if use_cached_baseline:
                base_prev, base_tgt = _get_section7_cached_pair_for_prompt(
                    feature_local_idx=feature_local_idx,
                    query_idx=query_idx,
                    variant_idx=variant_idx,
                )
                baseline_source = "section7_cached_lfh_tensors"
            else:
                baseline = measure_prompt_feature_with_head_alpha(
                    prompt=prompt,
                    feature_idx=feature_idx,
                    head_layer_idx=layer_idx,
                    head_idx=head_idx,
                    feature_layer_idx=feature_score_layer,
                    head_alpha=0.0,
                    previous_block_index=previous_block_index,
                    target_block_index=target_block_index,
                    token_scope=token_scope,
                    expected_num_blocks=NUM_BLOCKS_TOTAL,
                )
                base_prev = baseline["previous_feature_mean"]
                base_tgt = baseline["target_feature_mean"]
                baseline_source = "recomputed_forward_pass"

            for alpha in alphas_to_run:
                if float(alpha) == 0.0:
                    previous_mean = base_prev
                    target_mean = base_tgt
                else:
                    intervened = measure_prompt_feature_with_head_alpha(
                        prompt=prompt,
                        feature_idx=feature_idx,
                        head_layer_idx=layer_idx,
                        head_idx=head_idx,
                        feature_layer_idx=feature_score_layer,
                        head_alpha=float(alpha),
                        previous_block_index=previous_block_index,
                        target_block_index=target_block_index,
                        token_scope=token_scope,
                        expected_num_blocks=NUM_BLOCKS_TOTAL,
                    )
                    previous_mean = intervened["previous_feature_mean"]
                    target_mean = intervened["target_feature_mean"]

                rows.append({
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "layer": int(layer_idx),
                    "head": int(head_idx),
                    "feature_score_layer": int(feature_score_layer),
                    "token_scope": token_scope,
                    "head_alpha": float(alpha),
                    "effective_head_multiplier": float(1.0 + float(alpha)),
                    "baseline_source": baseline_source,
                    "baseline_previous_feature_mean": float(base_prev),
                    "baseline_target_feature_mean": float(base_tgt),
                    "intervened_previous_feature_mean": float(previous_mean),
                    "intervened_target_feature_mean": float(target_mean),
                    "target_delta_intervened_minus_baseline": float(target_mean - base_tgt),
                    "previous_delta_intervened_minus_baseline": float(previous_mean - base_prev),
                })

        if verbose_every is not None and (query_counter % max(verbose_every, 1) == 0 or query_counter == len(query_indices)):
            print(f"processed query {query_counter}/{len(query_indices)}; rows={len(rows)}")

    records_df = pd.DataFrame(rows)

    # Baseline correlation summary from alpha 0.0 rows.
    baseline_rows = records_df[records_df["head_alpha"] == 0.0].copy()
    baseline_pooled_r, baseline_mean_query_r, baseline_std_query_r, baseline_n_query = _querywise_corr_summary(
        baseline_rows,
        target_col="baseline_target_feature_mean",
        prev_col="baseline_previous_feature_mean",
    )

    if use_cached_baseline and np.isfinite(cached_ref["mean_query_corr"]):
        diff = abs(float(baseline_mean_query_r) - float(cached_ref["mean_query_corr"]))
        if diff > 1e-12:
            raise AssertionError(
                "Causal baseline does not match Section 7 cached LFH correlation. "
                f"baseline_mean_query_r={baseline_mean_query_r:.17g}, "
                f"cached_ref={cached_ref['mean_query_corr']:.17g}, diff={diff:.3g}."
            )
        if max_queries is None and max_variants is None and np.isfinite(cached_ref["final_rho_df_full_corr"]):
            diff_full = abs(float(baseline_mean_query_r) - float(cached_ref["final_rho_df_full_corr"]))
            if diff_full > 1e-12:
                raise AssertionError(
                    "Causal alpha=0 baseline does not match Final_rho_df on the full prompt set. "
                    f"baseline_mean_query_r={baseline_mean_query_r:.17g}, "
                    f"Final_rho_df={cached_ref['final_rho_df_full_corr']:.17g}, diff={diff_full:.3g}."
                )

    summary_rows = []
    for alpha in requested_alphas:
        sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
        if len(sub) == 0:
            continue

        pooled_r, mean_query_r, std_query_r, n_query = _querywise_corr_summary(
            sub,
            target_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
        )

        delta_pooled_r, delta_mean_query_r, delta_std_query_r, delta_n_query = _querywise_corr_summary(
            sub,
            target_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
        )

        high_low_target_delta_mean, high_low_target_delta_std, high_low_n = _high_low_delta_by_query(
            sub,
            value_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        high_low_target_mean, high_low_target_std, high_low_n = _high_low_delta_by_query(
            sub,
            value_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        alpha0_mean_diff = np.nan
        alpha0_pooled_diff = np.nan
        if float(alpha) == 0.0:
            alpha0_mean_diff = float(mean_query_r - baseline_mean_query_r)
            alpha0_pooled_diff = float(pooled_r - baseline_pooled_r)

        summary_rows.append({
            "feature_idx": int(feature_idx),
            "layer": int(layer_idx),
            "head": int(head_idx),
            "feature_score_layer": int(feature_score_layer),
            "token_scope": token_scope,
            "head_alpha": float(alpha),
            "effective_head_multiplier": float(1.0 + float(alpha)),
            "baseline_source": "section7_cached_lfh_tensors" if use_cached_baseline else "recomputed_forward_pass",

            "baseline_pooled_corr_prev_to_target": baseline_pooled_r,
            "baseline_mean_query_corr_prev_to_target": baseline_mean_query_r,
            "baseline_std_query_corr_prev_to_target": baseline_std_query_r,

            "intervened_pooled_corr_prev_to_target": pooled_r,
            "intervened_mean_query_corr_prev_to_target": mean_query_r,
            "intervened_std_query_corr_prev_to_target": std_query_r,

            "corr_change_pooled_intervened_minus_baseline": pooled_r - baseline_pooled_r,
            "corr_change_mean_query_intervened_minus_baseline": mean_query_r - baseline_mean_query_r,
            "alpha0_pooled_corr_minus_baseline": alpha0_pooled_diff,
            "alpha0_mean_query_corr_minus_baseline": alpha0_mean_diff,

            "section7_cached_subset_pooled_corr": cached_ref["pooled_corr"],
            "section7_cached_subset_mean_query_corr": cached_ref["mean_query_corr"],
            "section7_cached_subset_std_query_corr": cached_ref["std_query_corr"],
            "section7_cached_subset_n_queries": int(cached_ref["n_queries"]),
            "final_rho_df_full_avg_querywise_pearson_r": cached_ref["final_rho_df_full_corr"],

            "corr_prev_to_target_delta_pooled": delta_pooled_r,
            "corr_prev_to_target_delta_mean_query": delta_mean_query_r,

            "mean_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].mean()),
            "median_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].median()),
            "fraction_target_delta_positive": float((sub["target_delta_intervened_minus_baseline"] > 0).mean()),

            "high_minus_low_prev_target_delta_mean": high_low_target_delta_mean,
            "high_minus_low_prev_target_delta_std": high_low_target_delta_std,
            "high_minus_low_prev_intervened_target_mean": high_low_target_mean,
            "high_minus_low_prev_intervened_target_std": high_low_target_std,
            "n_queries_for_high_low": int(high_low_n),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("head_alpha").reset_index(drop=True)

    print("=" * 120)
    print(f"Head causal test: feature={feature_idx}, layer={layer_idx}, head={head_idx}")
    print(f"Baseline pooled corr(prev, target): {baseline_pooled_r:.12f}")
    print(f"Baseline mean query-wise corr(prev, target): {baseline_mean_query_r:.12f}")
    if use_cached_baseline:
        print(f"Section 7 cached subset mean query-wise corr: {cached_ref['mean_query_corr']:.12f}")
        if np.isfinite(cached_ref["final_rho_df_full_corr"]):
            print(f"Final_rho_df full avg_querywise_pearson_r: {cached_ref['final_rho_df_full_corr']:.12f}")
    print("Interpretation:")
    print("  alpha=0.0 leaves this head unchanged on target tokens.")
    print("  alpha=-1.0 ablates this head on target tokens.")
    print("  alpha>0 adds extra copies of this head's contribution on target tokens.")
    print("  effective multiplier = 1 + alpha.")
    print("  If the head implements negative write, ablation should make a negative corr less negative,")
    print("  while amplification should make it more negative.")
    print("=" * 120)
    display(summary_df)

    return summary_df, records_df


# ------------------------------------------------------------
# Plotting helpers (Updated with Custom Font Properties)
# ------------------------------------------------------------

# aaaaa
def plot_head_causal_dose_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["intervened_mean_query_corr_prev_to_target"],
        mode="lines+markers",
        name="intervened corr(prev, target), mean query-wise",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    baseline = float(summary_df["baseline_mean_query_corr_prev_to_target"].iloc[0])
    fig.add_hline(y=baseline, line_dash="dash", annotation_text="baseline corr")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title=(
            "Attention-Head Intervention Response on Local Feature Homeostasis<br>"
            f"Model={MODEL_SHORT_NAME}, Feature={int(summary_df['feature_idx'].iloc[0])}, "
            f"Layer={int(summary_df['layer'].iloc[0])}, "
            f"Head={int(summary_df['head'].iloc[0])}"
        ),
        title_font_size=26,
        xaxis_title="Intervention Strength (Alpha) on Target-Block Tokens",
        xaxis_title_font_size=22,
        yaxis_title="Mean corr(source feature, target feature)",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    
    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'head_causal_dose_response',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("head_causal_dose_response_highres.png", scale=4)

    return fig


def plot_head_causal_delta_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["corr_prev_to_target_delta_mean_query"],
        mode="lines+markers",
        name="corr(prev, target_delta)",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["high_minus_low_prev_target_delta_mean"],
        mode="lines+markers",
        name="high-prev minus low-prev target delta",
        hovertemplate="alpha=%{x}<br>high-low delta=%{y:.4f}<extra></extra>",
    ))

    fig.add_hline(y=0.0, line_dash="dash")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title="Does the intervention affect high-previous-activation variants differently?",
        title_font_size=26,
        xaxis_title="Head alpha on target-block tokens (effective multiplier = 1 + alpha)",
        xaxis_title_font_size=22,
        yaxis_title="Effect size",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    fig.show()
    return fig


def plot_head_intervention_scatter(records_df, alpha: float):
    sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
    if len(sub) == 0:
        raise ValueError(f"No rows for alpha={alpha}")

    x = sub["baseline_previous_feature_mean"].to_numpy(dtype=np.float64)
    y_base = sub["baseline_target_feature_mean"].to_numpy(dtype=np.float64)
    y_int = sub["intervened_target_feature_mean"].to_numpy(dtype=np.float64)

    r_base = _safe_pearson_local(x, y_base)
    r_int = _safe_pearson_local(x, y_int)

    df_plot = pd.DataFrame({
        "previous_feature_mean": np.concatenate([x, x]),
        "target_feature_mean": np.concatenate([y_base, y_int]),
        "condition": ["baseline"] * len(x) + [f"head_alpha={alpha}"] * len(x),
        "query_idx": np.concatenate([sub["query_idx"].values, sub["query_idx"].values]).astype(str),
        "variant_idx": np.concatenate([sub["variant_idx"].values, sub["variant_idx"].values]),
    })

    fig = px.scatter(
        df_plot,
        x="previous_feature_mean",
        y="target_feature_mean",
        color="condition",
        symbol="condition",
        hover_data=["query_idx", "variant_idx"],
        title=(
            f"Source-to-target feature relation before/after head intervention<br>"
            f"alpha={alpha}, baseline r={r_base:.3f}, intervened r={r_int:.3f}"
        ),
    )

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title_font_size=26,
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source exemplar",
        xaxis_title_font_size=22,
        yaxis_title="Mean SAE pre-activation on target exemplar",
        yaxis_title_font_size=22,
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the head intervention test
# ------------------------------------------------------------

head_causal_summary_df, head_causal_records_df = run_attention_head_alpha_causal_test(
    feature_idx=HEAD_CAUSAL_FEATURE_IDX,
    layer_idx=HEAD_CAUSAL_LAYER,
    head_idx=HEAD_CAUSAL_HEAD,
    prompts_all_agnews=prompts_all_agnews,
    feature_score_layer=HEAD_CAUSAL_FEATURE_SCORE_LAYER,
    head_alphas=HEAD_CAUSAL_HEAD_ALPHAS,
    previous_block_index=HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX,
    target_block_index=HEAD_CAUSAL_TARGET_BLOCK_INDEX,
    token_scope=HEAD_CAUSAL_TOKEN_SCOPE,
    max_queries=HEAD_CAUSAL_MAX_QUERIES,
    max_variants=HEAD_CAUSAL_MAX_VARIANTS,
    quantile=HEAD_CAUSAL_QUANTILE,
    verbose_every=5,
)

_ = plot_head_causal_dose_response(head_causal_summary_df)
_ = plot_head_causal_delta_response(head_causal_summary_df)

# Plot ablation and amplification views if present.
if -1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=-1.0)
if 1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=1.0)

# 16. Feature-specific causal workflow for top-20 LFH features

Use this section when you want to run the existing attention-head causal intervention on a particular LFH feature.

Workflow:

1. Run the first code cell below. Set `LFH_CAUSAL_FEATURE_IDX` to one of the top-20 LFH features, then it displays the top-20 candidate attention heads for that feature.
2. Run the second code cell below. Set `LFH_CAUSAL_SELECTED_LAYER` and `LFH_CAUSAL_SELECTED_HEAD` from the displayed table, then it runs the same head ablation/amplification causal experiment used above.

By default, the head search scans only `TARGET_LAYER` for tractability. To scan every causally upstream layer, set `LFH_HEAD_SEARCH_LAYERS = list(range(TARGET_LAYER + 1))`; only heads at or before the SAE score layer can affect the measured residual activation in the same forward pass.

In [ ]:
# ============================================================
# 16a. Pick one top-20 LFH feature and show its top-20 candidate heads
# ============================================================

import numpy as np
import pandas as pd
import torch
from IPython.display import display

_required_for_lfh_head_search = [
    "Final_rho_df",
    "prompts_all_agnews",
    "sae_single_feature_pre_activation",
    "get_head_projection_vectors_for_feature",
    "_summarize_head_write_test",
    "tokenize_prompt_for_offsets",
    "get_example_block_token_spans",
    "token_selection_to_torch_indices",
]
_missing_for_lfh_head_search = [name for name in _required_for_lfh_head_search if name not in globals()]
if len(_missing_for_lfh_head_search) > 0:
    raise RuntimeError(
        "Run the LFH feature-ranking cells and Section 12 attention-head helper cell before this workflow. "
        f"Missing: {_missing_for_lfh_head_search}"
    )

# ------------------------------------------------------------
# User-editable config: choose the LFH feature first
# ------------------------------------------------------------

LFH_TOP_N_FEATURES_FOR_CAUSAL = 20
lfh_top20_features_df = Final_rho_df.head(LFH_TOP_N_FEATURES_FOR_CAUSAL).copy()

print("Top-20 LFH features by strongest negative source->target correlation")
display(lfh_top20_features_df)

# Edit this to any feature_idx in the table above.
# Default: strongest negative-correlation LFH feature.
LFH_CAUSAL_FEATURE_IDX = int(lfh_top20_features_df.iloc[0]["feature_idx"])

# Candidate attention layers to search before choosing one head for causal intervention.
# Default scans all layers that can causally affect the TARGET_LAYER SAE activation.
# For a faster smoke test, use: LFH_HEAD_SEARCH_LAYERS = [TARGET_LAYER]
LFH_HEAD_SEARCH_LAYERS = [TARGET_LAYER]

# Match the existing causal/attention-write cells unless you explicitly want more data.
LFH_HEAD_SEARCH_MAX_QUERIES = min(10, len(prompts_all_agnews))
LFH_HEAD_SEARCH_MAX_VARIANTS = None
LFH_HEAD_SEARCH_TOP_N_HEADS = 20
LFH_HEAD_SEARCH_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
LFH_HEAD_SEARCH_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)
LFH_HEAD_SEARCH_FEATURE_SCORE_LAYER = int(TARGET_LAYER)
LFH_HEAD_SEARCH_QUANTILE = 0.25
LFH_HEAD_SEARCH_VERBOSE_EVERY = 5

# _summarize_head_write_test uses this global quantile in the original notebook.
ATTN_WRITE_TEST_QUANTILE = float(LFH_HEAD_SEARCH_QUANTILE)

lfh_top20_feature_ids = set(lfh_top20_features_df["feature_idx"].astype(int).tolist())
if int(LFH_CAUSAL_FEATURE_IDX) not in lfh_top20_feature_ids:
    print(
        f"Warning: LFH_CAUSAL_FEATURE_IDX={LFH_CAUSAL_FEATURE_IDX} is not in the current top-20 LFH table. "
        "The code will still run, but this is outside the requested top-20 feature set."
    )

print("=" * 120)
print("Selected LFH feature for head search:", LFH_CAUSAL_FEATURE_IDX)
print("Head-search layers:", LFH_HEAD_SEARCH_LAYERS)
print("Feature score layer:", LFH_HEAD_SEARCH_FEATURE_SCORE_LAYER)
print("Max queries:", LFH_HEAD_SEARCH_MAX_QUERIES)
print("Max variants:", LFH_HEAD_SEARCH_MAX_VARIANTS)
print("=" * 120)


@torch.no_grad()
def capture_feature_layer_output_and_head_o_proj_input(
    model,
    input_ids,
    attention_mask,
    *,
    feature_layer_idx: int,
    head_layer_idx: int,
):
    """
    Capture two tensors in one forward pass:
      - residual output at feature_layer_idx, used to score the SAE feature;
      - input to self_attn.o_proj at head_layer_idx, split later by attention head.

    This generalizes the original single-layer head-write diagnostic so that
    candidate heads from layers <= TARGET_LAYER can be ranked for the same
    TARGET_LAYER SAE feature activation.
    """
    if int(head_layer_idx) > int(feature_layer_idx):
        raise ValueError(
            f"head_layer_idx={head_layer_idx} is later than feature_layer_idx={feature_layer_idx}. "
            "It cannot causally affect that feature-layer activation in the same forward pass."
        )

    cache = {}
    feature_block = model.get_submodule(f"model.layers.{int(feature_layer_idx)}")
    o_proj = model.get_submodule(f"model.layers.{int(head_layer_idx)}.self_attn.o_proj")

    def feature_block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["feature_layer_output"] = x.detach()
        return output

    def o_proj_pre_hook(module, inputs):
        cache["head_o_proj_input"] = inputs[0].detach()
        return None

    h_feature = feature_block.register_forward_hook(feature_block_hook)
    h_head = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    finally:
        h_head.remove()
        h_feature.remove()

    if "feature_layer_output" not in cache:
        raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")
    if "head_o_proj_input" not in cache:
        raise RuntimeError(f"Failed to capture layer {head_layer_idx} o_proj input.")

    return cache["feature_layer_output"], cache["head_o_proj_input"]


@torch.no_grad()
def measure_prompt_feature_and_candidate_head_writes(
    prompt: str,
    *,
    feature_idx: int,
    head_layer_idx: int,
    feature_layer_idx: int = TARGET_LAYER,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt variant, measure:
      - source-block and target-block SAE preactivation for feature_idx at feature_layer_idx;
      - every head at head_layer_idx's signed target-block write into the same SAE feature directions.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    prev_span = block_spans[int(previous_block_index)]
    tgt_span = block_spans[int(target_block_index)]

    feature_layer_output, head_o_proj_input = capture_feature_layer_output_and_head_o_proj_input(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        feature_layer_idx=int(feature_layer_idx),
        head_layer_idx=int(head_layer_idx),
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=feature_layer_output,
        feature_idx=int(feature_idx),
    )[0].detach().float()

    prev_idx = token_selection_to_torch_indices(prev_span, device=pre_f.device)
    tgt_idx = token_selection_to_torch_indices(tgt_span, device=pre_f.device)
    prev_feature_mean = float(pre_f.index_select(0, prev_idx).mean().detach().cpu())
    target_feature_mean = float(pre_f.index_select(0, tgt_idx).mean().detach().cpu())

    head_geom = get_head_projection_vectors_for_feature(
        model=model,
        sae=sae,
        layer_idx=int(head_layer_idx),
        feature_idx=int(feature_idx),
    )

    num_heads = int(head_geom["num_heads"])
    head_dim = int(head_geom["head_dim"])
    head_vec_read = head_geom["head_vec_read"].to(device=head_o_proj_input.device)
    head_vec_write = head_geom["head_vec_write"].to(device=head_o_proj_input.device)

    tgt_idx_o = token_selection_to_torch_indices(tgt_span, device=head_o_proj_input.device)
    z = head_o_proj_input[0].index_select(0, tgt_idx_o)
    z_heads = z.reshape(tgt_idx_o.numel(), num_heads, head_dim).float()

    target_head_write_to_read = torch.einsum("thd,hd->th", z_heads, head_vec_read.float()).mean(dim=0)
    target_head_write_to_write = torch.einsum("thd,hd->th", z_heads, head_vec_write.float()).mean(dim=0)

    return {
        "prev_feature_mean": prev_feature_mean,
        "target_feature_mean": target_feature_mean,
        "target_head_write_to_read": target_head_write_to_read.detach().cpu().numpy(),
        "target_head_write_to_write": target_head_write_to_write.detach().cpu().numpy(),
        "prev_span": prev_span,
        "target_span": tgt_span,
        "num_heads": num_heads,
    }


@torch.no_grad()
def run_lfh_head_search_for_feature(
    *,
    feature_idx: int,
    candidate_layers,
    prompts_all_agnews,
    feature_layer_idx: int = TARGET_LAYER,
    previous_block_index: int = LFH_SOURCE_BLOCK_INDEX,
    target_block_index: int = LFH_TARGET_BLOCK_INDEX,
    max_queries=None,
    max_variants=None,
    top_n: int = 20,
    verbose_every: int | None = 5,
):
    """
    Rank candidate (layer, head) pairs for one LFH feature using the same
    negative-write score as the original attention-head diagnostic.
    """
    candidate_layers = [int(x) for x in candidate_layers]
    bad_layers = [x for x in candidate_layers if x < 0 or x > int(feature_layer_idx)]
    if len(bad_layers) > 0:
        raise ValueError(
            f"candidate_layers contains invalid causal layers {bad_layers}; "
            f"all layers must be in [0, {int(feature_layer_idx)}]."
        )

    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(int(max_queries), num_queries_total)))

    all_summary_dfs = []
    all_prompt_records = {}
    all_head_mats = {}

    for layer_idx in candidate_layers:
        print("=" * 120)
        print(f"Head search for feature {feature_idx}: candidate attention layer {layer_idx}")
        print("=" * 120)

        prompt_rows = []
        read_rows = []
        write_rows = []

        for query_pos, query_idx in enumerate(query_indices, start=1):
            variants = prompts_all_agnews[query_idx]
            if max_variants is not None:
                variants = variants[:int(max_variants)]

            for variant_idx, prompt in enumerate(variants):
                m = measure_prompt_feature_and_candidate_head_writes(
                    prompt=prompt,
                    feature_idx=int(feature_idx),
                    head_layer_idx=int(layer_idx),
                    feature_layer_idx=int(feature_layer_idx),
                    previous_block_index=int(previous_block_index),
                    target_block_index=int(target_block_index),
                    expected_num_blocks=NUM_BLOCKS_TOTAL,
                )

                prompt_rows.append({
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "layer": int(layer_idx),
                    "feature_score_layer": int(feature_layer_idx),
                    "prev_feature_mean": m["prev_feature_mean"],
                    "target_feature_mean": m["target_feature_mean"],
                    "prev_span_start": int(m["prev_span"][0]),
                    "prev_span_end": int(m["prev_span"][1]),
                    "target_span_start": int(m["target_span"][0]),
                    "target_span_end": int(m["target_span"][1]),
                })
                read_rows.append(m["target_head_write_to_read"])
                write_rows.append(m["target_head_write_to_write"])

            if verbose_every is not None and (query_pos % max(int(verbose_every), 1) == 0 or query_pos == len(query_indices)):
                print(f"  processed query {query_pos}/{len(query_indices)}; prompt rows={len(prompt_rows)}")

        prompt_records_df = pd.DataFrame(prompt_rows)
        head_read_mat = np.stack(read_rows, axis=0)
        head_write_mat = np.stack(write_rows, axis=0)

        summary_df = _summarize_head_write_test(
            prompt_records_df=prompt_records_df,
            head_read_mat=head_read_mat,
            head_write_mat=head_write_mat,
            layer_idx=int(layer_idx),
            feature_idx=int(feature_idx),
        )
        summary_df["feature_score_layer"] = int(feature_layer_idx)

        all_summary_dfs.append(summary_df)
        all_prompt_records[int(layer_idx)] = prompt_records_df
        all_head_mats[int(layer_idx)] = {
            "read": head_read_mat,
            "write": head_write_mat,
        }

    combined = pd.concat(all_summary_dfs, ignore_index=True)
    combined = combined.sort_values(
        ["anti_write_score_read", "corr_prev_to_target_head_write_to_negative_read_mean_query"],
        ascending=False,
    ).reset_index(drop=True)
    combined["rank_across_layers"] = np.arange(1, len(combined) + 1)

    display_cols = [
        "rank_across_layers",
        "layer",
        "head",
        "feature_idx",
        "feature_score_layer",
        "anti_write_score_read",
        "corr_prev_to_target_head_write_to_negative_read_mean_query",
        "high_minus_low_head_write_to_negative_read_z_mean",
        "corr_head_write_to_read_with_target_feature_mean_query",
        "feature_prev_to_target_corr_mean_query",
        "feature_prev_to_target_corr_pooled",
    ]
    display_cols = [c for c in display_cols if c in combined.columns]

    print("=" * 120)
    print(f"Top-{top_n} candidate heads for LFH feature {feature_idx}")
    print("Choose LFH_CAUSAL_SELECTED_LAYER and LFH_CAUSAL_SELECTED_HEAD from this table in the next cell.")
    print("=" * 120)
    display(combined.loc[: int(top_n) - 1, display_cols])

    return combined, all_prompt_records, all_head_mats


lfh_feature_head_search_df, lfh_feature_head_search_prompt_records, lfh_feature_head_search_head_mats = run_lfh_head_search_for_feature(
    feature_idx=int(LFH_CAUSAL_FEATURE_IDX),
    candidate_layers=LFH_HEAD_SEARCH_LAYERS,
    prompts_all_agnews=prompts_all_agnews,
    feature_layer_idx=int(LFH_HEAD_SEARCH_FEATURE_SCORE_LAYER),
    previous_block_index=int(LFH_HEAD_SEARCH_PREVIOUS_BLOCK_INDEX),
    target_block_index=int(LFH_HEAD_SEARCH_TARGET_BLOCK_INDEX),
    max_queries=LFH_HEAD_SEARCH_MAX_QUERIES,
    max_variants=LFH_HEAD_SEARCH_MAX_VARIANTS,
    top_n=int(LFH_HEAD_SEARCH_TOP_N_HEADS),
    verbose_every=LFH_HEAD_SEARCH_VERBOSE_EVERY,
)

lfh_top20_candidate_heads_df = lfh_feature_head_search_df.head(int(LFH_HEAD_SEARCH_TOP_N_HEADS)).copy()
LFH_CAUSAL_DEFAULT_LAYER = int(lfh_top20_candidate_heads_df.iloc[0]["layer"])
LFH_CAUSAL_DEFAULT_HEAD = int(lfh_top20_candidate_heads_df.iloc[0]["head"])

print("Default selected head for the causal cell below:", (LFH_CAUSAL_DEFAULT_LAYER, LFH_CAUSAL_DEFAULT_HEAD))

In [ ]:
# ============================================================
# 16b. Select one displayed head, then run the existing causal intervention
# ============================================================

_required_for_lfh_causal_run = [
    "run_attention_head_alpha_causal_test",
    "plot_head_causal_dose_response",
    "plot_head_causal_delta_response",
    "plot_head_intervention_scatter",
    "lfh_top20_candidate_heads_df",
    "LFH_CAUSAL_DEFAULT_LAYER",
    "LFH_CAUSAL_DEFAULT_HEAD",
]
_missing_for_lfh_causal_run = [name for name in _required_for_lfh_causal_run if name not in globals()]
if len(_missing_for_lfh_causal_run) > 0:
    raise RuntimeError(
        "Run Section 14 causal helper cell and the immediately preceding feature-specific head-search cell before this cell. "
        f"Missing: {_missing_for_lfh_causal_run}"
    )

# ------------------------------------------------------------
# User-editable config: choose the layer/head from lfh_top20_candidate_heads_df
# ------------------------------------------------------------

# Defaults use the top row from the previous cell. Edit these two lines after inspecting the table.
LFH_CAUSAL_SELECTED_LAYER = int(LFH_CAUSAL_DEFAULT_LAYER)
LFH_CAUSAL_SELECTED_HEADS = [int(LFH_CAUSAL_DEFAULT_HEAD)]

for LFH_CAUSAL_SELECTED_HEAD in LFH_CAUSAL_SELECTED_HEADS:
    # Same intervention settings as the existing causal section, but namespaced for this feature-specific workflow.
    LFH_CAUSAL_FEATURE_SCORE_LAYER = int(TARGET_LAYER)
    LFH_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0]
    LFH_CAUSAL_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
    LFH_CAUSAL_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)
    LFH_CAUSAL_TOKEN_SCOPE = "target_block"
    LFH_CAUSAL_MAX_QUERIES = None  # full Section 7 prompt set; set smaller manually only for a smoke test
    LFH_CAUSAL_MAX_VARIANTS = None  # all variants, matching Section 7 by default
    LFH_CAUSAL_QUANTILE = 0.25

    if int(LFH_CAUSAL_SELECTED_LAYER) > int(LFH_CAUSAL_FEATURE_SCORE_LAYER):
        raise ValueError(
            f"LFH_CAUSAL_SELECTED_LAYER={LFH_CAUSAL_SELECTED_LAYER} is later than "
            f"LFH_CAUSAL_FEATURE_SCORE_LAYER={LFH_CAUSAL_FEATURE_SCORE_LAYER}. "
            "Choose a head layer <= TARGET_LAYER."
        )

    print("=" * 120)
    print("Feature-specific LFH causal intervention config")
    print("=" * 120)
    print("feature_idx:", LFH_CAUSAL_FEATURE_IDX)
    print("attention layer:", LFH_CAUSAL_SELECTED_LAYER)
    print("attention head:", LFH_CAUSAL_SELECTED_HEAD)
    print("feature score layer:", LFH_CAUSAL_FEATURE_SCORE_LAYER)
    print("token scope:", LFH_CAUSAL_TOKEN_SCOPE)
    print("head alphas:", LFH_CAUSAL_HEAD_ALPHAS)
    print("max queries:", LFH_CAUSAL_MAX_QUERIES)
    print("max variants:", LFH_CAUSAL_MAX_VARIANTS)
    print("=" * 120)

    lfh_feature_specific_causal_summary_df, lfh_feature_specific_causal_records_df = run_attention_head_alpha_causal_test(
        feature_idx=int(LFH_CAUSAL_FEATURE_IDX),
        layer_idx=int(LFH_CAUSAL_SELECTED_LAYER),
        head_idx=int(LFH_CAUSAL_SELECTED_HEAD),
        prompts_all_agnews=prompts_all_agnews,
        feature_score_layer=int(LFH_CAUSAL_FEATURE_SCORE_LAYER),
        head_alphas=LFH_CAUSAL_HEAD_ALPHAS,
        previous_block_index=int(LFH_CAUSAL_PREVIOUS_BLOCK_INDEX),
        target_block_index=int(LFH_CAUSAL_TARGET_BLOCK_INDEX),
        token_scope=LFH_CAUSAL_TOKEN_SCOPE,
        max_queries=LFH_CAUSAL_MAX_QUERIES,
        max_variants=LFH_CAUSAL_MAX_VARIANTS,
        quantile=float(LFH_CAUSAL_QUANTILE),
        verbose_every=5,
    )

    _ = plot_head_causal_dose_response(lfh_feature_specific_causal_summary_df)
    _ = plot_head_causal_delta_response(lfh_feature_specific_causal_summary_df)

    if -1.0 in set(lfh_feature_specific_causal_records_df["head_alpha"].astype(float)):
        _ = plot_head_intervention_scatter(lfh_feature_specific_causal_records_df, alpha=-1.0)
    if 1.0 in set(lfh_feature_specific_causal_records_df["head_alpha"].astype(float)):
        _ = plot_head_intervention_scatter(lfh_feature_specific_causal_records_df, alpha=1.0)

# Geometric Signature

In [ ]:
# ============================================================
# Weight-only OV anti-write analysis for top LFH features
#
# Required existing notebook objects:
#   model, sae, TARGET_LAYER, Final_rho_df
#
# Main outputs:
#   ov_all_feature_df
#   lfh_ov_alignment_df
#   paper_lfh_ov_table_df
#   ov_residual_cosine_matrix
#   ov_headspace_cosine_matrix
#   ov_head_metadata_df
# ============================================================

import gc
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import rankdata


# ------------------------------------------------------------
# User-editable configuration
# ------------------------------------------------------------

# Number of strongest LFH features to report.
OV_TOP_N_LFH_FEATURES = 10

# By default, test attention heads in the same transformer block
# whose resid_post representation is encoded by the SAE.
OV_CANDIDATE_LAYERS = [int(TARGET_LAYER)]

# Alternatives:
# OV_CANDIDATE_LAYERS = list(range(int(TARGET_LAYER) + 1))
# scans every attention layer that can precede the SAE measurement.

# Ranking metric:
#
# "residual_cosine"
#     cos(W_O^h W_V^h d_f, d_f)
#     This exactly matches the equation in the paper.
#
# "headspace_cosine"
#     cos(W_V^h d_f, (W_O^h)^T d_f)
#     This matches the description "value-side feature image versus
#     output-side feature readout."
OV_RANK_METRIC = "headspace_cosine"

# Chunk over SAE features to control GPU memory.
OV_FEATURE_CHUNK_SIZE = 1024

# Percentiles at or above this value are bolded in the LaTeX table.
OV_HIGHLIGHT_PERCENTILE = 0.95

# Small denominator guard.
OV_EPS = 1e-12


if OV_RANK_METRIC not in {
    "residual_cosine",
    "headspace_cosine",
}:
    raise ValueError(
        "OV_RANK_METRIC must be 'residual_cosine' or "
        f"'headspace_cosine'; got {OV_RANK_METRIC!r}."
    )

OV_CANDIDATE_LAYERS = [
    int(layer_idx)
    for layer_idx in OV_CANDIDATE_LAYERS
]

if len(OV_CANDIDATE_LAYERS) == 0:
    raise ValueError("OV_CANDIDATE_LAYERS must be nonempty.")

if len(set(OV_CANDIDATE_LAYERS)) != len(OV_CANDIDATE_LAYERS):
    raise ValueError(
        "OV_CANDIDATE_LAYERS contains duplicate layers."
    )

if OV_FEATURE_CHUNK_SIZE <= 0:
    raise ValueError("OV_FEATURE_CHUNK_SIZE must be positive.")


# ------------------------------------------------------------
# Model-layout helpers
# ------------------------------------------------------------

def _get_transformer_layer(
    model: torch.nn.Module,
    layer_idx: int,
) -> torch.nn.Module:
    """
    Retrieve a Hugging Face decoder layer for Gemma, Llama, or Qwen.
    """
    candidate_paths = [
        f"model.layers.{int(layer_idx)}",
        f"transformer.h.{int(layer_idx)}",
    ]

    for path in candidate_paths:
        try:
            return model.get_submodule(path)
        except (AttributeError, KeyError):
            continue

    raise AttributeError(
        f"Could not locate transformer layer {layer_idx}. "
        f"Tried: {candidate_paths}"
    )


def _get_attention_module(
    model: torch.nn.Module,
    layer_idx: int,
) -> torch.nn.Module:
    layer = _get_transformer_layer(model, layer_idx)

    for name in ["self_attn", "attn", "attention"]:
        if hasattr(layer, name):
            return getattr(layer, name)

    raise AttributeError(
        f"Could not find an attention module in layer {layer_idx}."
    )


def _get_model_text_config(model: torch.nn.Module):
    """
    Some multimodal/config wrappers place language-model fields under
    config.text_config; ordinary causal LMs place them directly in config.
    """
    return getattr(model.config, "text_config", model.config)


def _infer_attention_layout(
    model: torch.nn.Module,
    layer_idx: int,
) -> Dict[str, int]:
    """
    Infer query-head, key/value-head, and head-dimension counts directly
    from the model configuration and V/O weight shapes.

    For grouped-query attention, several query heads share one value head.
    """
    attn = _get_attention_module(model, layer_idx)

    if not hasattr(attn, "v_proj") or not hasattr(attn, "o_proj"):
        raise AttributeError(
            f"Layer {layer_idx} attention must expose v_proj and o_proj."
        )

    W_V = attn.v_proj.weight
    W_O = attn.o_proj.weight

    if W_V.ndim != 2 or W_O.ndim != 2:
        raise ValueError(
            "Expected two-dimensional v_proj/o_proj weights, got "
            f"{tuple(W_V.shape)} and {tuple(W_O.shape)}."
        )

    config = _get_model_text_config(model)

    num_query_heads = getattr(
        attn,
        "num_heads",
        getattr(config, "num_attention_heads", None),
    )

    if num_query_heads is None:
        raise AttributeError(
            "Could not infer num_attention_heads from the attention "
            "module or model configuration."
        )

    num_query_heads = int(num_query_heads)

    # Hugging Face Linear convention:
    #   o_proj.weight: [d_model, num_query_heads * head_dim]
    #   v_proj.weight: [num_kv_heads * head_dim, d_model]
    o_input_width = int(W_O.shape[1])

    if o_input_width % num_query_heads != 0:
        raise ValueError(
            f"o_proj input width {o_input_width} is not divisible by "
            f"num_query_heads={num_query_heads}."
        )

    head_dim = o_input_width // num_query_heads

    if int(W_V.shape[0]) % head_dim != 0:
        raise ValueError(
            f"v_proj output width {W_V.shape[0]} is not divisible by "
            f"head_dim={head_dim}."
        )

    num_kv_heads = int(W_V.shape[0]) // head_dim

    if num_query_heads % num_kv_heads != 0:
        raise ValueError(
            f"num_query_heads={num_query_heads} must be divisible by "
            f"num_kv_heads={num_kv_heads}."
        )

    query_heads_per_kv_head = (
        num_query_heads // num_kv_heads
    )

    d_model = int(W_O.shape[0])

    if int(W_V.shape[1]) != d_model:
        raise ValueError(
            f"V/O residual widths disagree: W_V={tuple(W_V.shape)}, "
            f"W_O={tuple(W_O.shape)}."
        )

    return {
        "layer": int(layer_idx),
        "d_model": d_model,
        "num_query_heads": num_query_heads,
        "num_kv_heads": num_kv_heads,
        "query_heads_per_kv_head": query_heads_per_kv_head,
        "head_dim": head_dim,
    }


def _query_head_to_kv_head(
    query_head: int,
    *,
    num_query_heads: int,
    num_kv_heads: int,
) -> int:
    """
    Map a query head to its shared key/value head under grouped-query
    attention. For ordinary multi-head attention, this is the identity.
    """
    query_head = int(query_head)

    if not 0 <= query_head < int(num_query_heads):
        raise ValueError(
            f"query_head={query_head} outside "
            f"[0, {num_query_heads})."
        )

    if int(num_query_heads) % int(num_kv_heads) != 0:
        raise ValueError(
            "num_query_heads must be divisible by num_kv_heads."
        )

    group_size = int(num_query_heads) // int(num_kv_heads)
    return query_head // group_size


# ------------------------------------------------------------
# Percentile helper
# ------------------------------------------------------------

def _negative_tail_percentile(
    cosine_values: np.ndarray,
) -> np.ndarray:
    """
    Convert cosines into anti-write percentiles.

    More negative cosine => larger -cosine => higher percentile.

    A value of 0.99 means the feature is more strongly anti-aligned
    than approximately 99% of valid SAE features.
    """
    cosine_values = np.asarray(
        cosine_values,
        dtype=np.float64,
    )

    percentiles = np.full(
        cosine_values.shape,
        np.nan,
        dtype=np.float64,
    )

    valid = np.isfinite(cosine_values)

    if not np.any(valid):
        return percentiles

    antiwrite_strength = -cosine_values[valid]
    ranks = rankdata(
        antiwrite_strength,
        method="average",
    )

    percentiles[valid] = ranks / float(valid.sum())
    return percentiles


def _best_head_summary(
    cosine_matrix: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Find the most negative head cosine for every SAE feature.

    Returns
    -------
    best_cosine:
        [n_features]
    best_column:
        [n_features], column in the concatenated head matrix
    percentile:
        Negative-tail percentile among all SAE features
    """
    cosine_matrix = np.asarray(
        cosine_matrix,
        dtype=np.float64,
    )

    safe_matrix = np.where(
        np.isfinite(cosine_matrix),
        cosine_matrix,
        np.inf,
    )

    best_column = np.argmin(
        safe_matrix,
        axis=1,
    )

    feature_rows = np.arange(
        cosine_matrix.shape[0]
    )

    best_cosine = safe_matrix[
        feature_rows,
        best_column,
    ]

    valid_feature = np.isfinite(best_cosine)
    best_cosine = np.where(
        valid_feature,
        best_cosine,
        np.nan,
    )

    best_column = np.where(
        valid_feature,
        best_column,
        -1,
    )

    percentile = _negative_tail_percentile(
        best_cosine
    )

    return best_cosine, best_column, percentile


# ------------------------------------------------------------
# Core OV computation
# ------------------------------------------------------------

@torch.inference_mode()
def compute_weight_only_ov_geometry(
    *,
    model: torch.nn.Module,
    sae,
    candidate_layers: Sequence[int],
    feature_chunk_size: int = 1024,
    eps: float = 1e-12,
):
    """
    Compute two weight-only OV metrics for every SAE decoder feature
    and every candidate query head.

    Let d_f be a column-vector interpretation of SAE decoder feature f.

    1. Residual-space cosine:
       cos(W_O^h W_V^{kv(h)} d_f, d_f)

    2. Head-space cosine:
       cos(W_V^{kv(h)} d_f, (W_O^h)^T d_f)

    Biases, attention patterns, RoPE, and layer normalization are omitted.
    This is deliberately a static, weight-only circuit diagnostic.
    """
    candidate_layers = [
        int(layer_idx)
        for layer_idx in candidate_layers
    ]

    W_dec_cpu = (
        sae.W_dec
        .detach()
        .float()
        .cpu()
        .contiguous()
    )

    if W_dec_cpu.ndim != 2:
        raise ValueError(
            "Expected sae.W_dec to have shape "
            f"[n_features, d_model], got {tuple(W_dec_cpu.shape)}."
        )

    n_features, sae_d_model = map(
        int,
        W_dec_cpu.shape,
    )

    residual_cosine_blocks: List[np.ndarray] = []
    headspace_cosine_blocks: List[np.ndarray] = []
    head_metadata_rows: List[Dict[str, int]] = []

    print("=" * 100)
    print("Weight-only OV anti-write analysis")
    print("SAE features:", n_features)
    print("SAE residual width:", sae_d_model)
    print("Candidate layers:", candidate_layers)
    print("Feature chunk size:", feature_chunk_size)
    print("=" * 100)

    for layer_idx in candidate_layers:
        attn = _get_attention_module(
            model,
            layer_idx,
        )

        layout = _infer_attention_layout(
            model,
            layer_idx,
        )

        if layout["d_model"] != sae_d_model:
            raise ValueError(
                f"Layer {layer_idx} residual width "
                f"{layout['d_model']} does not match SAE width "
                f"{sae_d_model}."
            )

        compute_device = attn.o_proj.weight.device

        W_V = (
            attn.v_proj.weight
            .detach()
            .to(
                device=compute_device,
                dtype=torch.float32,
            )
        )

        W_O = (
            attn.o_proj.weight
            .detach()
            .to(
                device=compute_device,
                dtype=torch.float32,
            )
        )

        num_query_heads = layout["num_query_heads"]
        num_kv_heads = layout["num_kv_heads"]
        head_dim = layout["head_dim"]

        print(
            f"Layer {layer_idx}: "
            f"query heads={num_query_heads}, "
            f"KV heads={num_kv_heads}, "
            f"head_dim={head_dim}, "
            f"device={compute_device}"
        )

        layer_residual_cosine = np.full(
            (n_features, num_query_heads),
            np.nan,
            dtype=np.float32,
        )

        layer_headspace_cosine = np.full(
            (n_features, num_query_heads),
            np.nan,
            dtype=np.float32,
        )

        # Precompute O_h^T O_h for the transformed-vector norm.
        head_cache = []

        for query_head in range(num_query_heads):
            kv_head = _query_head_to_kv_head(
                query_head,
                num_query_heads=num_query_heads,
                num_kv_heads=num_kv_heads,
            )

            q_start = query_head * head_dim
            q_end = q_start + head_dim

            kv_start = kv_head * head_dim
            kv_end = kv_start + head_dim

            # W_O^h: [d_model, head_dim]
            O_h = W_O[:, q_start:q_end]

            # W_V^{kv(h)}: [head_dim, d_model]
            V_h = W_V[kv_start:kv_end, :]

            # Used to compute ||W_O^h W_V^h d|| without creating
            # a full [features, d_model] transformed matrix.
            O_gram = O_h.T @ O_h

            head_cache.append({
                "query_head": query_head,
                "kv_head": kv_head,
                "O_h": O_h,
                "V_h": V_h,
                "O_gram": O_gram,
            })

            head_metadata_rows.append({
                "matrix_column": (
                    len(head_metadata_rows)
                ),
                "layer": int(layer_idx),
                "query_head": int(query_head),
                "kv_head": int(kv_head),
                "head_dim": int(head_dim),
                "num_query_heads": int(num_query_heads),
                "num_kv_heads": int(num_kv_heads),
            })

        for start in range(
            0,
            n_features,
            int(feature_chunk_size),
        ):
            end = min(
                start + int(feature_chunk_size),
                n_features,
            )

            # Decoder directions as row vectors: [B, d_model].
            D = W_dec_cpu[start:end].to(
                device=compute_device,
                dtype=torch.float32,
            )

            d_norm = D.norm(
                dim=-1
            )

            for head_info in head_cache:
                query_head = head_info["query_head"]
                O_h = head_info["O_h"]
                V_h = head_info["V_h"]
                O_gram = head_info["O_gram"]

                # Row-vector equivalent of W_V^h d:
                # [B, d_model] @ [d_model, head_dim]
                value_image = D @ V_h.T

                # Row-vector equivalent of (W_O^h)^T d:
                # [B, d_model] @ [d_model, head_dim]
                output_readout = D @ O_h

                # d^T W_O^h W_V^h d.
                numerator = (
                    value_image
                    * output_readout
                ).sum(dim=-1)

                # ------------------------------------------------
                # Metric 1: exact residual-space cosine
                #
                # ||W_O^h W_V^h d||^2
                #   = v^T (W_O^h)^T W_O^h v
                # ------------------------------------------------

                transformed_norm_sq = (
                    (value_image @ O_gram)
                    * value_image
                ).sum(dim=-1).clamp_min(0.0)

                transformed_norm = torch.sqrt(
                    transformed_norm_sq
                )

                residual_denominator = (
                    d_norm
                    * transformed_norm
                )

                residual_cosine = torch.where(
                    residual_denominator > float(eps),
                    numerator / residual_denominator,
                    torch.full_like(
                        numerator,
                        torch.nan,
                    ),
                )

                # ------------------------------------------------
                # Metric 2: head-space cosine
                # ------------------------------------------------

                headspace_denominator = (
                    value_image.norm(dim=-1)
                    * output_readout.norm(dim=-1)
                )

                headspace_cosine = torch.where(
                    headspace_denominator > float(eps),
                    numerator / headspace_denominator,
                    torch.full_like(
                        numerator,
                        torch.nan,
                    ),
                )

                layer_residual_cosine[
                    start:end,
                    query_head,
                ] = (
                    residual_cosine
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

                layer_headspace_cosine[
                    start:end,
                    query_head,
                ] = (
                    headspace_cosine
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

            del D

            if (
                start == 0
                or end == n_features
                or (
                    start // int(feature_chunk_size)
                ) % 10 == 0
            ):
                print(
                    f"  layer {layer_idx}: "
                    f"features {start:,}–{end - 1:,}"
                )

        residual_cosine_blocks.append(
            layer_residual_cosine
        )

        headspace_cosine_blocks.append(
            layer_headspace_cosine
        )

        del W_V, W_O, head_cache

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    residual_cosine_matrix = np.concatenate(
        residual_cosine_blocks,
        axis=1,
    )

    headspace_cosine_matrix = np.concatenate(
        headspace_cosine_blocks,
        axis=1,
    )

    head_metadata_df = pd.DataFrame(
        head_metadata_rows
    )

    if (
        residual_cosine_matrix.shape[1]
        != len(head_metadata_df)
    ):
        raise AssertionError(
            "Head metadata does not align with cosine matrices."
        )

    # Best head independently under each metric.
    (
        best_residual_cosine,
        best_residual_column,
        residual_percentile,
    ) = _best_head_summary(
        residual_cosine_matrix
    )

    (
        best_headspace_cosine,
        best_headspace_column,
        headspace_percentile,
    ) = _best_head_summary(
        headspace_cosine_matrix
    )

    all_feature_rows = []

    for feature_idx in range(n_features):
        residual_col = int(
            best_residual_column[feature_idx]
        )

        headspace_col = int(
            best_headspace_column[feature_idx]
        )

        if residual_col >= 0:
            residual_meta = head_metadata_df.iloc[
                residual_col
            ]
        else:
            residual_meta = None

        if headspace_col >= 0:
            headspace_meta = head_metadata_df.iloc[
                headspace_col
            ]
        else:
            headspace_meta = None

        all_feature_rows.append({
            "feature_idx": int(feature_idx),

            "best_residual_ov_cosine": float(
                best_residual_cosine[feature_idx]
            ),
            "residual_antiwrite_percentile": float(
                residual_percentile[feature_idx]
            ),
            "best_residual_layer": (
                int(residual_meta["layer"])
                if residual_meta is not None
                else -1
            ),
            "best_residual_head": (
                int(residual_meta["query_head"])
                if residual_meta is not None
                else -1
            ),
            "best_residual_kv_head": (
                int(residual_meta["kv_head"])
                if residual_meta is not None
                else -1
            ),
            "best_residual_matrix_column": residual_col,

            "best_headspace_ov_cosine": float(
                best_headspace_cosine[feature_idx]
            ),
            "headspace_antiwrite_percentile": float(
                headspace_percentile[feature_idx]
            ),
            "best_headspace_layer": (
                int(headspace_meta["layer"])
                if headspace_meta is not None
                else -1
            ),
            "best_headspace_head": (
                int(headspace_meta["query_head"])
                if headspace_meta is not None
                else -1
            ),
            "best_headspace_kv_head": (
                int(headspace_meta["kv_head"])
                if headspace_meta is not None
                else -1
            ),
            "best_headspace_matrix_column": headspace_col,
        })

    all_feature_df = pd.DataFrame(
        all_feature_rows
    )

    return {
        "all_feature_df": all_feature_df,
        "residual_cosine_matrix": (
            residual_cosine_matrix
        ),
        "headspace_cosine_matrix": (
            headspace_cosine_matrix
        ),
        "head_metadata_df": head_metadata_df,
    }


# ------------------------------------------------------------
# Run the weight-only analysis
# ------------------------------------------------------------

ov_geometry_results = compute_weight_only_ov_geometry(
    model=model,
    sae=sae,
    candidate_layers=OV_CANDIDATE_LAYERS,
    feature_chunk_size=OV_FEATURE_CHUNK_SIZE,
    eps=OV_EPS,
)

ov_all_feature_df = (
    ov_geometry_results["all_feature_df"]
)

ov_residual_cosine_matrix = (
    ov_geometry_results["residual_cosine_matrix"]
)

ov_headspace_cosine_matrix = (
    ov_geometry_results["headspace_cosine_matrix"]
)

ov_head_metadata_df = (
    ov_geometry_results["head_metadata_df"]
)


# ------------------------------------------------------------
# Select the top LFH features from Final_rho_df
# ------------------------------------------------------------

_required_lfh_columns = {
    "feature_idx",
    "avg_querywise_pearson_r",
}

_missing_lfh_columns = (
    _required_lfh_columns
    - set(Final_rho_df.columns)
)

if _missing_lfh_columns:
    raise KeyError(
        "Final_rho_df is missing required columns: "
        f"{sorted(_missing_lfh_columns)}"
    )

top_lfh_df = (
    Final_rho_df[
        ["feature_idx", "avg_querywise_pearson_r"]
    ]
    .dropna()
    .drop_duplicates("feature_idx")
    .sort_values(
        "avg_querywise_pearson_r",
        ascending=True,
    )
    .head(int(OV_TOP_N_LFH_FEATURES))
    .reset_index(drop=True)
)

top_lfh_df.insert(
    0,
    "lfh_rank",
    np.arange(1, len(top_lfh_df) + 1),
)

top_lfh_df = top_lfh_df.rename(
    columns={
        "avg_querywise_pearson_r": "rho_f",
    }
)

if not (
    top_lfh_df["feature_idx"].between(
        0,
        int(sae.n_features) - 1,
    )
).all():
    raise IndexError(
        "At least one LFH feature index lies outside the SAE."
    )


# ------------------------------------------------------------
# Build the paper-facing table under the selected metric
# ------------------------------------------------------------

if OV_RANK_METRIC == "residual_cosine":
    selected_columns = {
        "best_residual_layer": "best_layer",
        "best_residual_head": "best_head_idx",
        "best_residual_kv_head": "best_kv_head_idx",
        "best_residual_matrix_column": "best_matrix_column",
        "best_residual_ov_cosine": "ov_cosine",
        "residual_antiwrite_percentile": "percentile",
    }
else:
    selected_columns = {
        "best_headspace_layer": "best_layer",
        "best_headspace_head": "best_head_idx",
        "best_headspace_kv_head": "best_kv_head_idx",
        "best_headspace_matrix_column": "best_matrix_column",
        "best_headspace_ov_cosine": "ov_cosine",
        "headspace_antiwrite_percentile": "percentile",
    }

selected_geometry_df = ov_all_feature_df[
    ["feature_idx"] + list(selected_columns)
].rename(
    columns=selected_columns
)

lfh_ov_alignment_df = top_lfh_df.merge(
    selected_geometry_df,
    on="feature_idx",
    how="left",
    validate="one_to_one",
)

# Report both cosine definitions at whichever head was selected.
_selected_residual_cosines = []
_selected_headspace_cosines = []

for row in lfh_ov_alignment_df.itertuples(
    index=False
):
    matrix_column = int(row.best_matrix_column)
    feature_idx = int(row.feature_idx)

    _selected_residual_cosines.append(
        float(
            ov_residual_cosine_matrix[
                feature_idx,
                matrix_column,
            ]
        )
    )

    _selected_headspace_cosines.append(
        float(
            ov_headspace_cosine_matrix[
                feature_idx,
                matrix_column,
            ]
        )
    )

lfh_ov_alignment_df[
    "residual_cosine_at_selected_head"
] = _selected_residual_cosines

lfh_ov_alignment_df[
    "headspace_cosine_at_selected_head"
] = _selected_headspace_cosines

lfh_ov_alignment_df["feature"] = [
    f"L{int(TARGET_LAYER)}-{int(feature_idx)}"
    for feature_idx
    in lfh_ov_alignment_df["feature_idx"]
]

_single_layer_scan = (
    len(set(OV_CANDIDATE_LAYERS)) == 1
)

if _single_layer_scan:
    lfh_ov_alignment_df["best_head"] = [
        f"H{int(head_idx)}"
        for head_idx
        in lfh_ov_alignment_df["best_head_idx"]
    ]
else:
    lfh_ov_alignment_df["best_head"] = [
        f"L{int(layer_idx)}-H{int(head_idx)}"
        for layer_idx, head_idx
        in zip(
            lfh_ov_alignment_df["best_layer"],
            lfh_ov_alignment_df["best_head_idx"],
        )
    ]

lfh_ov_alignment_df["ov_metric"] = OV_RANK_METRIC

paper_lfh_ov_table_df = lfh_ov_alignment_df[
    [
        "feature",
        "rho_f",
        "best_head",
        "ov_cosine",
        "percentile",
    ]
].copy()

print("=" * 100)
print("Top LFH feature OV anti-write alignment")
print("Ranking metric:", OV_RANK_METRIC)
print(
    "Metric equation:",
    (
        "cos(W_O^h W_V^h d_f, d_f)"
        if OV_RANK_METRIC == "residual_cosine"
        else "cos(W_V^h d_f, (W_O^h)^T d_f)"
    ),
)
print("=" * 100)

display(paper_lfh_ov_table_df)

print(
    "\nDetailed table containing both cosine definitions:"
)
display(
    lfh_ov_alignment_df[
        [
            "lfh_rank",
            "feature_idx",
            "rho_f",
            "best_layer",
            "best_head_idx",
            "best_kv_head_idx",
            "ov_metric",
            "ov_cosine",
            "percentile",
            "residual_cosine_at_selected_head",
            "headspace_cosine_at_selected_head",
        ]
    ]
)

_n_above_95 = int(
    (
        paper_lfh_ov_table_df["percentile"]
        >= float(OV_HIGHLIGHT_PERCENTILE)
    ).sum()
)

_median_percentile = float(
    paper_lfh_ov_table_df["percentile"].median()
)

print(
    f"\n{_n_above_95}/{len(paper_lfh_ov_table_df)} "
    f"top LFH features are at or above the "
    f"{OV_HIGHLIGHT_PERCENTILE:.0%} anti-write percentile."
)

print(
    f"Median anti-write percentile: "
    f"{_median_percentile:.3f}"
)


# ------------------------------------------------------------
# Direct full-vector verification of matrix orientation
# ------------------------------------------------------------

@torch.inference_mode()
def verify_one_ov_entry_directly(
    *,
    feature_idx: int,
    layer_idx: int,
    query_head_idx: int,
) -> Dict[str, float]:
    """
    Explicitly form W_O^h W_V^h d_f for one feature/head and verify
    the optimized all-feature computation.
    """
    feature_idx = int(feature_idx)
    layer_idx = int(layer_idx)
    query_head_idx = int(query_head_idx)

    attn = _get_attention_module(
        model,
        layer_idx,
    )

    layout = _infer_attention_layout(
        model,
        layer_idx,
    )

    device = attn.o_proj.weight.device
    head_dim = layout["head_dim"]

    kv_head_idx = _query_head_to_kv_head(
        query_head_idx,
        num_query_heads=layout["num_query_heads"],
        num_kv_heads=layout["num_kv_heads"],
    )

    q_start = query_head_idx * head_dim
    q_end = q_start + head_dim

    kv_start = kv_head_idx * head_dim
    kv_end = kv_start + head_dim

    d_f = (
        sae.W_dec[feature_idx]
        .detach()
        .to(
            device=device,
            dtype=torch.float32,
        )
    )

    W_V_h = (
        attn.v_proj.weight[
            kv_start:kv_end,
            :
        ]
        .detach()
        .to(
            device=device,
            dtype=torch.float32,
        )
    )

    W_O_h = (
        attn.o_proj.weight[
            :,
            q_start:q_end,
        ]
        .detach()
        .to(
            device=device,
            dtype=torch.float32,
        )
    )

    value_image = W_V_h @ d_f
    output_readout = W_O_h.T @ d_f
    transformed_direction = W_O_h @ value_image

    residual_cosine = (
        torch.nn.functional.cosine_similarity(
            transformed_direction,
            d_f,
            dim=0,
        )
    )

    headspace_cosine = (
        torch.nn.functional.cosine_similarity(
            value_image,
            output_readout,
            dim=0,
        )
    )

    return {
        "feature_idx": feature_idx,
        "layer": layer_idx,
        "query_head": query_head_idx,
        "kv_head": kv_head_idx,
        "residual_cosine": float(
            residual_cosine.cpu()
        ),
        "headspace_cosine": float(
            headspace_cosine.cpu()
        ),
        "feature_norm": float(
            d_f.norm().cpu()
        ),
        "value_image_norm": float(
            value_image.norm().cpu()
        ),
        "transformed_direction_norm": float(
            transformed_direction.norm().cpu()
        ),
    }


_first_row = lfh_ov_alignment_df.iloc[0]

ov_direct_check = verify_one_ov_entry_directly(
    feature_idx=int(_first_row["feature_idx"]),
    layer_idx=int(_first_row["best_layer"]),
    query_head_idx=int(_first_row["best_head_idx"]),
)

print("\nDirect orientation check on the first LFH feature:")
display(pd.DataFrame([ov_direct_check]))

if not np.isclose(
    ov_direct_check["residual_cosine"],
    float(
        _first_row[
            "residual_cosine_at_selected_head"
        ]
    ),
    atol=1e-5,
    rtol=1e-5,
):
    raise AssertionError(
        "Direct residual-space cosine does not match "
        "the optimized computation."
    )

if not np.isclose(
    ov_direct_check["headspace_cosine"],
    float(
        _first_row[
            "headspace_cosine_at_selected_head"
        ]
    ),
    atol=1e-5,
    rtol=1e-5,
):
    raise AssertionError(
        "Direct head-space cosine does not match "
        "the optimized computation."
    )

print("Direct OV orientation check passed.")


# ------------------------------------------------------------
# Generate LaTeX rows matching the paper table
# ------------------------------------------------------------

def make_lfh_ov_latex_rows(
    table_df: pd.DataFrame,
    *,
    bold_percentile_at: float = 0.95,
) -> str:
    rows = []

    for row in table_df.itertuples(index=False):
        percentile_text = f"{float(row.percentile):.3f}"

        if float(row.percentile) >= float(
            bold_percentile_at
        ):
            percentile_text = (
                "\\textbf{"
                + percentile_text
                + "}"
            )

        rows.append(
            f"{row.feature} & "
            f"{float(row.rho_f):.3f} & "
            f"{row.best_head} & "
            f"{float(row.ov_cosine):.3f} & "
            f"{percentile_text} \\\\"
        )

    return "\n".join(rows)


lfh_ov_latex_rows = make_lfh_ov_latex_rows(
    paper_lfh_ov_table_df,
    bold_percentile_at=OV_HIGHLIGHT_PERCENTILE,
)

print("\nLaTeX table rows:\n")
print(lfh_ov_latex_rows)


# Optional cleanup of temporary names.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Encoder-aware OV extension + all-feature histogram
#
# Run AFTER the previous weight-only OV-analysis cell.
#
# Existing required objects:
#   model
#   sae
#   TARGET_LAYER
#   Final_rho_df
#   ov_residual_cosine_matrix
#   ov_headspace_cosine_matrix
#   ov_head_metadata_df
#   _get_attention_module
#   _infer_attention_layout
#
# New principal outputs:
#   ov_decoder_to_encoder_residual_cosine_matrix
#   ov_decoder_to_encoder_headspace_cosine_matrix
#   ov_all_feature_df
#   lfh_ov_alignment_df
#   paper_lfh_ov_table_df
#   ov_histogram_feature_df
#   ov_histogram_bin_df
#   ov_selected_metric_histogram_fig
# ============================================================

import gc
from typing import Dict, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go

from IPython.display import display
from scipy.stats import rankdata


# ------------------------------------------------------------
# User-editable configuration
# ------------------------------------------------------------

# Recommended for the most direct LFH preactivation-suppression test:
OV_SELECTED_METRIC = "decoder_to_encoder_headspace"

# Available metrics:
#
# 1. "decoder_to_decoder_residual"
#       cos(W_O^h W_V^h d_f, d_f)
#
# 2. "decoder_to_decoder_headspace"
#       cos(W_V^h d_f, (W_O^h)^T d_f)
#
# 3. "decoder_to_encoder_residual"
#       cos(W_O^h W_V^h d_f, e_f)
#
# 4. "decoder_to_encoder_headspace"
#       cos(W_V^h d_f, (W_O^h)^T e_f)
#
# Here:
#   d_f = SAE decoder direction
#   e_f = SAE encoder/read direction

OV_ENCODER_METRIC_CHUNK_SIZE = int(
    globals().get("OV_FEATURE_CHUNK_SIZE", 1024)
)

OV_TOP_N_LFH_FEATURES = int(
    globals().get("OV_TOP_N_LFH_FEATURES", 10)
)

OV_HISTOGRAM_BINS = 50
OV_HIGHLIGHT_PERCENTILE = 0.95
OV_ENCODER_METRIC_EPS = 1e-12


VALID_OV_SELECTED_METRICS = {
    "decoder_to_decoder_residual",
    "decoder_to_decoder_headspace",
    "decoder_to_encoder_residual",
    "decoder_to_encoder_headspace",
}

if OV_SELECTED_METRIC not in VALID_OV_SELECTED_METRICS:
    raise ValueError(
        f"Unknown OV_SELECTED_METRIC={OV_SELECTED_METRIC!r}. "
        f"Expected one of {sorted(VALID_OV_SELECTED_METRICS)}."
    )

if OV_ENCODER_METRIC_CHUNK_SIZE <= 0:
    raise ValueError(
        "OV_ENCODER_METRIC_CHUNK_SIZE must be positive."
    )

if OV_HISTOGRAM_BINS <= 0:
    raise ValueError(
        "OV_HISTOGRAM_BINS must be positive."
    )


# ------------------------------------------------------------
# Verify that the original OV cell has been run
# ------------------------------------------------------------

_REQUIRED_OV_OBJECTS = [
    "model",
    "sae",
    "Final_rho_df",
    "ov_residual_cosine_matrix",
    "ov_headspace_cosine_matrix",
    "ov_head_metadata_df",
    "_get_attention_module",
    "_infer_attention_layout",
]

_missing_ov_objects = [
    name
    for name in _REQUIRED_OV_OBJECTS
    if name not in globals()
]

if _missing_ov_objects:
    raise RuntimeError(
        "Run the original weight-only OV-analysis cell before "
        "this encoder-aware extension. Missing objects: "
        f"{_missing_ov_objects}"
    )


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def _ov_negative_tail_percentile(
    cosine_values: np.ndarray,
) -> np.ndarray:
    """
    Convert cosine values into anti-write percentiles.

    More negative cosine -> larger anti-write strength -> higher percentile.

    Each feature contributes one value: its most negative candidate-head
    cosine under the corresponding metric.
    """
    cosine_values = np.asarray(
        cosine_values,
        dtype=np.float64,
    )

    result = np.full(
        cosine_values.shape,
        np.nan,
        dtype=np.float64,
    )

    valid = np.isfinite(cosine_values)

    if not valid.any():
        return result

    antiwrite_strength = -cosine_values[valid]

    result[valid] = (
        rankdata(
            antiwrite_strength,
            method="average",
        )
        / float(valid.sum())
    )

    return result


def _ov_best_head_summary(
    cosine_matrix: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    For every SAE feature, identify the most negative candidate head.

    Parameters
    ----------
    cosine_matrix:
        Shape [n_features, n_candidate_heads].

    Returns
    -------
    best_cosine:
        Most negative cosine for each feature.

    best_column:
        Column of the selected head in ov_head_metadata_df.

    percentile:
        Anti-write percentile of each feature's best-head cosine
        relative to all SAE features.
    """
    cosine_matrix = np.asarray(
        cosine_matrix,
        dtype=np.float64,
    )

    if cosine_matrix.ndim != 2:
        raise ValueError(
            "cosine_matrix must be two-dimensional; "
            f"got shape {cosine_matrix.shape}."
        )

    safe_matrix = np.where(
        np.isfinite(cosine_matrix),
        cosine_matrix,
        np.inf,
    )

    best_column = np.argmin(
        safe_matrix,
        axis=1,
    )

    feature_rows = np.arange(
        safe_matrix.shape[0]
    )

    best_cosine = safe_matrix[
        feature_rows,
        best_column,
    ]

    valid = np.isfinite(best_cosine)

    best_cosine = np.where(
        valid,
        best_cosine,
        np.nan,
    )

    best_column = np.where(
        valid,
        best_column,
        -1,
    )

    percentile = _ov_negative_tail_percentile(
        best_cosine
    )

    return (
        best_cosine,
        best_column,
        percentile,
    )


# ------------------------------------------------------------
# Compute decoder-to-encoder OV metrics
# ------------------------------------------------------------

@torch.inference_mode()
def compute_decoder_to_encoder_ov_metrics(
    *,
    model: torch.nn.Module,
    sae,
    head_metadata_df: pd.DataFrame,
    feature_chunk_size: int = 1024,
    eps: float = 1e-12,
) -> Dict[str, np.ndarray]:
    """
    Compute, for every feature f and candidate head h:

    Residual-space decoder-to-encoder cosine:
        cos(W_O^h W_V^{kv(h)} d_f, e_f)

    Head-space decoder-to-encoder cosine:
        cos(W_V^{kv(h)} d_f, (W_O^h)^T e_f)

    where:
        d_f = sae.W_dec[f]
        e_f = sae.W_enc[:, f]

    A negative value means that a residual component along d_f is
    mapped toward a direction that decreases feature f's preactivation.
    """
    required_metadata_columns = {
        "matrix_column",
        "layer",
        "query_head",
        "kv_head",
        "head_dim",
        "num_query_heads",
        "num_kv_heads",
    }

    missing_metadata_columns = (
        required_metadata_columns
        - set(head_metadata_df.columns)
    )

    if missing_metadata_columns:
        raise KeyError(
            "ov_head_metadata_df is missing columns: "
            f"{sorted(missing_metadata_columns)}"
        )

    # Decoder directions: [n_features, d_model]
    W_dec_cpu = (
        sae.W_dec
        .detach()
        .float()
        .cpu()
        .contiguous()
    )

    # Encoder/read directions: [n_features, d_model]
    W_enc_cpu = (
        sae.W_enc
        .detach()
        .float()
        .cpu()
        .T
        .contiguous()
    )

    if W_dec_cpu.ndim != 2:
        raise ValueError(
            f"Expected sae.W_dec to be 2D; got {W_dec_cpu.shape}."
        )

    if W_enc_cpu.ndim != 2:
        raise ValueError(
            f"Expected sae.W_enc to be 2D; got {W_enc_cpu.shape}."
        )

    if W_dec_cpu.shape != W_enc_cpu.shape:
        raise ValueError(
            "Encoder and decoder feature arrays must have matching "
            f"[n_features, d_model] shapes; got "
            f"W_dec={tuple(W_dec_cpu.shape)}, "
            f"W_enc.T={tuple(W_enc_cpu.shape)}."
        )

    n_features, d_model = map(
        int,
        W_dec_cpu.shape,
    )

    n_head_columns = int(
        len(head_metadata_df)
    )

    residual_cosine_matrix = np.full(
        (n_features, n_head_columns),
        np.nan,
        dtype=np.float32,
    )

    headspace_cosine_matrix = np.full(
        (n_features, n_head_columns),
        np.nan,
        dtype=np.float32,
    )

    print("=" * 100)
    print("Computing encoder-aware OV metrics")
    print("Features:", n_features)
    print("Residual width:", d_model)
    print("Candidate head columns:", n_head_columns)
    print("Chunk size:", feature_chunk_size)
    print("=" * 100)

    # Process one transformer layer at a time.
    for layer_idx, layer_metadata in head_metadata_df.groupby(
        "layer",
        sort=False,
    ):
        layer_idx = int(layer_idx)

        attn = _get_attention_module(
            model,
            layer_idx,
        )

        layout = _infer_attention_layout(
            model,
            layer_idx,
        )

        if int(layout["d_model"]) != d_model:
            raise ValueError(
                f"Layer {layer_idx} residual width "
                f"{layout['d_model']} does not match SAE width "
                f"{d_model}."
            )

        compute_device = (
            attn.o_proj.weight.device
        )

        # Hugging Face Linear layout:
        #   W_V: [num_kv_heads * head_dim, d_model]
        #   W_O: [d_model, num_query_heads * head_dim]
        W_V = (
            attn.v_proj.weight
            .detach()
            .to(
                device=compute_device,
                dtype=torch.float32,
            )
        )

        W_O = (
            attn.o_proj.weight
            .detach()
            .to(
                device=compute_device,
                dtype=torch.float32,
            )
        )

        head_cache = []

        for row in layer_metadata.itertuples(
            index=False
        ):
            query_head = int(row.query_head)
            kv_head = int(row.kv_head)
            head_dim = int(row.head_dim)
            matrix_column = int(row.matrix_column)

            query_start = (
                query_head * head_dim
            )
            query_end = (
                query_start + head_dim
            )

            kv_start = (
                kv_head * head_dim
            )
            kv_end = (
                kv_start + head_dim
            )

            # W_O^h: [d_model, head_dim]
            O_h = W_O[
                :,
                query_start:query_end,
            ]

            # W_V^{kv(h)}: [head_dim, d_model]
            V_h = W_V[
                kv_start:kv_end,
                :,
            ]

            # Used to calculate ||W_O^h W_V^h d_f|| without
            # explicitly materializing a [chunk, d_model] tensor.
            O_gram = O_h.T @ O_h

            head_cache.append({
                "matrix_column": matrix_column,
                "query_head": query_head,
                "kv_head": kv_head,
                "O_h": O_h,
                "V_h": V_h,
                "O_gram": O_gram,
            })

        for start in range(
            0,
            n_features,
            int(feature_chunk_size),
        ):
            end = min(
                start + int(feature_chunk_size),
                n_features,
            )

            # Source-side decoder directions.
            D = W_dec_cpu[
                start:end
            ].to(
                device=compute_device,
                dtype=torch.float32,
            )

            # Target-side encoder/read directions.
            E = W_enc_cpu[
                start:end
            ].to(
                device=compute_device,
                dtype=torch.float32,
            )

            encoder_norm = E.norm(
                dim=-1
            )

            for head_info in head_cache:
                O_h = head_info["O_h"]
                V_h = head_info["V_h"]
                O_gram = head_info["O_gram"]
                matrix_column = int(
                    head_info["matrix_column"]
                )

                # Value-space image of decoder direction:
                #   W_V^h d_f
                # Row-vector representation: D @ V_h.T
                value_image = D @ V_h.T

                # Output-side readout of encoder direction:
                #   (W_O^h)^T e_f
                # Row-vector representation: E @ O_h
                encoder_output_readout = E @ O_h

                # Shared numerator:
                #
                #   <W_V^h d_f, (W_O^h)^T e_f>
                # = e_f^T W_O^h W_V^h d_f
                numerator = (
                    value_image
                    * encoder_output_readout
                ).sum(dim=-1)

                # ------------------------------------------------
                # Decoder-to-encoder residual-space cosine
                #
                # cos(W_O^h W_V^h d_f, e_f)
                # ------------------------------------------------

                transformed_norm_sq = (
                    (value_image @ O_gram)
                    * value_image
                ).sum(
                    dim=-1
                ).clamp_min(0.0)

                transformed_norm = torch.sqrt(
                    transformed_norm_sq
                )

                residual_denominator = (
                    transformed_norm
                    * encoder_norm
                )

                residual_cosine = torch.where(
                    residual_denominator > float(eps),
                    numerator / residual_denominator,
                    torch.full_like(
                        numerator,
                        torch.nan,
                    ),
                )

                residual_cosine = torch.clamp(
                    residual_cosine,
                    min=-1.0,
                    max=1.0,
                )

                # ------------------------------------------------
                # Decoder-to-encoder head-space cosine
                #
                # cos(W_V^h d_f, (W_O^h)^T e_f)
                # ------------------------------------------------

                headspace_denominator = (
                    value_image.norm(dim=-1)
                    * encoder_output_readout.norm(
                        dim=-1
                    )
                )

                headspace_cosine = torch.where(
                    headspace_denominator > float(eps),
                    numerator / headspace_denominator,
                    torch.full_like(
                        numerator,
                        torch.nan,
                    ),
                )

                headspace_cosine = torch.clamp(
                    headspace_cosine,
                    min=-1.0,
                    max=1.0,
                )

                residual_cosine_matrix[
                    start:end,
                    matrix_column,
                ] = (
                    residual_cosine
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

                headspace_cosine_matrix[
                    start:end,
                    matrix_column,
                ] = (
                    headspace_cosine
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

            del D, E

            if (
                start == 0
                or end == n_features
                or (
                    start
                    // int(feature_chunk_size)
                ) % 10 == 0
            ):
                print(
                    f"  layer {layer_idx}: "
                    f"features {start:,}--{end - 1:,}"
                )

        del W_V, W_O, head_cache

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "decoder_to_encoder_residual": (
            residual_cosine_matrix
        ),
        "decoder_to_encoder_headspace": (
            headspace_cosine_matrix
        ),
    }


ov_encoder_metric_results = (
    compute_decoder_to_encoder_ov_metrics(
        model=model,
        sae=sae,
        head_metadata_df=ov_head_metadata_df,
        feature_chunk_size=(
            OV_ENCODER_METRIC_CHUNK_SIZE
        ),
        eps=OV_ENCODER_METRIC_EPS,
    )
)

ov_decoder_to_encoder_residual_cosine_matrix = (
    ov_encoder_metric_results[
        "decoder_to_encoder_residual"
    ]
)

ov_decoder_to_encoder_headspace_cosine_matrix = (
    ov_encoder_metric_results[
        "decoder_to_encoder_headspace"
    ]
)


# ------------------------------------------------------------
# Gather all four metric matrices
# ------------------------------------------------------------

ov_metric_matrices = {
    "decoder_to_decoder_residual": np.clip(
        np.asarray(
            ov_residual_cosine_matrix,
            dtype=np.float32,
        ),
        -1.0,
        1.0,
    ),
    "decoder_to_decoder_headspace": np.clip(
        np.asarray(
            ov_headspace_cosine_matrix,
            dtype=np.float32,
        ),
        -1.0,
        1.0,
    ),
    "decoder_to_encoder_residual": (
        ov_decoder_to_encoder_residual_cosine_matrix
    ),
    "decoder_to_encoder_headspace": (
        ov_decoder_to_encoder_headspace_cosine_matrix
    ),
}

_expected_shape = (
    int(sae.n_features),
    len(ov_head_metadata_df),
)

for metric_name, metric_matrix in (
    ov_metric_matrices.items()
):
    if metric_matrix.shape != _expected_shape:
        raise AssertionError(
            f"{metric_name} has shape "
            f"{metric_matrix.shape}; expected "
            f"{_expected_shape}."
        )


OV_METRIC_FORMULAS = {
    "decoder_to_decoder_residual": (
        "cos(W_O^h W_V^h d_f, d_f)"
    ),
    "decoder_to_decoder_headspace": (
        "cos(W_V^h d_f, (W_O^h)^T d_f)"
    ),
    "decoder_to_encoder_residual": (
        "cos(W_O^h W_V^h d_f, e_f)"
    ),
    "decoder_to_encoder_headspace": (
        "cos(W_V^h d_f, (W_O^h)^T e_f)"
    ),
}


# ------------------------------------------------------------
# Find each feature's best head under every metric
# ------------------------------------------------------------

ov_all_feature_df = pd.DataFrame({
    "feature_idx": np.arange(
        int(sae.n_features),
        dtype=int,
    )
})

_metadata_layer = (
    ov_head_metadata_df["layer"]
    .to_numpy(dtype=int)
)

_metadata_query_head = (
    ov_head_metadata_df["query_head"]
    .to_numpy(dtype=int)
)

_metadata_kv_head = (
    ov_head_metadata_df["kv_head"]
    .to_numpy(dtype=int)
)

for metric_name, metric_matrix in (
    ov_metric_matrices.items()
):
    (
        best_cosine,
        best_column,
        percentile,
    ) = _ov_best_head_summary(
        metric_matrix
    )

    best_layer = np.full(
        len(best_column),
        -1,
        dtype=int,
    )

    best_query_head = np.full(
        len(best_column),
        -1,
        dtype=int,
    )

    best_kv_head = np.full(
        len(best_column),
        -1,
        dtype=int,
    )

    valid = best_column >= 0

    best_layer[valid] = (
        _metadata_layer[
            best_column[valid]
        ]
    )

    best_query_head[valid] = (
        _metadata_query_head[
            best_column[valid]
        ]
    )

    best_kv_head[valid] = (
        _metadata_kv_head[
            best_column[valid]
        ]
    )

    ov_all_feature_df[
        f"best_{metric_name}_cosine"
    ] = best_cosine

    ov_all_feature_df[
        f"{metric_name}_antiwrite_percentile"
    ] = percentile

    ov_all_feature_df[
        f"best_{metric_name}_matrix_column"
    ] = best_column

    ov_all_feature_df[
        f"best_{metric_name}_layer"
    ] = best_layer

    ov_all_feature_df[
        f"best_{metric_name}_head"
    ] = best_query_head

    ov_all_feature_df[
        f"best_{metric_name}_kv_head"
    ] = best_kv_head


# ------------------------------------------------------------
# Construct the top-LFH paper table under selected metric
# ------------------------------------------------------------

required_lfh_columns = {
    "feature_idx",
    "avg_querywise_pearson_r",
}

missing_lfh_columns = (
    required_lfh_columns
    - set(Final_rho_df.columns)
)

if missing_lfh_columns:
    raise KeyError(
        "Final_rho_df is missing required columns: "
        f"{sorted(missing_lfh_columns)}"
    )

top_lfh_df = (
    Final_rho_df[
        [
            "feature_idx",
            "avg_querywise_pearson_r",
        ]
    ]
    .dropna()
    .drop_duplicates("feature_idx")
    .sort_values(
        "avg_querywise_pearson_r",
        ascending=True,
    )
    .head(OV_TOP_N_LFH_FEATURES)
    .reset_index(drop=True)
    .rename(
        columns={
            "avg_querywise_pearson_r": (
                "rho_f"
            )
        }
    )
)

top_lfh_df.insert(
    0,
    "lfh_rank",
    np.arange(
        1,
        len(top_lfh_df) + 1,
    ),
)

selected_prefix = OV_SELECTED_METRIC

selected_geometry_columns = {
    f"best_{selected_prefix}_cosine": (
        "ov_cosine"
    ),
    f"{selected_prefix}_antiwrite_percentile": (
        "percentile"
    ),
    f"best_{selected_prefix}_matrix_column": (
        "best_matrix_column"
    ),
    f"best_{selected_prefix}_layer": (
        "best_layer"
    ),
    f"best_{selected_prefix}_head": (
        "best_head_idx"
    ),
    f"best_{selected_prefix}_kv_head": (
        "best_kv_head_idx"
    ),
}

selected_geometry_df = (
    ov_all_feature_df[
        ["feature_idx"]
        + list(
            selected_geometry_columns.keys()
        )
    ]
    .rename(
        columns=selected_geometry_columns
    )
)

lfh_ov_alignment_df = top_lfh_df.merge(
    selected_geometry_df,
    on="feature_idx",
    how="left",
    validate="one_to_one",
)

if (
    lfh_ov_alignment_df[
        "best_matrix_column"
    ] < 0
).any():
    raise RuntimeError(
        "At least one top LFH feature had no valid "
        "candidate-head OV cosine."
    )


# ------------------------------------------------------------
# Report all four metrics at the same selected head
# ------------------------------------------------------------

selected_feature_indices = (
    lfh_ov_alignment_df[
        "feature_idx"
    ].to_numpy(dtype=int)
)

selected_head_columns = (
    lfh_ov_alignment_df[
        "best_matrix_column"
    ].to_numpy(dtype=int)
)

for metric_name, metric_matrix in (
    ov_metric_matrices.items()
):
    lfh_ov_alignment_df[
        f"{metric_name}_cosine_at_selected_head"
    ] = metric_matrix[
        selected_feature_indices,
        selected_head_columns,
    ]


# Backward-compatible aliases from the previous block.
lfh_ov_alignment_df[
    "residual_cosine_at_selected_head"
] = lfh_ov_alignment_df[
    "decoder_to_decoder_residual_cosine_at_selected_head"
]

lfh_ov_alignment_df[
    "headspace_cosine_at_selected_head"
] = lfh_ov_alignment_df[
    "decoder_to_decoder_headspace_cosine_at_selected_head"
]


lfh_ov_alignment_df["feature"] = [
    f"L{int(TARGET_LAYER)}-{int(feature_idx)}"
    for feature_idx
    in lfh_ov_alignment_df["feature_idx"]
]

_multiple_layers = (
    lfh_ov_alignment_df[
        "best_layer"
    ].nunique()
    > 1
)

if _multiple_layers:
    lfh_ov_alignment_df["best_head"] = [
        f"L{int(layer)}-H{int(head)}"
        for layer, head in zip(
            lfh_ov_alignment_df[
                "best_layer"
            ],
            lfh_ov_alignment_df[
                "best_head_idx"
            ],
        )
    ]
else:
    lfh_ov_alignment_df["best_head"] = [
        f"H{int(head)}"
        for head
        in lfh_ov_alignment_df[
            "best_head_idx"
        ]
    ]

lfh_ov_alignment_df[
    "ov_metric"
] = OV_SELECTED_METRIC

lfh_ov_alignment_df[
    "ov_metric_formula"
] = OV_METRIC_FORMULAS[
    OV_SELECTED_METRIC
]


paper_lfh_ov_table_df = (
    lfh_ov_alignment_df[
        [
            "feature",
            "rho_f",
            "best_head",
            "ov_cosine",
            "percentile",
        ]
    ]
    .copy()
)


print("=" * 110)
print("Top LFH feature OV alignment")
print("Selected metric:", OV_SELECTED_METRIC)
print(
    "Formula:",
    OV_METRIC_FORMULAS[
        OV_SELECTED_METRIC
    ],
)
print("=" * 110)

display(paper_lfh_ov_table_df)

print(
    "\nDetailed table with decoder- and encoder-aware metrics "
    "evaluated at the same selected head:"
)

display(
    lfh_ov_alignment_df[
        [
            "lfh_rank",
            "feature_idx",
            "rho_f",
            "best_layer",
            "best_head_idx",
            "best_kv_head_idx",
            "ov_metric",
            "ov_cosine",
            "percentile",
            "decoder_to_decoder_residual_cosine_at_selected_head",
            "decoder_to_decoder_headspace_cosine_at_selected_head",
            "decoder_to_encoder_residual_cosine_at_selected_head",
            "decoder_to_encoder_headspace_cosine_at_selected_head",
        ]
    ]
)


n_above_threshold = int(
    (
        paper_lfh_ov_table_df[
            "percentile"
        ]
        >= float(
            OV_HIGHLIGHT_PERCENTILE
        )
    ).sum()
)

median_percentile = float(
    paper_lfh_ov_table_df[
        "percentile"
    ].median()
)

print(
    f"\n{n_above_threshold}/"
    f"{len(paper_lfh_ov_table_df)} "
    f"top LFH features are at or above the "
    f"{OV_HIGHLIGHT_PERCENTILE:.0%} "
    "anti-write percentile."
)

print(
    "Median anti-write percentile:",
    f"{median_percentile:.3f}",
)


# ------------------------------------------------------------
# Histogram across all SAE features
#
# Each feature contributes one value:
# its most negative candidate-head cosine under OV_SELECTED_METRIC.
# ------------------------------------------------------------

selected_best_cosine_column = (
    f"best_{OV_SELECTED_METRIC}_cosine"
)

ov_histogram_feature_df = (
    ov_all_feature_df[
        [
            "feature_idx",
            selected_best_cosine_column,
        ]
    ]
    .rename(
        columns={
            selected_best_cosine_column: (
                "selected_metric_best_head_cosine"
            )
        }
    )
    .dropna()
    .reset_index(drop=True)
)

# Numerical clipping only; mathematically every cosine lies in [-1, 1].
ov_histogram_feature_df[
    "selected_metric_best_head_cosine"
] = np.clip(
    ov_histogram_feature_df[
        "selected_metric_best_head_cosine"
    ].to_numpy(dtype=float),
    -1.0,
    1.0,
)

histogram_values = (
    ov_histogram_feature_df[
        "selected_metric_best_head_cosine"
    ].to_numpy(dtype=float)
)

histogram_edges = np.linspace(
    -1.0,
    1.0,
    OV_HISTOGRAM_BINS + 1,
)

histogram_counts, _ = np.histogram(
    histogram_values,
    bins=histogram_edges,
)

ov_histogram_bin_df = pd.DataFrame({
    "bin_left": histogram_edges[:-1],
    "bin_right": histogram_edges[1:],
    "bin_center": (
        histogram_edges[:-1]
        + histogram_edges[1:]
    ) / 2.0,
    "feature_count": histogram_counts,
})

selected_metric_median = float(
    np.median(histogram_values)
)

selected_metric_mean = float(
    np.mean(histogram_values)
)


ov_selected_metric_histogram_fig = go.Figure()

ov_selected_metric_histogram_fig.add_trace(
    go.Histogram(
        x=histogram_values,
        xbins=dict(
            start=-1.0,
            end=1.0,
            size=(
                2.0
                / float(
                    OV_HISTOGRAM_BINS
                )
            ),
        ),
        histnorm=None,
        name="SAE features",
        hovertemplate=(
            "Cosine bin=%{x}<br>"
            "Number of features=%{y}"
            "<extra></extra>"
        ),
    )
)

ov_selected_metric_histogram_fig.add_vline(
    x=selected_metric_median,
    line_dash="dash",
    annotation_text=(
        f"Median = "
        f"{selected_metric_median:.3f}"
    ),
    annotation_position="top",
)

ov_selected_metric_histogram_fig.update_layout(
    title=dict(
        text=(
            "Distribution of Best-Head OV Alignment "
            "Across All SAE Features"
            "<br><sup>"
            f"Metric: "
            f"{OV_METRIC_FORMULAS[OV_SELECTED_METRIC]} | "
            f"one most-negative candidate head per feature | "
            f"n={len(histogram_values):,}"
            "</sup>"
        ),
        x=0.5,
    ),
    xaxis=dict(
        title="Best-head OV cosine",
        range=[-1.0, 1.0],
        tick0=-1.0,
        dtick=0.2,
    ),
    yaxis=dict(
        title="Number of SAE features",
    ),
    width=1000,
    height=650,
    template="plotly_white",
    font=dict(
        family="Times New Roman",
        size=17,
    ),
    bargap=0.02,
    showlegend=False,
)

ov_selected_metric_histogram_fig.show()

print(
    "\nHistogram summary"
)
print(
    "  metric:",
    OV_SELECTED_METRIC,
)
print(
    "  formula:",
    OV_METRIC_FORMULAS[
        OV_SELECTED_METRIC
    ],
)
print(
    "  number of valid SAE features:",
    f"{len(histogram_values):,}",
)
print(
    "  mean best-head cosine:",
    f"{selected_metric_mean:.4f}",
)
print(
    "  median best-head cosine:",
    f"{selected_metric_median:.4f}",
)


# ------------------------------------------------------------
# Generate LaTeX rows for the top-LFH table
# ------------------------------------------------------------

def make_encoder_aware_lfh_ov_latex_rows(
    table_df: pd.DataFrame,
    *,
    bold_percentile_at: float = 0.95,
) -> str:
    rows = []

    for row in table_df.itertuples(
        index=False
    ):
        percentile_text = (
            f"{float(row.percentile):.3f}"
        )

        if (
            float(row.percentile)
            >= float(bold_percentile_at)
        ):
            percentile_text = (
                "\\textbf{"
                + percentile_text
                + "}"
            )

        rows.append(
            f"{row.feature} & "
            f"{float(row.rho_f):.3f} & "
            f"{row.best_head} & "
            f"{float(row.ov_cosine):.3f} & "
            f"{percentile_text} \\\\"
        )

    return "\n".join(rows)


lfh_ov_latex_rows = (
    make_encoder_aware_lfh_ov_latex_rows(
        paper_lfh_ov_table_df,
        bold_percentile_at=(
            OV_HIGHLIGHT_PERCENTILE
        ),
    )
)

print("\nLaTeX table rows:\n")
print(lfh_ov_latex_rows)


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 17. Feature-specific MLP-layer causal steering

This final cell runs the same LFH causal dose-response measurement as the attention-head steering section, but the edited component is the entire MLP contribution from a selected transformer layer. The default uses the selected top LFH feature, edits target-block tokens only, and uses the Section 7 cached LFH tensors for an exact `alpha = 0` baseline whenever the configuration matches the LFH ranking run.

In [ ]:
# ============================================================
# 17. Feature-specific causal MLP-layer steering
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# This cell performs the same LFH causal dose-response measurement as the
# attention-head intervention, but the edited component is the whole MLP
# contribution at one transformer layer. The hook scales the output of
# model.layers[MLP_LAYER].mlp on the selected tokens before it is added to the
# residual stream.
#
# alpha semantics:
#   alpha =  0.0: no intervention / baseline
#   alpha = -1.0: ablate this layer's MLP contribution on edited tokens
#   alpha =  1.0: add one extra copy of this layer's MLP contribution
#   alpha =  a:   effective multiplier is (1 + a)

_required_for_lfh_mlp_causal = [
    "Final_rho_df",
    "prompts_all_agnews",
    "tokenize_prompt_for_offsets",
    "get_example_block_token_spans",
    "token_selection_to_torch_indices",
    "sae_single_feature_pre_activation",
    "prev_logits_by_query",
    "tgt_logits_by_query",
    "selected_feature_ids",
]
_missing_for_lfh_mlp_causal = [name for name in _required_for_lfh_mlp_causal if name not in globals()]
if len(_missing_for_lfh_mlp_causal) > 0:
    raise RuntimeError(
        "Run the LFH ranking cells and the causal-helper cells before this MLP steering cell. "
        f"Missing: {_missing_for_lfh_mlp_causal}"
    )

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

LFH_MLP_TOP_N_FEATURES_FOR_CAUSAL = 20
lfh_mlp_top20_features_df = Final_rho_df.head(LFH_MLP_TOP_N_FEATURES_FOR_CAUSAL).copy()

print("Top-20 LFH features by strongest negative source->target correlation")
display(lfh_mlp_top20_features_df)

# Edit this to any feature_idx in the table above.
# Default: reuse LFH_CAUSAL_FEATURE_IDX from the attention-head workflow if it
# exists; otherwise use the strongest negative-correlation LFH feature.
LFH_MLP_CAUSAL_FEATURE_IDX = int(
    LFH_CAUSAL_FEATURE_IDX if "LFH_CAUSAL_FEATURE_IDX" in globals()
    else lfh_mlp_top20_features_df.iloc[0]["feature_idx"]
)

# Edit this layer. It must be <= LFH_MLP_CAUSAL_FEATURE_SCORE_LAYER.
# Default: TARGET_LAYER, so the intervention edits the MLP in the same block
# whose residual output is scored by the SAE.
LFH_MLP_CAUSAL_SELECTED_LAYER = int(TARGET_LAYER)
LFH_MLP_CAUSAL_FEATURE_SCORE_LAYER = int(TARGET_LAYER)

# Same source/target pair and token scope as the attention-head causal test.
LFH_MLP_CAUSAL_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
LFH_MLP_CAUSAL_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)
LFH_MLP_CAUSAL_TOKEN_SCOPE = "target_block"

# Use the full Section 7 prompt set by default so alpha=0 exactly matches
# Final_rho_df['avg_querywise_pearson_r'] for the selected feature.
LFH_MLP_CAUSAL_MAX_QUERIES = None
LFH_MLP_CAUSAL_MAX_VARIANTS = None
LFH_MLP_CAUSAL_QUANTILE = 0.25

LFH_MLP_CAUSAL_ALPHAS = [-5, -2.0, -1.0, 0.0, 1.0, 2.0, 5]

if int(LFH_MLP_CAUSAL_SELECTED_LAYER) > int(LFH_MLP_CAUSAL_FEATURE_SCORE_LAYER):
    raise ValueError(
        f"LFH_MLP_CAUSAL_SELECTED_LAYER={LFH_MLP_CAUSAL_SELECTED_LAYER} is later than "
        f"LFH_MLP_CAUSAL_FEATURE_SCORE_LAYER={LFH_MLP_CAUSAL_FEATURE_SCORE_LAYER}. "
        "Choose an MLP layer <= TARGET_LAYER."
    )

print("=" * 120)
print("Feature-specific LFH MLP-layer causal steering config")
print("=" * 120)
print("feature_idx:", LFH_MLP_CAUSAL_FEATURE_IDX)
print("MLP layer:", LFH_MLP_CAUSAL_SELECTED_LAYER)
print("feature score layer:", LFH_MLP_CAUSAL_FEATURE_SCORE_LAYER)
print("token scope:", LFH_MLP_CAUSAL_TOKEN_SCOPE)
print("MLP alphas:", LFH_MLP_CAUSAL_ALPHAS)
print("max queries:", LFH_MLP_CAUSAL_MAX_QUERIES)
print("max variants:", LFH_MLP_CAUSAL_MAX_VARIANTS)
print("=" * 120)


# ------------------------------------------------------------
# Shared exact-Section-7 correlation helpers
# ------------------------------------------------------------

EPS = 1e-9


def _mlp_original_lfh_pearson_1d(x, y):
    """
    Pearson r using the same algebraic form as Section 7, specialized to one
    feature. This avoids scipy.stats.pearsonr so alpha=0 stays numerically
    aligned with the cached LFH tensors.
    """
    if "_original_lfh_pearson_1d" in globals():
        return _original_lfh_pearson_1d(x, y)

    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if x.size < 3:
        return np.nan

    x = x - x.mean()
    y = y - y.mean()
    num = float((x * y).sum())
    den = float(np.sqrt((x * x).sum() * (y * y).sum()))
    if den <= 0.0:
        return np.nan
    return num / den


def _mlp_querywise_corr_summary(records_df, target_col: str, prev_col: str = "baseline_previous_feature_mean"):
    if "_querywise_corr_summary" in globals():
        return _querywise_corr_summary(records_df, target_col=target_col, prev_col=prev_col)

    pooled = _mlp_original_lfh_pearson_1d(records_df[prev_col].values, records_df[target_col].values)
    q_corrs = []
    for _, g in records_df.groupby("query_idx", sort=True):
        r = _mlp_original_lfh_pearson_1d(g[prev_col].values, g[target_col].values)
        if np.isfinite(r):
            q_corrs.append(r)
    if len(q_corrs) == 0:
        return pooled, np.nan, np.nan, 0
    return pooled, float(np.mean(q_corrs)), float(np.std(q_corrs)), int(len(q_corrs))


def _mlp_high_low_delta_by_query(records_df, value_col: str, prev_col: str = "baseline_previous_feature_mean", quantile: float = 0.25):
    if "_high_low_delta_by_query" in globals():
        return _high_low_delta_by_query(records_df, value_col=value_col, prev_col=prev_col, quantile=quantile)

    deltas = []
    for _, g in records_df.groupby("query_idx"):
        x = g[prev_col].to_numpy(dtype=np.float64)
        y = g[value_col].to_numpy(dtype=np.float64)
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]
        if len(x) < 4:
            continue
        lo = np.quantile(x, quantile)
        hi = np.quantile(x, 1.0 - quantile)
        low_mask = x <= lo
        high_mask = x >= hi
        if low_mask.sum() == 0 or high_mask.sum() == 0:
            continue
        deltas.append(float(y[high_mask].mean() - y[low_mask].mean()))
    if len(deltas) == 0:
        return np.nan, np.nan, 0
    return float(np.mean(deltas)), float(np.std(deltas)), int(len(deltas))


def _mlp_normalize_lfh_block_index(block_index: int, total_blocks: int = NUM_BLOCKS_TOTAL) -> int:
    if "_normalize_lfh_block_index" in globals():
        return _normalize_lfh_block_index(block_index, total_blocks=total_blocks)

    block_index = int(block_index)
    if block_index < 0:
        block_index = int(total_blocks) + block_index
    if block_index < 0 or block_index >= int(total_blocks):
        raise ValueError(f"block_index={block_index} outside [0, {total_blocks}).")
    return block_index


def _mlp_get_feature_local_idx_in_selected_features(feature_idx: int):
    if "_get_feature_local_idx_in_selected_features" in globals():
        return _get_feature_local_idx_in_selected_features(feature_idx)

    selected = [int(f) for f in selected_feature_ids]
    try:
        return selected.index(int(feature_idx))
    except ValueError:
        return None


def _mlp_can_use_section7_cached_lfh_tensors(
    *,
    feature_idx: int,
    previous_block_index: int,
    target_block_index: int,
    feature_score_layer: int,
):
    if "_can_use_section7_cached_lfh_tensors" in globals():
        return _can_use_section7_cached_lfh_tensors(
            feature_idx=feature_idx,
            previous_block_index=previous_block_index,
            target_block_index=target_block_index,
            feature_score_layer=feature_score_layer,
        )

    if int(feature_score_layer) != int(TARGET_LAYER):
        return False, f"feature_score_layer={feature_score_layer} != TARGET_LAYER={TARGET_LAYER}", None

    try:
        prev_idx = _mlp_normalize_lfh_block_index(previous_block_index)
        tgt_idx = _mlp_normalize_lfh_block_index(target_block_index)
        src_idx = _mlp_normalize_lfh_block_index(LFH_SOURCE_BLOCK_INDEX)
        main_tgt_idx = _mlp_normalize_lfh_block_index(LFH_TARGET_BLOCK_INDEX)
    except Exception as e:
        return False, str(e), None

    if prev_idx != src_idx or tgt_idx != main_tgt_idx:
        return (
            False,
            f"requested blocks source={prev_idx}, target={tgt_idx} do not match "
            f"Section 7 LFH blocks source={src_idx}, target={main_tgt_idx}",
            None,
        )

    feature_local_idx = _mlp_get_feature_local_idx_in_selected_features(feature_idx)
    if feature_local_idx is None:
        return False, f"feature_idx={feature_idx} is not in selected_feature_ids", None

    return True, "using Section 7 cached LFH tensors", int(feature_local_idx)


def _mlp_get_section7_cached_pair_for_prompt(feature_local_idx: int, query_idx: int, variant_idx: int):
    if "_get_section7_cached_pair_for_prompt" in globals():
        return _get_section7_cached_pair_for_prompt(feature_local_idx, query_idx, variant_idx)

    if int(query_idx) >= int(prev_logits_by_query.shape[0]) or int(variant_idx) >= int(prev_logits_by_query.shape[1]):
        raise IndexError(
            f"cached LFH tensor index out of range: query_idx={query_idx}, variant_idx={variant_idx}, "
            f"shape={tuple(prev_logits_by_query.shape)}"
        )
    base_prev = float(prev_logits_by_query[int(query_idx), int(variant_idx), int(feature_local_idx)].detach().cpu())
    base_tgt = float(tgt_logits_by_query[int(query_idx), int(variant_idx), int(feature_local_idx)].detach().cpu())
    return base_prev, base_tgt


def _mlp_section7_cached_lfh_reference_for_subset(
    *,
    feature_idx: int,
    query_indices,
    max_variants,
    previous_block_index: int,
    target_block_index: int,
    feature_score_layer: int,
):
    if "_section7_cached_lfh_reference_for_subset" in globals():
        return _section7_cached_lfh_reference_for_subset(
            feature_idx=feature_idx,
            query_indices=query_indices,
            max_variants=max_variants,
            previous_block_index=previous_block_index,
            target_block_index=target_block_index,
            feature_score_layer=feature_score_layer,
        )

    ok, reason, feature_local_idx = _mlp_can_use_section7_cached_lfh_tensors(
        feature_idx=feature_idx,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        feature_score_layer=feature_score_layer,
    )
    if not ok:
        return {
            "available": False,
            "reason": reason,
            "feature_local_idx": None,
            "pooled_corr": np.nan,
            "mean_query_corr": np.nan,
            "std_query_corr": np.nan,
            "n_queries": 0,
            "final_rho_df_full_corr": np.nan,
        }

    prev_arr = prev_logits_by_query.detach().cpu().numpy()
    tgt_arr = tgt_logits_by_query.detach().cpu().numpy()

    q_corrs = []
    pooled_prev = []
    pooled_tgt = []

    for q in query_indices:
        q = int(q)
        n_variants_total = int(prev_arr.shape[1])
        n_variants = n_variants_total if max_variants is None else min(int(max_variants), n_variants_total)
        x = prev_arr[q, :n_variants, int(feature_local_idx)]
        y = tgt_arr[q, :n_variants, int(feature_local_idx)]
        pooled_prev.append(x)
        pooled_tgt.append(y)
        r = _mlp_original_lfh_pearson_1d(x, y)
        if np.isfinite(r):
            q_corrs.append(r)

    if len(pooled_prev) > 0:
        pooled_corr = _mlp_original_lfh_pearson_1d(np.concatenate(pooled_prev), np.concatenate(pooled_tgt))
    else:
        pooled_corr = np.nan

    if len(q_corrs) == 0:
        mean_query_corr = np.nan
        std_query_corr = np.nan
        n_queries = 0
    else:
        mean_query_corr = float(np.mean(q_corrs))
        std_query_corr = float(np.std(q_corrs))
        n_queries = int(len(q_corrs))

    final_rho_full = np.nan
    if "Final_rho_results" in globals() and int(feature_idx) in Final_rho_results:
        final_rho_full = float(Final_rho_results[int(feature_idx)])
    elif "Final_rho_df" in globals() and isinstance(Final_rho_df, pd.DataFrame) and "feature_idx" in Final_rho_df.columns:
        match = Final_rho_df[Final_rho_df["feature_idx"].astype(int) == int(feature_idx)]
        if len(match) > 0 and "avg_querywise_pearson_r" in match.columns:
            final_rho_full = float(match["avg_querywise_pearson_r"].iloc[0])

    return {
        "available": True,
        "reason": reason,
        "feature_local_idx": int(feature_local_idx),
        "pooled_corr": float(pooled_corr),
        "mean_query_corr": mean_query_corr,
        "std_query_corr": std_query_corr,
        "n_queries": n_queries,
        "final_rho_df_full_corr": final_rho_full,
    }


# ------------------------------------------------------------
# MLP intervention helpers
# ------------------------------------------------------------


def build_mlp_intervention_token_mask(
    seq_len: int,
    block_spans,
    previous_block_index: int,
    target_block_index: int,
    token_scope: str = "target_block",
):
    """
    Build a [1, seq_len] bool mask specifying which token positions receive
    the MLP ablation/amplification.
    """
    mask = torch.zeros((1, seq_len), dtype=torch.bool)

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    def _mark(selection):
        idx = token_selection_to_torch_indices(selection, device=mask.device)
        mask[0, idx] = True

    if token_scope == "target_block":
        _mark(tgt_span)
    elif token_scope == "previous_block":
        _mark(prev_span)
    elif token_scope == "previous_and_target":
        _mark(prev_span)
        _mark(tgt_span)
    elif token_scope == "all_example_tokens":
        for selection in block_spans:
            _mark(selection)
    else:
        raise ValueError(f"Unknown token_scope: {token_scope}")

    return mask


@torch.no_grad()
def run_forward_capture_feature_layer_with_mlp_alpha(
    model,
    input_ids,
    attention_mask,
    *,
    mlp_layer_idx: int,
    feature_layer_idx: int,
    token_mask: torch.Tensor,
    mlp_alpha: float = 0.0,
):
    """
    Run one forward pass while optionally scaling one layer's MLP output on
    selected tokens, then capture the feature_layer_idx residual output.
    """
    mlp_layer_idx = int(mlp_layer_idx)
    feature_layer_idx = int(feature_layer_idx)

    if mlp_layer_idx > feature_layer_idx:
        raise ValueError(
            f"mlp_layer_idx={mlp_layer_idx} > feature_layer_idx={feature_layer_idx}. "
            "This cannot affect the measured feature activation in the same forward pass."
        )

    cache = {}
    feature_block = model.get_submodule(f"model.layers.{feature_layer_idx}")
    mlp_module = model.get_submodule(f"model.layers.{mlp_layer_idx}.mlp")

    def feature_block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["feature_layer_output"] = x.detach().clone()
        return output

    def mlp_forward_hook(module, inputs, output):
        # alpha 0.0 is exactly no intervention.
        if float(mlp_alpha) == 0.0:
            return output

        y = output[0] if isinstance(output, tuple) else output
        edit_mask = token_mask.to(device=y.device, dtype=torch.bool)
        if edit_mask.ndim != 2 or edit_mask.shape[0] != y.shape[0] or edit_mask.shape[1] != y.shape[1]:
            raise RuntimeError(
                f"token_mask shape {tuple(edit_mask.shape)} incompatible with MLP output shape {tuple(y.shape)}."
            )

        effective_multiplier = 1.0 + float(mlp_alpha)
        y_scaled = y * effective_multiplier
        y_edit = torch.where(edit_mask[..., None], y_scaled, y)

        if isinstance(output, tuple):
            return (y_edit,) + tuple(output[1:])
        return y_edit

    h_feature = feature_block.register_forward_hook(feature_block_hook)
    h_mlp = mlp_module.register_forward_hook(mlp_forward_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h_mlp.remove()
        h_feature.remove()

    if "feature_layer_output" not in cache:
        raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")

    return cache["feature_layer_output"]


@torch.no_grad()
def measure_prompt_feature_with_mlp_alpha(
    prompt: str,
    *,
    feature_idx: int,
    mlp_layer_idx: int,
    feature_layer_idx: int,
    mlp_alpha: float,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt, run a selected MLP-layer intervention and measure feature
    pre-activation on source and target exemplar blocks.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    token_mask = build_mlp_intervention_token_mask(
        seq_len=input_ids.shape[1],
        block_spans=block_spans,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        token_scope=token_scope,
    ).to(MODEL_INPUT_DEVICE)

    layer_output = run_forward_capture_feature_layer_with_mlp_alpha(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        mlp_layer_idx=mlp_layer_idx,
        feature_layer_idx=feature_layer_idx,
        token_mask=token_mask,
        mlp_alpha=mlp_alpha,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_selection = block_spans[previous_block_index]
    target_selection = block_spans[target_block_index]
    prev_idx = token_selection_to_torch_indices(prev_selection, device=pre_f.device)
    target_idx = token_selection_to_torch_indices(target_selection, device=pre_f.device)

    return {
        "previous_feature_mean": float(pre_f.index_select(0, prev_idx).mean().detach().cpu()),
        "target_feature_mean": float(pre_f.index_select(0, target_idx).mean().detach().cpu()),
        "prev_span": block_spans[previous_block_index],
        "target_span": block_spans[target_block_index],
    }


def run_mlp_layer_alpha_causal_test(
    *,
    feature_idx: int,
    mlp_layer_idx: int,
    prompts_all_agnews,
    feature_score_layer: int = TARGET_LAYER,
    mlp_alphas=(-5.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0),
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    max_queries=None,
    max_variants=None,
    quantile: float = 0.25,
    verbose_every: int = 10,
    use_section7_cached_baseline: bool = True,
):
    """
    Causal test for one MLP layer.

    For each prompt variant:
      1. Use the exact Section 7 cached source/target feature means for the
         alpha=0 baseline when the feature/source/target/layer match the LFH run.
      2. Scale the selected layer's MLP output on the selected tokens for each
         nonzero alpha.
      3. Recompute corr(source_feature_baseline, target_feature_intervened)
         using the same mean query-wise Pearson statistic as Section 7.
    """
    feature_idx = int(feature_idx)
    mlp_layer_idx = int(mlp_layer_idx)
    feature_score_layer = int(feature_score_layer)

    if mlp_layer_idx > feature_score_layer:
        raise ValueError(
            f"mlp_layer_idx={mlp_layer_idx} > feature_score_layer={feature_score_layer}; "
            "the intervention cannot affect the measured feature in the same forward pass."
        )

    requested_alphas = []
    for alpha in mlp_alphas:
        af = float(alpha)
        if af not in requested_alphas:
            requested_alphas.append(af)
    if 0.0 not in requested_alphas:
        alphas_to_run = [0.0] + requested_alphas
    else:
        alphas_to_run = requested_alphas

    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    cached_ref = _mlp_section7_cached_lfh_reference_for_subset(
        feature_idx=feature_idx,
        query_indices=query_indices,
        max_variants=max_variants,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        feature_score_layer=feature_score_layer,
    )
    use_cached_baseline = bool(use_section7_cached_baseline and cached_ref["available"])
    feature_local_idx = cached_ref["feature_local_idx"] if use_cached_baseline else None

    if use_cached_baseline:
        print("Using Section 7 cached LFH source/target feature means for MLP causal baseline.")
        print(f"Cached subset mean query-wise corr: {cached_ref['mean_query_corr']:.12f}")
        if np.isfinite(cached_ref["final_rho_df_full_corr"]):
            print(f"Final_rho_df full mean query-wise corr: {cached_ref['final_rho_df_full_corr']:.12f}")
    else:
        print("Warning: not using Section 7 cached baseline:", cached_ref["reason"])
        print("The alpha=0 baseline will be recomputed, so exact equality to Final_rho_df is not guaranteed.")

    rows = []

    for query_counter, query_idx in enumerate(query_indices, start=1):
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            if use_cached_baseline:
                base_prev, base_tgt = _mlp_get_section7_cached_pair_for_prompt(
                    feature_local_idx=feature_local_idx,
                    query_idx=query_idx,
                    variant_idx=variant_idx,
                )
                baseline_source = "section7_cached_lfh_tensors"
            else:
                baseline = measure_prompt_feature_with_mlp_alpha(
                    prompt=prompt,
                    feature_idx=feature_idx,
                    mlp_layer_idx=mlp_layer_idx,
                    feature_layer_idx=feature_score_layer,
                    mlp_alpha=0.0,
                    previous_block_index=previous_block_index,
                    target_block_index=target_block_index,
                    token_scope=token_scope,
                    expected_num_blocks=NUM_BLOCKS_TOTAL,
                )
                base_prev = baseline["previous_feature_mean"]
                base_tgt = baseline["target_feature_mean"]
                baseline_source = "recomputed_forward_pass"

            for alpha in alphas_to_run:
                if float(alpha) == 0.0:
                    previous_mean = base_prev
                    target_mean = base_tgt
                else:
                    intervened = measure_prompt_feature_with_mlp_alpha(
                        prompt=prompt,
                        feature_idx=feature_idx,
                        mlp_layer_idx=mlp_layer_idx,
                        feature_layer_idx=feature_score_layer,
                        mlp_alpha=float(alpha),
                        previous_block_index=previous_block_index,
                        target_block_index=target_block_index,
                        token_scope=token_scope,
                        expected_num_blocks=NUM_BLOCKS_TOTAL,
                    )
                    previous_mean = intervened["previous_feature_mean"]
                    target_mean = intervened["target_feature_mean"]

                rows.append({
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "mlp_layer": int(mlp_layer_idx),
                    "feature_score_layer": int(feature_score_layer),
                    "token_scope": token_scope,
                    "mlp_alpha": float(alpha),
                    "effective_mlp_multiplier": float(1.0 + float(alpha)),
                    "baseline_source": baseline_source,
                    "baseline_previous_feature_mean": float(base_prev),
                    "baseline_target_feature_mean": float(base_tgt),
                    "intervened_previous_feature_mean": float(previous_mean),
                    "intervened_target_feature_mean": float(target_mean),
                    "target_delta_intervened_minus_baseline": float(target_mean - base_tgt),
                    "previous_delta_intervened_minus_baseline": float(previous_mean - base_prev),
                })

        if verbose_every is not None and (query_counter % max(verbose_every, 1) == 0 or query_counter == len(query_indices)):
            print(f"processed query {query_counter}/{len(query_indices)}; rows={len(rows)}")

    records_df = pd.DataFrame(rows)

    baseline_rows = records_df[records_df["mlp_alpha"] == 0.0].copy()
    baseline_pooled_r, baseline_mean_query_r, baseline_std_query_r, baseline_n_query = _mlp_querywise_corr_summary(
        baseline_rows,
        target_col="baseline_target_feature_mean",
        prev_col="baseline_previous_feature_mean",
    )

    if use_cached_baseline and np.isfinite(cached_ref["mean_query_corr"]):
        diff = abs(float(baseline_mean_query_r) - float(cached_ref["mean_query_corr"]))
        if diff > 1e-12:
            raise AssertionError(
                "MLP causal baseline does not match Section 7 cached LFH correlation. "
                f"baseline_mean_query_r={baseline_mean_query_r:.17g}, "
                f"cached_ref={cached_ref['mean_query_corr']:.17g}, diff={diff:.3g}."
            )
        if max_queries is None and max_variants is None and np.isfinite(cached_ref["final_rho_df_full_corr"]):
            diff_full = abs(float(baseline_mean_query_r) - float(cached_ref["final_rho_df_full_corr"]))
            if diff_full > 1e-12:
                raise AssertionError(
                    "MLP causal alpha=0 baseline does not match Final_rho_df on the full prompt set. "
                    f"baseline_mean_query_r={baseline_mean_query_r:.17g}, "
                    f"Final_rho_df={cached_ref['final_rho_df_full_corr']:.17g}, diff={diff_full:.3g}."
                )

    summary_rows = []
    for alpha in requested_alphas:
        sub = records_df[records_df["mlp_alpha"] == float(alpha)].copy()
        if len(sub) == 0:
            continue

        pooled_r, mean_query_r, std_query_r, n_query = _mlp_querywise_corr_summary(
            sub,
            target_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
        )

        delta_pooled_r, delta_mean_query_r, delta_std_query_r, delta_n_query = _mlp_querywise_corr_summary(
            sub,
            target_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
        )

        high_low_target_delta_mean, high_low_target_delta_std, high_low_n = _mlp_high_low_delta_by_query(
            sub,
            value_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        high_low_target_mean, high_low_target_std, high_low_n = _mlp_high_low_delta_by_query(
            sub,
            value_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        alpha0_mean_diff = np.nan
        alpha0_pooled_diff = np.nan
        if float(alpha) == 0.0:
            alpha0_mean_diff = float(mean_query_r - baseline_mean_query_r)
            alpha0_pooled_diff = float(pooled_r - baseline_pooled_r)

        summary_rows.append({
            "feature_idx": int(feature_idx),
            "mlp_layer": int(mlp_layer_idx),
            "feature_score_layer": int(feature_score_layer),
            "token_scope": token_scope,
            "mlp_alpha": float(alpha),
            "effective_mlp_multiplier": float(1.0 + float(alpha)),
            "baseline_source": "section7_cached_lfh_tensors" if use_cached_baseline else "recomputed_forward_pass",

            "baseline_pooled_corr_prev_to_target": baseline_pooled_r,
            "baseline_mean_query_corr_prev_to_target": baseline_mean_query_r,
            "baseline_std_query_corr_prev_to_target": baseline_std_query_r,

            "intervened_pooled_corr_prev_to_target": pooled_r,
            "intervened_mean_query_corr_prev_to_target": mean_query_r,
            "intervened_std_query_corr_prev_to_target": std_query_r,

            "corr_change_pooled_intervened_minus_baseline": pooled_r - baseline_pooled_r,
            "corr_change_mean_query_intervened_minus_baseline": mean_query_r - baseline_mean_query_r,
            "alpha0_pooled_corr_minus_baseline": alpha0_pooled_diff,
            "alpha0_mean_query_corr_minus_baseline": alpha0_mean_diff,

            "section7_cached_subset_pooled_corr": cached_ref["pooled_corr"],
            "section7_cached_subset_mean_query_corr": cached_ref["mean_query_corr"],
            "section7_cached_subset_std_query_corr": cached_ref["std_query_corr"],
            "section7_cached_subset_n_queries": int(cached_ref["n_queries"]),
            "final_rho_df_full_avg_querywise_pearson_r": cached_ref["final_rho_df_full_corr"],

            "corr_prev_to_target_delta_pooled": delta_pooled_r,
            "corr_prev_to_target_delta_mean_query": delta_mean_query_r,

            "mean_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].mean()),
            "median_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].median()),
            "fraction_target_delta_positive": float((sub["target_delta_intervened_minus_baseline"] > 0).mean()),

            "high_minus_low_prev_target_delta_mean": high_low_target_delta_mean,
            "high_minus_low_prev_target_delta_std": high_low_target_delta_std,
            "high_minus_low_prev_intervened_target_mean": high_low_target_mean,
            "high_minus_low_prev_intervened_target_std": high_low_target_std,
            "n_queries_for_high_low": int(high_low_n),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("mlp_alpha").reset_index(drop=True)

    print("=" * 120)
    print(f"MLP causal test: feature={feature_idx}, mlp_layer={mlp_layer_idx}")
    print(f"Baseline pooled corr(prev, target): {baseline_pooled_r:.12f}")
    print(f"Baseline mean query-wise corr(prev, target): {baseline_mean_query_r:.12f}")
    if use_cached_baseline:
        print(f"Section 7 cached subset mean query-wise corr: {cached_ref['mean_query_corr']:.12f}")
        if np.isfinite(cached_ref["final_rho_df_full_corr"]):
            print(f"Final_rho_df full avg_querywise_pearson_r: {cached_ref['final_rho_df_full_corr']:.12f}")
    print("Interpretation:")
    print("  alpha=0.0 leaves this MLP layer unchanged on target tokens.")
    print("  alpha=-1.0 ablates this MLP layer on target tokens.")
    print("  alpha>0 adds extra copies of this MLP layer's contribution on target tokens.")
    print("  effective multiplier = 1 + alpha.")
    print("=" * 120)
    display(summary_df)

    return summary_df, records_df


# ------------------------------------------------------------
# Plotting helpers
# ------------------------------------------------------------


def plot_mlp_causal_dose_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["mlp_alpha"],
        y=summary_df["intervened_mean_query_corr_prev_to_target"],
        mode="lines+markers",
        name="intervened corr(prev, target), mean query-wise",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    baseline = float(summary_df["baseline_mean_query_corr_prev_to_target"].iloc[0])
    fig.add_hline(y=baseline, line_dash="dash", annotation_text="baseline corr")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title=(
            "MLP-Layer Intervention Response on Local Feature Homeostasis<br>"
            f"Model={MODEL_SHORT_NAME}, Feature={int(summary_df['feature_idx'].iloc[0])}, "
            f"MLP layer={int(summary_df['mlp_layer'].iloc[0])}"
        ),
        title_font_size=26,
        xaxis_title="Intervention Strength (Alpha) on Target-Block Tokens",
        xaxis_title_font_size=22,
        yaxis_title="Mean corr(source feature, target feature)",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "mlp_causal_dose_response",
            "height": 600,
            "width": 900,
            "scale": 4,
        }
    })
    return fig


def plot_mlp_causal_delta_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["mlp_alpha"],
        y=summary_df["corr_prev_to_target_delta_mean_query"],
        mode="lines+markers",
        name="corr(prev, target_delta)",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    fig.add_trace(go.Scatter(
        x=summary_df["mlp_alpha"],
        y=summary_df["high_minus_low_prev_target_delta_mean"],
        mode="lines+markers",
        name="high-prev minus low-prev target delta",
        hovertemplate="alpha=%{x}<br>high-low delta=%{y:.4f}<extra></extra>",
    ))

    fig.add_hline(y=0.0, line_dash="dash")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title=(
            "Does MLP Steering Change the Target Feature as a Function of Source Activation?<br>"
            f"Model={MODEL_SHORT_NAME}, Feature={int(summary_df['feature_idx'].iloc[0])}, "
            f"MLP layer={int(summary_df['mlp_layer'].iloc[0])}"
        ),
        title_font_size=24,
        xaxis_title="Intervention Strength (Alpha) on Target-Block Tokens",
        xaxis_title_font_size=22,
        yaxis_title="Effect summary",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "mlp_causal_delta_response",
            "height": 600,
            "width": 900,
            "scale": 4,
        }
    })
    return fig


def plot_mlp_intervention_scatter(records_df, alpha: float):
    sub = records_df[records_df["mlp_alpha"] == float(alpha)].copy()
    if len(sub) == 0:
        raise ValueError(f"No rows for alpha={alpha}")

    x = sub["baseline_previous_feature_mean"].to_numpy(dtype=np.float64)
    y_base = sub["baseline_target_feature_mean"].to_numpy(dtype=np.float64)
    y_int = sub["intervened_target_feature_mean"].to_numpy(dtype=np.float64)

    r_base = _mlp_original_lfh_pearson_1d(x, y_base)
    r_int = _mlp_original_lfh_pearson_1d(x, y_int)

    df_plot = pd.DataFrame({
        "previous_feature_mean": np.concatenate([x, x]),
        "target_feature_mean": np.concatenate([y_base, y_int]),
        "condition": ["baseline"] * len(x) + [f"mlp_alpha={alpha}"] * len(x),
        "query_idx": np.concatenate([sub["query_idx"].values, sub["query_idx"].values]).astype(str),
        "variant_idx": np.concatenate([sub["variant_idx"].values, sub["variant_idx"].values]),
    })

    fig = px.scatter(
        df_plot,
        x="previous_feature_mean",
        y="target_feature_mean",
        color="condition",
        symbol="condition",
        hover_data=["query_idx", "variant_idx"],
        title=(
            f"Source-to-target feature relation before/after MLP intervention<br>"
            f"alpha={alpha}, baseline r={r_base:.3f}, intervened r={r_int:.3f}"
        ),
    )
    fig.update_layout(
        font=dict(family="Times New Roman"),
        title_font_size=26,
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source exemplar",
        xaxis_title_font_size=22,
        yaxis_title="Mean SAE pre-activation on target exemplar",
        yaxis_title_font_size=22,
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the MLP-layer steering test
# ------------------------------------------------------------

lfh_mlp_causal_summary_df, lfh_mlp_causal_records_df = run_mlp_layer_alpha_causal_test(
    feature_idx=int(LFH_MLP_CAUSAL_FEATURE_IDX),
    mlp_layer_idx=int(LFH_MLP_CAUSAL_SELECTED_LAYER),
    prompts_all_agnews=prompts_all_agnews,
    feature_score_layer=int(LFH_MLP_CAUSAL_FEATURE_SCORE_LAYER),
    mlp_alphas=LFH_MLP_CAUSAL_ALPHAS,
    previous_block_index=int(LFH_MLP_CAUSAL_PREVIOUS_BLOCK_INDEX),
    target_block_index=int(LFH_MLP_CAUSAL_TARGET_BLOCK_INDEX),
    token_scope=LFH_MLP_CAUSAL_TOKEN_SCOPE,
    max_queries=LFH_MLP_CAUSAL_MAX_QUERIES,
    max_variants=LFH_MLP_CAUSAL_MAX_VARIANTS,
    quantile=float(LFH_MLP_CAUSAL_QUANTILE),
    verbose_every=5,
)

_ = plot_mlp_causal_dose_response(lfh_mlp_causal_summary_df)
_ = plot_mlp_causal_delta_response(lfh_mlp_causal_summary_df)

if -1.0 in set(lfh_mlp_causal_records_df["mlp_alpha"].astype(float)):
    _ = plot_mlp_intervention_scatter(lfh_mlp_causal_records_df, alpha=-1.0)
if 1.0 in set(lfh_mlp_causal_records_df["mlp_alpha"].astype(float)):
    _ = plot_mlp_intervention_scatter(lfh_mlp_causal_records_df, alpha=1.0)


In [ ]:
# ============================================================
# Combined LFH causal plot: Attention-head vs MLP steering
# ============================================================

import pandas as pd
import plotly.graph_objects as go

# ------------------------------------------------------------
# User-provided results
# ------------------------------------------------------------

alphas = [-5, -2, -1, 0, 1, 2, 5]

attention_rho = [-0.4548, -0.6668, -0.7544, -0.8048, -0.8227, -0.8270, -0.8266]
mlp_rho       = [-0.8045, -0.8045, -0.8073, -0.8048, -0.8048, -0.8048, -0.8048]

# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------

combined_lfh_causal_df = pd.DataFrame({
    "alpha": alphas,
    "attention_head_rho": attention_rho,
    "mlp_rho": mlp_rho,
})

display(combined_lfh_causal_df)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=combined_lfh_causal_df["alpha"],
        y=combined_lfh_causal_df["attention_head_rho"],
        mode="lines+markers",
        name="Attention head steering",
        line=dict(width=3),
        marker=dict(size=9),
        hovertemplate=(
            "Method: Attention head<br>"
            "alpha=%{x}<br>"
            "mean query-wise r=%{y:.4f}"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=combined_lfh_causal_df["alpha"],
        y=combined_lfh_causal_df["mlp_rho"],
        mode="lines+markers",
        name="MLP steering",
        line=dict(width=3, dash="dash"),
        marker=dict(size=9),
        hovertemplate=(
            "Method: MLP<br>"
            "alpha=%{x}<br>"
            "mean query-wise r=%{y:.4f}"
            "<extra></extra>"
        ),
    )
)

# Optional horizontal baseline at alpha = 0 result
baseline_rho = combined_lfh_causal_df.loc[
    combined_lfh_causal_df["alpha"] == 0, "attention_head_rho"
].iloc[0]

fig.add_hline(
    y=baseline_rho,
    line_dash="dot",
    annotation_text=f"Baseline (α=0): {baseline_rho:.4f}",
    annotation_position="top left",
)

fig.update_layout(
    title=dict(
        text=(
            "LFH Causal Effect: Attention-Head vs MLP Steering"
#             "<br><sup>More negative values indicate stronger LFH</sup>"
        ),
        x=0.5,
    ),
    xaxis=dict(
        title="Steering strength (α)",
        tickmode="array",
        tickvals=alphas,
    ),
    yaxis=dict(
        title="LFH correlation",
    ),
    width=950,
    height=600,
#     template="plotly_white",
    font=dict(
        family="Times New Roman",
        size=22,
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
)

fig.show()